In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  GAME REPORT GENERATOR  ·  xG + Event Visuals  ·  Marc Lamberts            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── CONFIGURE ─────────────────────────────────────────────────────────────────
JSON_DIR   = '/Users/marclamberts/Event data/WC 2026'
MODEL_DIR  = '/Users/marclamberts/Downloads/Danger Model/xg_output'
OUT_DIR    = '/Users/marclamberts/Event data/WC 2026/reports'
BADGE_DIR  = None         # optional folder with <TeamName>.png badges
ONLY_FILES = []           # e.g. ['2026-06-11_Korea Republic - Czechia.json']
SAVE_INDIVIDUAL = True
DPI = 180


# ── DEPENDENCIES ──────────────────────────────────────────────────────────────
import subprocess, sys
def _pip(*pkgs): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])
try: import mplsoccer
except ImportError: _pip('mplsoccer')
try: import xgboost
except ImportError: _pip('xgboost')


# ── IMPORTS ───────────────────────────────────────────────────────────────────
import os, glob, json, warnings
warnings.filterwarnings('ignore')
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.image as mpimg
import matplotlib.gridspec as gridspec
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.colors import Normalize, LinearSegmentedColormap
import matplotlib.cm as cm
from mplsoccer import VerticalPitch, Pitch
from scipy.stats import gaussian_kde
import joblib

%matplotlib inline
plt.rcParams['figure.dpi'] = 120


import os, glob, json, warnings
warnings.filterwarnings('ignore')
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.image as mpimg
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.colors import Normalize
import matplotlib.cm as cm
from mplsoccer import VerticalPitch, Pitch
from scipy.stats import gaussian_kde
import joblib

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# ── Palette ───────────────────────────────────────────────────────────────────
BG         = '#0d1117'
PITCH_LINE = '#2a3f55'
HOME_COL   = '#E8720C'
AWAY_COL   = '#1A78CF'
C_YELLOW   = '#f1c40f'
C_BLUE     = '#58a6ff'
C_GREEN    = '#3fb950'
C_ORANGE   = '#f78166'
C_PURPLE   = '#bc8cff'
C_MUTED    = '#2a3f55'
AUTHOR     = 'Marc Lamberts'

OUTCOME_COLORS = {
    'Goal':      '#FFE045',   # bright gold  — goals pop
    'On Target': '#4DBBFF',   # sky blue      — saved shots visible
    'Post':      '#E8E8E8',   # near-white    — post hits distinct
    'Blocked':   '#BC8CFF',   # violet
    'Missed':    '#2a3f55',   # muted navy    — off-target recedes
}

# ── Opta constants ────────────────────────────────────────────────────────────
GOAL_X = 100.0; GOAL_Y_LEFT = 44.0; GOAL_Y_RIGHT = 56.0
GOAL_Y_CENTRE = 50.0; GOAL_WIDTH = 12.0; GOAL_H_HIGH = 62.0

Q = dict(
    PENALTY=9, HEADER=15, RIGHT_FOOT=20, LEFT_FOOT=72,
    REGULAR_PLAY=22, FAST_BREAK=23, SET_PIECE=24, FROM_CORNER=25,
    FREE_KICK=26, DIRECT_FK=28, VOLLEY=108, DEFLECTION=133,
    PULL_BACK=195, BIG_CHANCE=233, FIRST_TIME=200,
    GOAL_Y=102, GOAL_HEIGHT=231,
    UNDER_PRESSURE=18, INTENTIONAL=154, BODY_SIDE=56, BLOCKED=82,
)

# ── Shared title / footer ─────────────────────────────────────────────────────
def _title_footer(fig, match_title, subtitle=''):
    """Stamp consistent header and attribution on every figure."""
    fig.text(0.5, 0.975, match_title, ha='center', va='top',
             fontsize=17, color='white', fontweight='bold')
    if subtitle:
        fig.text(0.5, 0.930, subtitle, ha='center', va='top',
                 fontsize=9, color='#7a9ab0')
    fig.text(0.98, 0.012, f'Data: Opta  //  {AUTHOR}', ha='right', va='bottom',
             fontsize=7.5, color='#4a6070')


def _save_or_show(fig, save_path):
    if save_path:
        d = os.path.dirname(save_path)
        if d:
            os.makedirs(d, exist_ok=True)
        fig.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show()
    plt.close(fig)


# ── Geometry ──────────────────────────────────────────────────────────────────
def open_angle(x, y):
    shot = np.array([x, y], dtype=float)
    v1 = np.array([GOAL_X, GOAL_Y_LEFT])  - shot
    v2 = np.array([GOAL_X, GOAL_Y_RIGHT]) - shot
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0: return 0.0
    return float(np.degrees(np.arccos(np.clip(np.dot(v1,v2)/(n1*n2), -1, 1))))

def dist_to_goal(x, y):
    return float(np.sqrt((GOAL_X - x)**2 + (y - GOAL_Y_CENTRE)**2))

def placement_score(gy_norm, gh_norm):
    if pd.isna(gy_norm) or pd.isna(gh_norm): return np.nan
    gy = float(np.clip(gy_norm, -1, 1))
    gh = float(np.clip(gh_norm,  0, 1))
    lat  = abs(gy)
    vert = abs(gh - 0.40) / max(0.40, 0.60)
    return float(np.clip(np.sqrt(0.6*lat**2 + 0.4*vert**2)*100, 0, 100))

def goal_zone(goal_y, goal_h):
    if pd.isna(goal_y) or pd.isna(goal_h): return 'Unknown'
    h = 'Top' if goal_h >= GOAL_H_HIGH else 'Bottom'
    s = 'Left' if goal_y < 47.5 else ('Right' if goal_y > 52.5 else 'Centre')
    return f'{h} {s}'

def get_q(event, qid):
    for q in event.get('qualifier', []):
        if q['qualifierId'] == qid: return q.get('value', 1)
    return None

def has_q(event, qid):
    return any(q['qualifierId'] == qid for q in event.get('qualifier', []))

def safe_float(v):
    try: return float(v)
    except: return np.nan

def shot_power(row):
    p = 50.0
    if row.get('is_volley',0):      p += 25
    if row.get('is_first_time',0):  p += 12
    if row.get('is_header',0):      p -= 8
    if row.get('weak_foot',0):      p -= 12
    if row.get('under_pressure',0): p -= 8
    if row.get('is_deflected',0):   p += 5
    return float(np.clip(p, 0, 100))

def outcome_label(row):
    if row.get('is_goal',0):    return 'Goal'
    if row.get('is_blocked',0): return 'On Target'
    if row.get('is_post',0):    return 'Post'
    return 'Missed'

def load_badge(name):
    if not BADGE_DIR: return None
    for ext in ['png','PNG','jpg','jpeg']:
        p = os.path.join(BADGE_DIR, f'{name}.{ext}')
        if os.path.exists(p):
            try: return OffsetImage(mpimg.imread(p), zoom=0.07)
            except: pass
    return None



# ── CalibratedXGB must be defined here so joblib can unpickle the saved models ─
class CalibratedXGB:
    """XGBoost + isotonic calibration wrapper (matches training notebook)."""
    def __init__(self, clf, iso):
        self.clf = clf; self.iso = iso
    def predict_proba(self, X):
        raw = self.clf.predict_proba(X)[:, 1]
        cal = self.iso.predict(raw)
        return np.column_stack([1 - cal, cal])

def _load_pkl(path):
    if os.path.exists(path):
        return joblib.load(path)
    return None

xg_model   = _load_pkl(os.path.join(MODEL_DIR, 'model_xg.pkl'))
psxg_model = _load_pkl(os.path.join(MODEL_DIR, 'model_psxg.pkl'))
sit_model  = _load_pkl(os.path.join(MODEL_DIR, 'model_situation.pkl'))
pot_model  = _load_pkl(os.path.join(MODEL_DIR, 'model_pontarget.pkl'))
xgot_model = _load_pkl(os.path.join(MODEL_DIR, 'model_xgot.pkl'))
mo_model   = _load_pkl(os.path.join(MODEL_DIR, 'model_multi_outcome.pkl'))
XCROSS_MODEL_PATH = '/Users/marclamberts/Documents/GitHub/hradeck-scouting/League Analysis/xcross_acc_model.pkl'
_xcross_pkg = _load_pkl(XCROSS_MODEL_PATH)
xcross_model   = _xcross_pkg['model']   if _xcross_pkg else None
xcross_features = _xcross_pkg['features'] if _xcross_pkg else []

assert xg_model   is not None, f'model_xg.pkl not found in {MODEL_DIR}'
assert psxg_model is not None, f'model_psxg.pkl not found in {MODEL_DIR}'

# Load feature lists — compatible with both old and new metadata format
meta      = joblib.load(os.path.join(MODEL_DIR, 'model_meta.pkl'))
psxg_meta = joblib.load(os.path.join(MODEL_DIR, 'model_psxg_meta.pkl'))

XG_FEATURES   = meta['features']
PSXG_FEATURES = psxg_meta['features']
PEN_XG        = float(meta.get('pen_xg', psxg_meta.get('pen_xg', 0.76)))

# Situation / P(OnTarget) feature lists — fall back to hardcoded if not in meta
SIT_FEATURES = meta.get('sit_features', [
    'is_big_chance','is_fast_break','is_from_corner','is_free_kick',
    'is_set_piece','is_open_play','is_pull_back','is_first_time',
    'is_header','under_pressure','is_intentional','period_id','time_min'
])
POT_FEATURES = meta.get('pot_features', [
    f for f in XG_FEATURES
    if f not in ('is_free_kick','is_set_piece','is_pull_back','is_intentional')
])

print(f'xG model loaded    — {len(XG_FEATURES)} features')
print(f'psxG model loaded  — {len(PSXG_FEATURES)} features')
print(f'Situation model:   {"✓" if sit_model  else "✗ (not in model folder)"}')
print(f'P(OnTarget) model: {"✓" if pot_model  else "✗ (not in model folder)"}')
print(f'xGOT model:        {"✓" if xgot_model else "✗ (not in model folder)"}')
print(f'Multi-outcome:     {"✓" if mo_model   else "✗ (not in model folder)"}')
print(f'Penalty xG:        {PEN_XG:.3f}')
print(f'\nxG features: {XG_FEATURES}')

def load_opta_json(path):
    with open(path, 'r', encoding='utf-8-sig') as f:
        data = json.load(f)
    events = data.get('liveData', {}).get('event', data.get('event', []))
    rows = []
    for e in events:
        tid = str(e.get('typeId'))
        if tid not in ('13','14','15','16'): continue
        x = safe_float(e.get('x')); y = safe_float(e.get('y'))
        if np.isnan(x) or np.isnan(y): continue

        gy_raw  = safe_float(get_q(e, Q['GOAL_Y']))
        gh_raw  = safe_float(get_q(e, Q['GOAL_HEIGHT']))
        gy_norm = (gy_raw - GOAL_Y_CENTRE)/(GOAL_WIDTH/2) if not np.isnan(gy_raw) else np.nan
        gh_norm = gh_raw/100.0 if not np.isnan(gh_raw) else np.nan

        if not np.isnan(gy_norm) and not np.isnan(gh_norm):
            gf_dist = np.sqrt(gy_norm**2 + (gh_norm - 0.35)**2)
            corner  = int(abs(gy_norm) > 0.55 or gh_norm > 0.60)
            ps      = placement_score(gy_norm, gh_norm)
        elif not np.isnan(gy_norm):
            gf_dist = abs(gy_norm); corner = int(abs(gy_norm) > 0.55); ps = abs(gy_norm)*100
        else:
            gf_dist = corner = ps = np.nan

        body_side = str(get_q(e, Q['BODY_SIDE']) or '').strip()
        is_rf = has_q(e, Q['RIGHT_FOOT']); is_lf = has_q(e, Q['LEFT_FOOT'])
        weak_foot = int((is_rf and body_side == 'Left') or (is_lf and body_side == 'Right'))
        dist = dist_to_goal(x, y); angle = open_angle(x, y); y_sym = abs(y - GOAL_Y_CENTRE)

        rows.append({
            'match_file': os.path.basename(path),
            'player_id':      str(e.get('playerId','')),
            'player_name':    e.get('playerName',''),
            'contestant_id':  str(e.get('contestantId','')),
            'type_id':        int(tid),
            'period_id':      int(e.get('periodId') or 0),
            'time_min':       int(e.get('timeMin') or 0),
            'x': x, 'y': y, 'y_sym': y_sym,
            'distance': dist, 'log_distance': np.log(max(dist, 0.5)),
            'angle': angle, 'angle_sin': np.sin(np.radians(angle)),
            'in_six_yard':    int(x >= 94.2 and 36.8 <= y <= 63.2),
            'in_penalty_box': int(x >= 83.0 and 21.1 <= y <= 78.9),
            'central_y':      int(y_sym < GOAL_WIDTH/2),
            'dist_to_post':   abs(y_sym - GOAL_WIDTH/2),
            'goal_y_raw': gy_raw, 'goal_y_norm': gy_norm,
            'goal_h_raw': gh_raw, 'goal_h_norm': gh_norm,
            'goal_frame_dist': gf_dist, 'corner_zone': corner,
            'placement_score': ps, 'goal_zone': goal_zone(gy_raw, gh_raw),
            'is_header':      int(has_q(e, Q['HEADER'])),
            'is_right_foot':  int(is_rf), 'is_left_foot': int(is_lf), 'weak_foot': weak_foot,
            'is_volley':      int(has_q(e, Q['VOLLEY'])),
            'is_deflected':   int(has_q(e, Q['DEFLECTION'])),
            'is_first_time':  int(has_q(e, Q['FIRST_TIME'])),
            'is_big_chance':  int(has_q(e, Q['BIG_CHANCE'])),
            'is_fast_break':  int(has_q(e, Q['FAST_BREAK'])),
            'is_from_corner': int(has_q(e, Q['FROM_CORNER'])),
            'is_free_kick':   int(has_q(e, Q['DIRECT_FK'])),
            'is_penalty':     int(has_q(e, Q['PENALTY'])),
            'is_set_piece':   int(has_q(e, Q['SET_PIECE'])),
            'is_open_play':   int(has_q(e, Q['REGULAR_PLAY'])),
            'is_pull_back':   int(has_q(e, Q['PULL_BACK'])),
            'under_pressure': int(has_q(e, Q['UNDER_PRESSURE'])),
            'is_intentional': int(has_q(e, Q['INTENTIONAL'])),
            'is_one_on_one':  int(has_q(e, 214)),
            'is_assisted':    int(has_q(e, 55)),
            'is_intentional_assist': int(has_q(e, 215)),
            'is_throw_in_set_piece': int(has_q(e, 63)),
            'is_goal':        int(tid == '16'),
            'is_on_target':   int(tid in ('15','16')),
            'is_post':        int(tid == '14'),
            'is_blocked':     int(tid == '15'),
            'is_outfield_block': int(tid == '15' and has_q(e, Q['BLOCKED'])),
            'is_keeper_save':    int(tid == '15' and not has_q(e, Q['BLOCKED'])),
        })
    df = pd.DataFrame(rows)

    # Interaction features (match training notebook)
    if len(df) > 0:
        df['x_angle']    = df['x'] * df['angle_sin']
        df['dist_sq']    = df['distance'] ** 2
        df['zone_angle'] = df['in_penalty_box'] * df['angle']

    return df


def run_models(shots):
    """Apply all loaded models to a shots DataFrame. Returns shots with model columns added."""
    shots = shots.copy()

    # Derive / backfill features the shot parser may not have set
    if 'is_other_body_part' not in shots.columns:
        shots['is_other_body_part'] = (
            (~shots.get('is_header', pd.Series(0, index=shots.index)).astype(bool)) &
            (~shots.get('is_right_foot', pd.Series(0, index=shots.index)).astype(bool)) &
            (~shots.get('is_left_foot', pd.Series(0, index=shots.index)).astype(bool))
        ).astype(int)
    for _col in ['is_one_on_one', 'is_assisted', 'is_intentional_assist', 'is_throw_in_set_piece']:
        if _col not in shots.columns:
            shots[_col] = 0
    if 'is_individual_play' not in shots.columns:
        shots['is_individual_play'] = (shots['is_assisted'] == 0).astype(int)

    pen_mask = shots['is_penalty'] == 1
    nonpen   = shots[~pen_mask].copy()

    # xG
    X_base = nonpen[XG_FEATURES].fillna(0).astype(float)
    nonpen['xg'] = xg_model.predict_proba(X_base)[:, 1]

    # psxG
    X_ps = nonpen[PSXG_FEATURES].fillna(0).astype(float)
    nonpen['psxg'] = psxg_model.predict_proba(X_ps)[:, 1]
    has_pl = nonpen[['goal_y_norm','goal_h_norm']].notna().any(axis=1)
    nonpen.loc[~has_pl, 'psxg'] = nonpen.loc[~has_pl, 'xg']

    # Situation danger
    if sit_model is not None:
        for _c in SIT_FEATURES:
            if _c not in nonpen.columns: nonpen[_c] = 0
        X_sit = nonpen[SIT_FEATURES].fillna(0).astype(float)
        nonpen['situation_danger'] = sit_model.predict_proba(X_sit)[:, 1]

    # P(On Target)
    if pot_model is not None:
        for _c in POT_FEATURES:
            if _c not in nonpen.columns: nonpen[_c] = 0
        X_pot = nonpen[POT_FEATURES].fillna(0).astype(float)
        nonpen['p_ontarget'] = pot_model.predict_proba(X_pot)[:, 1]

    # xGOT (on-target shots only)
    if xgot_model is not None:
        ot_mask = nonpen['is_on_target'] == 1
        if ot_mask.sum() > 0:
            X_xgot = nonpen.loc[ot_mask, PSXG_FEATURES].fillna(0).astype(float)
            nonpen.loc[ot_mask, 'xgot'] = xgot_model.predict_proba(X_xgot)[:, 1]
        nonpen.loc[~ot_mask, 'xgot'] = 0.0

    # Multi-outcome
    if mo_model is not None:
        X_mo = nonpen[XG_FEATURES].fillna(0).astype(float)
        mo_proba = mo_model.predict_proba(X_mo)
        for i, col in enumerate(['p_miss','p_post','p_save','p_goal_mo']):
            if mo_proba.shape[1] > i:
                nonpen[col] = mo_proba[:, i]

    # Write back to full df
    for col in ['xg','psxg','situation_danger','p_ontarget','xgot',
                'p_miss','p_post','p_save','p_goal_mo']:
        if col in nonpen.columns:
            shots.loc[~pen_mask, col] = nonpen[col].values
    shots.loc[pen_mask, 'xg']   = PEN_XG
    shots.loc[pen_mask, 'psxg'] = PEN_XG

    # Derived
    shots['placement_quality'] = shots['psxg'] - shots['xg']
    shots['finishing_luck']    = shots['is_goal'] - shots['psxg']
    shots['shot_power']        = shots.apply(shot_power, axis=1)
    shots['outcome']           = shots.apply(outcome_label, axis=1)
    return shots


def parse_match_meta(path, shots):
    """Extract team names, IDs, match title, competition and date from filename + JSON."""
    from datetime import datetime
    filename_base = os.path.splitext(os.path.basename(path))[0]
    try:
        date_str, match_str = filename_base.split('_', 1)
        home_name, away_name = match_str.split(' - ', 1)
    except ValueError:
        date_str = ''; home_name = 'Home'; away_name = 'Away'

    competition = ''
    match_date  = date_str
    try:
        with open(path) as f:
            raw = json.load(f)
        score = raw.get('matchDetails', {}).get('scores', {}).get('ft', {})
        score_str = f"{score.get('home','?')} – {score.get('away','?')}"
        match_title = f"{home_name}  {score_str}  {away_name}"
        mi = raw.get('matchInfo', raw.get('matchDetails', {}))
        competition = (mi.get('competition', {}).get('name', '')
                       or mi.get('competitionName', ''))
        raw_date = mi.get('date', '') or mi.get('matchDate', '') or date_str
        try:
            dt = datetime.strptime(raw_date[:10], '%Y-%m-%d')
            match_date = dt.strftime('%-d %B %Y')
        except Exception:
            match_date = date_str
    except Exception:
        match_title = f"{home_name} vs {away_name}"

    cnt = Counter(shots['contestant_id'])
    team_ids = list(shots['contestant_id'].unique())
    home_id = None

    # Determine home_id by matching goal counts to the final score (most reliable signal)
    try:
        events = raw.get('liveData', {}).get('event', raw.get('event', []))
        h_goals = score.get('home')
        a_goals = score.get('away')
        if h_goals is not None and a_goals is not None and h_goals != a_goals:
            goal_cnt = Counter(str(e.get('contestantId', '')) for e in events
                               if str(e.get('typeId')) == '16')
            cands = [cid for cid in team_ids if goal_cnt.get(cid, 0) == h_goals]
            if len(cands) == 1:
                home_id = cands[0]
        # Fallback: first pre-match (period 16) event contestant
        if home_id is None:
            for e in events:
                if int(e.get('periodId', 0)) == 16:
                    cid = str(e.get('contestantId', ''))
                    if cid in team_ids:
                        home_id = cid
                        break
    except Exception:
        pass

    if home_id is None:
        home_id = cnt.most_common(1)[0][0]

    away_id  = [t for t in team_ids if t != home_id][0] if len(team_ids) > 1 else home_id
    id_to_name = {home_id: home_name, away_id: away_name}
    shots['team'] = shots['contestant_id'].map(id_to_name).fillna('Unknown')
    return shots, home_id, away_id, home_name, away_name, match_title, filename_base, competition, match_date



# ══════════════════════════════════════════════════════════════════════════════
# xG VISUALS
# ══════════════════════════════════════════════════════════════════════════════

# ── V1 & V2: xG Shot Maps ─────────────────────────────────────────────────────
def _shot_map_ax(ax, t, metric='xg', scale=1800):
    """Draw shot markers on a VerticalPitch axes — enhanced."""
    MARKER_MAP  = {'Goal': '*',  'On Target': 'o', 'Post': 'D', 'Blocked': 's', 'Missed': 'o'}
    SIZE_SCALE  = {'Goal': 2.6,  'On Target': 1.8, 'Post': 1.5, 'Blocked': 1.4, 'Missed': 1.1}
    GOAL_CENTRE = 50.0   # opta pitch y-centre of goal

    # Angle lines to goal for notable shots (xg >= 0.10)
    for _, row in t.iterrows():
        xg_val = float(row[metric]) if not pd.isna(row[metric]) else 0.0
        if xg_val >= 0.10:
            col = OUTCOME_COLORS.get(row['outcome'], C_MUTED)
            ax.plot([row['y'], GOAL_CENTRE], [row['x'], 99.5],
                    color=col, lw=0.5, alpha=0.14, zorder=2, solid_capstyle='round')

    placed_labels = []
    for _, row in t.iterrows():
        outcome = row['outcome']
        col     = OUTCOME_COLORS.get(outcome, C_MUTED)
        xg_val  = float(row[metric]) if not pd.isna(row[metric]) else 0.0
        sz      = max(60, xg_val * scale * SIZE_SCALE.get(outcome, 1.0))
        is_goal = (outcome == 'Goal')
        marker  = MARKER_MAP.get(outcome, 'o')
        missed  = (outcome == 'Missed')
        ec      = '#ffffff' if is_goal else ('#aaccee' if not missed else col)
        lw      = 2.8 if is_goal else (0.9 if not missed else 0.0)
        fc      = 'none' if missed else col
        alpha   = 0.97 if is_goal else (0.82 if not missed else 0.50)

        if missed:
            ax.scatter(row['y'], row['x'], s=sz, facecolors='none',
                       edgecolors=ec, linewidths=0.0, marker=marker,
                       zorder=4, alpha=alpha)
        else:
            ax.scatter(row['y'], row['x'], s=sz, color=col,
                       edgecolors=ec, linewidths=lw, marker=marker,
                       zorder=6 if is_goal else 5, alpha=alpha)

        # Collect labels for goals and very high-xG shots
        if is_goal or xg_val >= 0.40:
            player = str(row.get('player_name', '')).split()[-1] if row.get('player_name') else ''
            minute = int(row.get('time_min', 0))
            label  = f"{player} {minute}\u2019" if is_goal else f"xG {xg_val:.2f}"
            placed_labels.append((float(row['y']), float(row['x']), label, col, is_goal))

    # Render labels with basic y-offset collision avoidance
    used_positions = []
    for lx, ly, label, col, is_goal in placed_labels:
        dy = 4.5
        for ux, uy in used_positions:
            if abs(lx - ux) < 9 and abs(ly + dy - uy) < 3.5:
                dy += 4.5
        used_positions.append((lx, ly + dy))
        ax.annotate(label, xy=(lx, ly), xytext=(lx, ly + dy),
                    fontsize=7.0 if is_goal else 6.0,
                    color='white' if is_goal else '#aabbcc',
                    fontweight='bold' if is_goal else 'normal',
                    ha='center', va='bottom', zorder=9,
                    arrowprops=dict(arrowstyle='-', color=col, lw=0.5, alpha=0.45))

    return int(t['is_goal'].sum()), t['xg'].sum(), t['psxg'].sum()


def plot_shot_map(shots, team_id, id_to_name, match_title, subtitle='',
                  metric='xg', save_path=None):
    name = id_to_name.get(team_id, str(team_id))
    t    = shots[shots['contestant_id'] == team_id].copy()

    team_col = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL

    fig = plt.figure(figsize=(11, 10.5), facecolor=BG)
    ax  = fig.add_axes([0.05, 0.09, 0.90, 0.76])

    pitch = VerticalPitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                          linewidth=1.2, half=True, pad_top=4, pad_bottom=2,
                          goal_type='box', goal_alpha=0.8)
    pitch.draw(ax=ax)

    # Subtle penalty-area glow to give spatial context
    from matplotlib.patches import FancyArrowPatch
    import matplotlib.patheffects as pe
    pen_box = plt.Polygon([[21.1, 83.0], [78.9, 83.0], [78.9, 100.0], [21.1, 100.0]],
                          closed=True, facecolor=team_col, alpha=0.06,
                          edgecolor='none', zorder=1)
    six_box = plt.Polygon([[36.8, 94.2], [63.2, 94.2], [63.2, 100.0], [36.8, 100.0]],
                          closed=True, facecolor=team_col, alpha=0.10,
                          edgecolor='none', zorder=1)
    ax.add_patch(pen_box)
    ax.add_patch(six_box)

    goals, xg_tot, ps_tot = _shot_map_ax(ax, t, metric=metric)

    # xG size reference legend + outcome legend
    legend_y = 0.065
    # outcome symbols
    mrkr_map = {'Goal': ('★', 14), 'On Target': ('●', 12), 'Post': ('◆', 11), 'Missed': ('○', 11)}
    x_start  = 0.08
    for lbl, lc in OUTCOME_COLORS.items():
        sym, fs = mrkr_map.get(lbl, ('●', 11))
        fig.text(x_start, legend_y, sym, ha='center', va='center', fontsize=fs, color=lc)
        fig.text(x_start + 0.03, legend_y, lbl, ha='left', va='center',
                 fontsize=7.8, color='#9ab0c0')
        x_start += 0.19
    # size reference
    fig.text(0.88, legend_y + 0.015, 'Bubble ∝ xG', ha='right', va='center',
             fontsize=7.5, color='#6a8090', style='italic')

    # Stats line
    home_col = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    g_xg_diff = goals - xg_tot
    diff_col  = '#3fb950' if g_xg_diff >= 0 else '#f78166'
    fig.text(0.5, 0.036,
             f'Goals: {goals}   ·   xG: {xg_tot:.2f}   ·   psxG: {ps_tot:.2f}   ·   G−xG: {g_xg_diff:+.2f}',
             ha='center', va='bottom', fontsize=9.5, color='white', fontweight='bold')

    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()



# ── V3: xG Flow Chart ─────────────────────────────────────────────────────────
def plot_xg_flow(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    fig, ax = plt.subplots(figsize=(14, 5.5), facecolor=BG)
    ax.set_facecolor(BG)

    max_min = max(int(shots['time_min'].max()) + 3, 95)

    placed = []
    for team_id, color in [(home_id, HOME_COL), (away_id, AWAY_COL)]:
        name = id_to_name.get(team_id, str(team_id))
        t = shots[shots['contestant_id'] == team_id].sort_values('time_min').copy()
        if len(t) == 0:
            continue
        t['cum_xg'] = t['xg'].cumsum()
        total_xg = t['xg'].sum()

        mins = np.concatenate([[0], t['time_min'].values, [max_min]])
        xgs  = np.concatenate([[0], t['cum_xg'].values,  [total_xg]])

        ax.step(mins, xgs, where='post', color=color, lw=2.2, zorder=4, solid_capstyle='round')
        ax.fill_between(mins, xgs, step='post', alpha=0.18, color=color, zorder=3)

        # Goal markers — vertical dashes + scorer label
        goals_t = t[t['is_goal'] == 1]
        for _, row in goals_t.iterrows():
            ax.axvline(row['time_min'], color=color, lw=1.0, alpha=0.55, ls='--', zorder=2)
            ax.text(row['time_min'] + 0.6, row['cum_xg'] + 0.04,
                    f"{row['player_name'].split(' ')[-1]} {int(row['time_min'])}\u2019",
                    fontsize=7.5, color=color, va='bottom', fontweight='bold')

        placed.append((name, total_xg, color))

    # Team labels stacked top-left
    for i, (name, xg, color) in enumerate(placed):
        ax.text(0.01, 0.97 - i * 0.11, f'{name}  ({xg:.2f} xG)',
                ha='left', va='top', fontsize=10.5, color=color, fontweight='bold',
                transform=ax.transAxes)

    ymax = ax.get_ylim()[1]
    ax.axvline(45, color='white', lw=0.7, alpha=0.18, ls='--')
    ax.text(45.8, ymax * 0.03, 'HT', fontsize=7.5, color='#7a9ab0', va='bottom')

    ax.set_xlabel('Minute', color='#8b9ab0', fontsize=9)
    ax.set_ylabel('Cumulative xG', color='#8b9ab0', fontsize=9)
    ax.tick_params(colors='#8b9ab0', labelsize=8)
    for spine in ax.spines.values(): spine.set_color(PITCH_LINE)
    ax.set_xlim(0, max_min); ax.set_ylim(bottom=0)
    ax.grid(axis='y', color=PITCH_LINE, lw=0.5, alpha=0.35)

    fig.subplots_adjust(top=0.86, bottom=0.11, left=0.07, right=0.97)
    _title_footer(fig, match_title, subtitle)

    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()



# ── V4: Shot Timeline ─────────────────────────────────────────────────────────
def plot_shot_timeline(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    fig, ax = plt.subplots(figsize=(16, 4.5), facecolor=BG)
    ax.set_facecolor(BG)

    home_name = id_to_name.get(home_id, 'Home')
    away_name = id_to_name.get(away_id, 'Away')

    for team_id, y_base, color, name in [
        (home_id,  0.55, HOME_COL, home_name),
        (away_id, -0.55, AWAY_COL, away_name),
    ]:
        t = shots[shots['contestant_id'] == team_id].copy()
        sign = 1 if y_base > 0 else -1
        for _, row in t.iterrows():
            col = OUTCOME_COLORS.get(row['outcome'], C_MUTED)
            sz  = max(35, row['xg'] * 800)
            is_goal = row['outcome'] == 'Goal'
            ax.scatter(row['time_min'], y_base + sign * row['xg'] * 0.30,
                       s=sz, color=col,
                       edgecolors='white' if is_goal else col,
                       linewidth=2.0 if is_goal else 0.0,
                       alpha=0.88, zorder=4)
        ax.text(-2, y_base, name, ha='right', va='center',
                fontsize=9.5, color=color, fontweight='bold')

    ax.axhline(0, color=PITCH_LINE, lw=1.0, alpha=0.6)
    ax.axvline(45, color='white', lw=0.7, alpha=0.2, ls='--')
    ax.text(45.6, 0.02, 'HT', ha='left', fontsize=7.5, color='#7a9ab0', va='bottom')
    ax.set_xlim(-5, max(shots['time_min'].max() + 3, 95))
    ax.set_ylim(-1.15, 1.15)
    ax.set_xlabel('Minute', color='#8b9ab0', fontsize=9)
    ax.set_yticks([])
    ax.tick_params(axis='x', colors='#8b9ab0', labelsize=8)
    for spine in ax.spines.values(): spine.set_color(PITCH_LINE)

    legend_els = [mpatches.Patch(color=c, label=l) for l, c in OUTCOME_COLORS.items()]
    ax.legend(handles=legend_els, fontsize=8, facecolor=BG, labelcolor='white',
              framealpha=0.4, loc='upper right', ncol=4, edgecolor=PITCH_LINE)

    fig.subplots_adjust(top=0.84, bottom=0.13, left=0.06, right=0.97)
    _title_footer(fig, match_title, subtitle)

    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()



# ── V5: Goal Frame Heatmap ────────────────────────────────────────────────────
def plot_goal_frame(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(16, 7.5), facecolor=BG)
    fig.subplots_adjust(top=0.83, bottom=0.15, left=0.04, right=0.97, wspace=0.10)

    for ax, team_id, color in [(axes[0], home_id, HOME_COL), (axes[1], away_id, AWAY_COL)]:
        name = id_to_name.get(team_id, str(team_id))
        t    = shots[(shots['contestant_id'] == team_id) & shots['goal_y_norm'].notna()].copy()
        t['goal_h_norm'] = t['goal_h_norm'].fillna(0.35).clip(0, 1)
        t['goal_y_norm'] = t['goal_y_norm'].clip(-1, 1)

        ax.set_facecolor(BG)
        ax.set_xlim(-1.22, 1.22)
        ax.set_ylim(-0.10, 1.20)
        ax.set_aspect('equal')
        ax.axis('off')

        # KDE density overlay when we have enough shots on frame
        if len(t) >= 4:
            try:
                from scipy.stats import gaussian_kde as _kde
                _cmap_name = 'YlOrBr' if color == HOME_COL else 'PuBu'
                _pts = np.vstack([t['goal_y_norm'].values, t['goal_h_norm'].values])
                _kde_fn = _kde(_pts, bw_method=0.45)
                _gx = np.linspace(-1, 1, 60); _gy = np.linspace(0, 1.05, 40)
                _GX, _GY = np.meshgrid(_gx, _gy)
                _ZZ = _kde_fn(np.vstack([_GX.ravel(), _GY.ravel()])).reshape(_GX.shape)
                ax.contourf(_GX, _GY, _ZZ, levels=10, cmap=_cmap_name, alpha=0.55, zorder=1)
            except Exception:
                pass

        # net grid
        net_col = '#1e3a50'
        for xg in np.linspace(-1, 1, 13):
            ax.plot([xg, xg], [0, 1], color=net_col, lw=0.6, alpha=0.45, zorder=2)
        for yg in np.linspace(0, 1, 9):
            ax.plot([-1, 1], [yg, yg], color=net_col, lw=0.6, alpha=0.45, zorder=2)

        # goal frame
        ax.plot([-1, 1], [1, 1], color='white', lw=5.0, alpha=0.90,
                solid_capstyle='round', zorder=4)
        ax.plot([-1, -1], [0, 1], color='white', lw=5.0, alpha=0.90,
                solid_capstyle='round', zorder=4)
        ax.plot([ 1,  1], [0, 1], color='white', lw=5.0, alpha=0.90,
                solid_capstyle='round', zorder=4)
        ax.plot([-1, 1], [0, 0], color='white', lw=2.0, alpha=0.40, zorder=4)
        # post highlights
        for xp in [-1, 1]:
            ax.plot([xp, xp], [0, 1], color='#a0c8e0', lw=1.2, alpha=0.25, zorder=3)

        # shots
        for _, row in t.iterrows():
            outcome = row.get('outcome', 'Missed')
            col     = OUTCOME_COLORS.get(outcome, C_MUTED)
            is_goal = (outcome == 'Goal')
            ax.scatter(row['goal_y_norm'], row['goal_h_norm'],
                       s=240 if is_goal else 90,
                       marker='*' if is_goal else 'o',
                       color=col,
                       edgecolors='white' if is_goal else color,
                       linewidth=1.0, alpha=0.95, zorder=6)

        # legend
        seen = t['outcome'].dropna().unique()
        for lbl, lc in OUTCOME_COLORS.items():
            if lbl in seen:
                ax.scatter([], [], s=60, color=lc, edgecolors='white',
                           linewidth=0.7, label=lbl)
        ax.legend(fontsize=7.5, facecolor=BG, labelcolor='white',
                  framealpha=0.35, edgecolor=PITCH_LINE,
                  loc='upper center', ncol=4, bbox_to_anchor=(0.5, -0.03))

        goals  = int(t['is_goal'].sum())
        saved  = int(t['outcome'].eq('Saved').sum())
        ax.text(0.5, -0.07,
                f'{name}   {len(t)} on frame  ·  Goals: {goals}  ·  Saved: {saved}',
                ha='center', va='top', fontsize=9.5, color=color,
                fontweight='bold', transform=ax.transAxes)

    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()



# ── V6: Match Stats Dashboard ──────────────────────────────────────────────────
def plot_stats_dashboard(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    home_name = id_to_name.get(home_id, 'Home')
    away_name = id_to_name.get(away_id, 'Away')

    def team_stats(tid):
        t = shots[shots['contestant_id'] == tid]
        return {
            'Shots':          len(t),
            'On Target':      int(t['is_on_target'].sum()),
            'Goals':          int(t['is_goal'].sum()),
            'xG':             round(t['xg'].sum(), 2),
            'psxG':           round(t['psxg'].sum(), 2),
            'G − xG':         round(t['is_goal'].sum() - t['xg'].sum(), 2),
            'G − psxG':       round(t['is_goal'].sum() - t['psxg'].sum(), 2),
            'Big Chances':    int(t['is_big_chance'].sum()),
            'Headers':        int(t['is_header'].sum()),
            'Set Pieces':     int(t['is_set_piece'].sum()),
            'Fast Breaks':    int(t['is_fast_break'].sum()),
            'Under Pressure': int(t['under_pressure'].sum()),
            'xG / Shot':      round(t['xg'].mean(), 3) if len(t) > 0 else 0,
        }

    hs = team_stats(home_id)
    as_ = team_stats(away_id)
    metrics = list(hs.keys())

    fig, ax = plt.subplots(figsize=(12, 7.5), facecolor=BG)
    ax.set_facecolor(BG); ax.axis('off')
    fig.subplots_adjust(top=0.88, bottom=0.04, left=0.0, right=1.0)

    # Team headers
    ax.text(0.18, 0.96, home_name, ha='center', va='top',
            fontsize=12, color=HOME_COL, fontweight='bold', transform=ax.transAxes)
    ax.text(0.82, 0.96, away_name, ha='center', va='top',
            fontsize=12, color=AWAY_COL, fontweight='bold', transform=ax.transAxes)

    n      = len(metrics)
    row_h  = 0.82 / n
    bar_w  = 0.27   # half-width of bar zone

    for i, metric in enumerate(metrics):
        y  = 0.90 - (i + 0.5) * row_h
        hv = hs[metric]; av = as_[metric]

        # Alternating row tint
        if i % 2 == 0:
            ax.axhspan(y - row_h * 0.48, y + row_h * 0.48,
                       xmin=0.02, xmax=0.98, color='white', alpha=0.03)

        total = abs(float(hv)) + abs(float(av)) if isinstance(hv, (int,float)) else 0
        if total > 0:
            hw = bar_w * abs(float(hv)) / max(total, 1e-9)
            aw = bar_w * abs(float(av)) / max(total, 1e-9)
            bar_h = row_h * 0.50
            ax.barh(y, -hw, height=bar_h, left=0.50,
                    color=HOME_COL, alpha=0.75, transform=ax.transAxes)
            ax.barh(y,  aw, height=bar_h, left=0.50,
                    color=AWAY_COL, alpha=0.75, transform=ax.transAxes)

        hcol = HOME_COL if isinstance(hv,(int,float)) and isinstance(av,(int,float)) and hv > av else 'white'
        acol = AWAY_COL if isinstance(av,(int,float)) and isinstance(hv,(int,float)) and av > hv else 'white'

        ax.text(0.18, y, str(hv), ha='center', va='center',
                fontsize=11.5, color=hcol, fontweight='bold', transform=ax.transAxes)
        ax.text(0.50, y, metric, ha='center', va='center',
                fontsize=8.5, color='#8b9ab0', transform=ax.transAxes)
        ax.text(0.82, y, str(av), ha='center', va='center',
                fontsize=11.5, color=acol, fontweight='bold', transform=ax.transAxes)

    _title_footer(fig, match_title, subtitle)

    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()



# ── V7: Shot Quality Scatter (distance vs xG, sized by psxG) ──────────────────
def plot_shot_scatter(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=BG)
    fig.subplots_adjust(top=0.84, bottom=0.11, left=0.07, right=0.97, wspace=0.22)

    for ax, team_id, color in [(axes[0], home_id, HOME_COL), (axes[1], away_id, AWAY_COL)]:
        name = id_to_name.get(team_id, str(team_id))
        t    = shots[shots['contestant_id'] == team_id].copy()
        ax.set_facecolor(BG)

        for _, row in t.iterrows():
            col = OUTCOME_COLORS.get(row.get('outcome','Missed'), C_MUTED)
            sz  = max(40, row['psxg'] * 1400)
            is_goal = row['outcome'] == 'Goal'
            ax.scatter(row['distance'], row['xg'], s=sz, color=col,
                       edgecolors='white' if is_goal else col,
                       linewidth=2.0 if is_goal else 0.0,
                       alpha=0.88, zorder=4)


        d_range = np.linspace(5, 35, 100)
        ax.plot(d_range, 0.5 * np.exp(-d_range/15), color='white', lw=0.7,
                ls=':', alpha=0.25)

        ax.set_xlabel('Distance to goal (m)', color='#8b9ab0', fontsize=9)
        ax.set_ylabel('xG', color='#8b9ab0', fontsize=9)
        ax.set_ylim(-0.02, 1.02)
        ax.tick_params(colors='#8b9ab0', labelsize=8)
        for spine in ax.spines.values(): spine.set_color(PITCH_LINE)
        ax.text(0.5, 1.03, name, ha='center', va='bottom', fontsize=11,
                color=color, fontweight='bold', transform=ax.transAxes)

        legend_els = [mpatches.Patch(color=c, label=l) for l, c in OUTCOME_COLORS.items()]
        ax.legend(handles=legend_els, fontsize=7.5, facecolor=BG, labelcolor='white',
                  framealpha=0.4, loc='upper right', edgecolor=PITCH_LINE)

    _title_footer(fig, match_title, subtitle)

    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()



# ── V8: Pressure Analysis ──────────────────────────────────────────────────────
def plot_pressure_analysis(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), facecolor=BG)
    fig.subplots_adjust(top=0.84, bottom=0.11, left=0.07, right=0.97, wspace=0.25)

    # Left: mean xG under / not under pressure
    ax = axes[0]; ax.set_facecolor(BG)
    w = 0.28; x_pos = np.arange(2)
    team_labels = []
    for i, (team_id, color) in enumerate([(home_id, HOME_COL), (away_id, AWAY_COL)]):
        t    = shots[shots['contestant_id'] == team_id]
        name = id_to_name.get(team_id, str(team_id))
        team_labels.append(name)
        np_xg = t[t['under_pressure']==0]['xg'].mean() if (t['under_pressure']==0).sum()>0 else 0
        pp_xg = t[t['under_pressure']==1]['xg'].mean() if (t['under_pressure']==1).sum()>0 else 0
        ax.bar(i - w/2, np_xg, w, color=color, alpha=0.88, label='No pressure' if i==0 else '')
        ax.bar(i + w/2, pp_xg, w, color=color, alpha=0.40, label='Under pressure' if i==0 else '')
        for val, xoff in [(np_xg, i-w/2), (pp_xg, i+w/2)]:
            ax.text(xoff, val + 0.002, f'{val:.3f}', ha='center', va='bottom',
                    fontsize=8.5, color='white')

    ax.set_xticks(range(2)); ax.set_xticklabels(team_labels, color='#8b9ab0', fontsize=9)
    ax.set_ylabel('Mean xG per shot', color='#8b9ab0', fontsize=9)
    ax.tick_params(colors='#8b9ab0', labelsize=8)
    for spine in ax.spines.values(): spine.set_color(PITCH_LINE)
    ax.legend(fontsize=8.5, facecolor=BG, labelcolor='white', framealpha=0.4,
              edgecolor=PITCH_LINE)
    ax.grid(axis='y', color=PITCH_LINE, lw=0.5, alpha=0.35)
    ax.text(0.5, 1.03, 'xG Under Pressure vs Free', ha='center', va='bottom',
            fontsize=10, color='white', transform=ax.transAxes)

    # Right: pitch scatter of pressured shots
    ax2 = axes[1]; ax2.set_facecolor(BG)
    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.9, half=True)
    pitch.draw(ax=ax2)
    for team_id, color in [(home_id, HOME_COL), (away_id, AWAY_COL)]:
        t = shots[(shots['contestant_id'] == team_id) & (shots['under_pressure'] == 1)]
        name = id_to_name.get(team_id, str(team_id))
        if len(t) > 0:
            ax2.scatter(t['x'], t['y'], s=t['xg']*900 + 25, color=color,
                        alpha=0.72, label=f'{name} ({len(t)})', zorder=4,
                        edgecolors=color, linewidth=0.0)
    ax2.legend(fontsize=8.5, facecolor=BG, labelcolor='white', framealpha=0.4,
               edgecolor=PITCH_LINE, loc='lower right')
    ax2.text(0.5, 1.03, 'Shots Under Pressure  (size = xG)', ha='center', va='bottom',
             fontsize=10, color='white', transform=ax2.transAxes)

    _title_footer(fig, match_title, subtitle)

    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()



# ── V9: Shot Type Breakdown ────────────────────────────────────────────────────
def plot_shot_types(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    type_cols = {
        'Open Play':    'is_open_play',
        'Header':       'is_header',
        'Set Piece':    'is_set_piece',
        'Corner':       'is_from_corner',
        'Fast Break':   'is_fast_break',
        'Free Kick':    'is_free_kick',
        'Penalty':      'is_penalty',
        'Volley':       'is_volley',
        'First Time':   'is_first_time',
        'Weak Foot':    'weak_foot',
        'Big Chance':   'is_big_chance',
        'Pull-back':    'is_pull_back',
    }
    home_name = id_to_name.get(home_id, 'Home')
    away_name = id_to_name.get(away_id, 'Away')
    ht = shots[shots['contestant_id'] == home_id]
    at = shots[shots['contestant_id'] == away_id]
    labels = list(type_cols.keys())
    h_vals = [int(ht[c].sum()) if c in ht.columns else 0 for c in type_cols.values()]
    a_vals = [int(at[c].sum()) if c in at.columns else 0 for c in type_cols.values()]

    fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), facecolor=BG)
    fig.subplots_adjust(top=0.84, bottom=0.08, left=0.12, right=0.97, wspace=0.35)

    for col_idx, (ax, vals_h, vals_a, xlabel, xg_mode) in enumerate([
        (axes[0], h_vals, a_vals, 'Shot count', False),
        (axes[1],
         [ht[ht[c]==1]['xg'].mean() if c in ht.columns and (ht[c]==1).sum()>0 else 0 for c in type_cols.values()],
         [at[at[c]==1]['xg'].mean() if c in at.columns and (at[c]==1).sum()>0 else 0 for c in type_cols.values()],
         'Mean xG', True),
    ]):
        ax.set_facecolor(BG)
        x_pos = np.arange(len(labels)); w = 0.35
        bars_h = ax.barh(x_pos + w/2, vals_h, w, color=HOME_COL, alpha=0.85, label=home_name)
        bars_a = ax.barh(x_pos - w/2, vals_a, w, color=AWAY_COL, alpha=0.85, label=away_name)
        ax.set_yticks(x_pos)
        ax.set_yticklabels(labels, fontsize=8.5, color='#8b9ab0')
        ax.set_xlabel(xlabel, color='#8b9ab0', fontsize=9)
        ax.tick_params(colors='#8b9ab0', labelsize=8)
        for spine in ax.spines.values(): spine.set_color(PITCH_LINE)
        ax.grid(axis='x', color=PITCH_LINE, lw=0.5, alpha=0.35)
        ax.legend(fontsize=9, facecolor=BG, labelcolor='white', framealpha=0.4,
                  edgecolor=PITCH_LINE)

    _title_footer(fig, match_title, subtitle)

    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()



# ── V10: xG vs psxG (Placement Quality) ───────────────────────────────────────
def plot_placement_quality(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=BG)

    for ax, team_id, color in [(axes[0], home_id, HOME_COL), (axes[1], away_id, AWAY_COL)]:
        name = id_to_name.get(team_id, str(team_id))
        t    = shots[shots['contestant_id'] == team_id].copy()
        ax.set_facecolor(BG)

        ax.axline((0,0), slope=1, color='white', lw=0.8, ls='--', alpha=0.3, label='xG = psxG')

        for _, row in t.iterrows():
            oc  = row.get('outcome','Missed')
            col = OUTCOME_COLORS.get(oc, C_MUTED)
            ax.scatter(row['xg'], row['psxg'], s=80, color=col,
                       edgecolors='white' if oc=='Goal' else 'none',
                       linewidth=1.5, alpha=0.85, zorder=4)
            if oc == 'Goal':
                ax.annotate(row['player_name'].split(' ')[-1],
                            xy=(row['xg'], row['psxg']),
                            xytext=(5, 4), textcoords='offset points',
                            fontsize=7, color=C_YELLOW)

        ax.fill_between([0, 1], [0, 1], [0, 0], alpha=0.04, color=C_ORANGE, label='psxG < xG')
        ax.fill_between([0, 1], [0, 1], [1, 1], alpha=0.04, color=C_GREEN,  label='psxG > xG')

        ax.set_xlabel('Pre-shot xG', color='white', fontsize=9)
        ax.set_ylabel('Post-shot psxG', color='white', fontsize=9)
        ax.set_title(name, color=color, fontsize=11, fontweight='bold')
        ax.set_xlim(0, None); ax.set_ylim(0, None)
        ax.tick_params(colors='white')
        for spine in ax.spines.values(): spine.set_color(PITCH_LINE)

        pq_mean = t['placement_quality'].mean()
        ax.text(0.98, 0.04, f'Mean psxG − xG: {pq_mean:+.3f}',
                ha='right', va='bottom', fontsize=9, color='white',
                fontweight='bold', transform=ax.transAxes)
        legend_els = [mpatches.Patch(color=c, label=l) for l, c in OUTCOME_COLORS.items()]
        ax.legend(handles=legend_els, fontsize=7, facecolor=BG, labelcolor='white',
                  framealpha=0.4, loc='upper left')

    plt.suptitle(f'{match_title}  —  Placement Quality (xG → psxG)',
                 color='white', fontsize=12, fontweight='bold')
    fig.subplots_adjust(top=0.84, bottom=0.09, left=0.07, right=0.97)
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── V11: P(On Target) Shot Map ────────────────────────────────────────────────
def plot_pontarget_map(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    if 'p_ontarget' not in shots.columns:
        print('    [skip] p_ontarget not available'); return

    cmap = LinearSegmentedColormap.from_list('pot', ['#1a1a2e', '#e63946', '#f4a261', '#2a9d8f'])
    norm = Normalize(vmin=0, vmax=1)

    fig, axes = plt.subplots(1, 2, figsize=(18, 9), facecolor=BG)
    fig.subplots_adjust(top=0.84, bottom=0.08, left=0.04, right=0.96, wspace=0.06)

    for ax, team_id, color in [(axes[0], home_id, HOME_COL), (axes[1], away_id, AWAY_COL)]:
        name  = id_to_name.get(team_id, str(team_id))
        t     = shots[shots['contestant_id'] == team_id].copy()
        pitch = VerticalPitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                              linewidth=0.9, half=True, pad_top=4, pad_bottom=2,
                              goal_type='box', goal_alpha=0.6)
        pitch.draw(ax=ax)

        for _, row in t.iterrows():
            pot     = float(row.get('p_ontarget', 0) or 0)
            is_goal = (row.get('outcome') == 'Goal')
            is_ot   = bool(row.get('is_on_target', 0))
            sz      = 60 + pot * 900
            marker  = '*' if is_goal else ('o' if is_ot else 'X')
            ec      = 'white' if is_goal else (color if is_ot else '#444')
            lw      = 1.5 if is_goal else (1.0 if is_ot else 0.4)
            ax.scatter(row['y'], row['x'], s=sz if not is_goal else sz*1.8,
                       marker=marker, color=cmap(norm(pot)),
                       edgecolors=ec, linewidth=lw, zorder=5, alpha=0.92)


        ot_rate  = t['is_on_target'].mean() if len(t) > 0 else 0
        mean_pot = t['p_ontarget'].mean()    if len(t) > 0 else 0
        goals    = int(t['is_goal'].sum())
        ax.text(0.5, 1.03,
                f'{name}   {len(t)} shots  ·  On Target: {ot_rate*100:.0f}%  ·  Mean P(OT): {mean_pot:.2f}  ·  Goals: {goals}',
                ha='center', va='bottom', fontsize=9.5, color=color,
                fontweight='bold', transform=ax.transAxes)

    # shared colourbar
    sm = cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes, shrink=0.45, pad=0.02, aspect=30)
    cbar.set_label('P(On Target)', color='#8b9ab0', fontsize=8)
    cbar.ax.tick_params(colors='#6a7a8a', labelsize=7)

    fig.text(0.5, 0.025,
             '● = on target  ·  ✕ = off target  ·  ★ = goal  ·  Colour & size = P(On Target)',
             ha='center', va='bottom', fontsize=8, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── V12: Situation Danger map + category bars ─────────────────────────────────
def plot_situation_danger(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    if 'situation_danger' not in shots.columns:
        print('    [skip] situation_danger not available'); return

    vmax = float(shots['situation_danger'].quantile(0.95))
    norm = Normalize(vmin=0, vmax=max(vmax, 0.01))
    cmap = LinearSegmentedColormap.from_list('danger', ['#0d1117', '#1a1a3e', '#6a0dad', '#e63946', '#ffd700'])

    home_name = id_to_name.get(home_id, 'Home')
    away_name = id_to_name.get(away_id, 'Away')

    fig = plt.figure(figsize=(21, 9), facecolor=BG)
    ax_h  = fig.add_axes([0.03, 0.08, 0.27, 0.74])
    ax_a  = fig.add_axes([0.33, 0.08, 0.27, 0.74])
    ax_b  = fig.add_axes([0.65, 0.12, 0.32, 0.68])

    for ax, team_id, color, tname in [
        (ax_h, home_id, HOME_COL, home_name),
        (ax_a, away_id, AWAY_COL, away_name),
    ]:
        t = shots[shots['contestant_id'] == team_id].copy()
        pitch = VerticalPitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                              linewidth=0.9, half=True, pad_top=4, pad_bottom=2,
                              goal_type='box', goal_alpha=0.6)
        pitch.draw(ax=ax)

        for _, row in t.iterrows():
            sd      = float(row.get('situation_danger', 0) or 0)
            is_goal = (row.get('outcome') == 'Goal')
            is_ot   = bool(row.get('is_on_target', 0))
            sz      = 55 + sd * 1200
            marker  = '*' if is_goal else 'o'
            ec      = 'white' if is_goal else (color if is_ot else '#444')
            lw      = 1.5 if is_goal else (0.9 if is_ot else 0.3)
            ax.scatter(row['y'], row['x'],
                       s=sz*2 if is_goal else sz,
                       marker=marker, color=cmap(norm(sd)),
                       edgecolors=ec, linewidth=lw, zorder=5, alpha=0.92)

        mean_sd     = t['situation_danger'].mean() if len(t) > 0 else 0
        high_danger = (t['situation_danger'] > 0.15).sum() if len(t) > 0 else 0
        goals       = int(t['is_goal'].sum())
        ax.text(0.5, 1.03,
                f'{tname}   Mean: {mean_sd:.3f}  ·  High-danger: {high_danger}  ·  Goals: {goals}',
                ha='center', va='bottom', fontsize=9, color=color,
                fontweight='bold', transform=ax.transAxes)

    # Situation danger bars
    ax_b.set_facecolor(BG)
    for spine in ax_b.spines.values(): spine.set_color(PITCH_LINE)
    ax_b.tick_params(colors='#6a7a8a')

    sit_cols = {'Big Chance':'is_big_chance','Fast Break':'is_fast_break',
                'From Corner':'is_from_corner','Free Kick':'is_free_kick',
                'Open Play':'is_open_play','Pull-back':'is_pull_back','Set Piece':'is_set_piece'}
    ht = shots[shots['contestant_id']==home_id]
    at = shots[shots['contestant_id']==away_id]
    labels = list(sit_cols.keys())
    h_sd = [ht[ht[c]==1]['situation_danger'].mean() if c in ht.columns and (ht[c]==1).sum()>0 else 0
            for c in sit_cols.values()]
    a_sd = [at[at[c]==1]['situation_danger'].mean() if c in at.columns and (at[c]==1).sum()>0 else 0
            for c in sit_cols.values()]
    y_pos = np.arange(len(labels)); w = 0.36
    ax_b.barh(y_pos + w/2, h_sd, w, color=HOME_COL, alpha=0.82, label=home_name)
    ax_b.barh(y_pos - w/2, a_sd, w, color=AWAY_COL, alpha=0.82, label=away_name)
    ax_b.set_yticks(y_pos)
    ax_b.set_yticklabels(labels, fontsize=9, color='white')
    ax_b.set_xlabel('Mean Situation Danger', color='#8b9ab0', fontsize=9)
    ax_b.set_title('Danger by Situation Type', color='#9ab0c0', fontsize=10, pad=8)
    ax_b.legend(fontsize=8.5, facecolor=BG, labelcolor='white',
                framealpha=0.35, edgecolor=PITCH_LINE)

    # shared colourbar
    sm = cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    cax = fig.add_axes([0.305, 0.12, 0.008, 0.68])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label('Situation Danger', color='#8b9ab0', fontsize=8)
    cbar.ax.tick_params(colors='#6a7a8a', labelsize=7)

    fig.text(0.5, 0.025, '★ = goal  ·  Colour & size = situation danger score',
             ha='center', va='bottom', fontsize=8, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── V13: Situation Danger flow (cumulative danger by minute) ──────────────────
def plot_danger_flow(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    if 'situation_danger' not in shots.columns:
        print('    [skip] situation_danger not available'); return

    fig, ax = plt.subplots(figsize=(14, 5), facecolor=BG)
    ax.set_facecolor(BG)

    for team_id, color in [(home_id, HOME_COL), (away_id, AWAY_COL)]:
        name = id_to_name.get(team_id, str(team_id))
        t = shots[shots['contestant_id']==team_id].sort_values('time_min').copy()
        t['cum_danger'] = t['situation_danger'].cumsum()
        ax.step(t['time_min'], t['cum_danger'], where='post', color=color, lw=2.5,
                label=f'{name} ({t["situation_danger"].sum():.2f} total)')
        goals_t = t[t['is_goal']==1]
        ax.scatter(goals_t['time_min'], goals_t['cum_danger'],
                   color=color, s=130, zorder=5, marker='*',
                   edgecolors='white', linewidth=0.8)

    ax.axvline(45, color='white', lw=0.8, alpha=0.25, ls='--', label='HT')
    ax.set_xlabel('Minute', color='white', fontsize=10)
    ax.set_ylabel('Cumulative Situation Danger', color='white', fontsize=10)
    ax.tick_params(colors='white')
    for spine in ax.spines.values(): spine.set_color(PITCH_LINE)
    ax.legend(fontsize=10, facecolor=BG, labelcolor='white', framealpha=0.4)

    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── V14: P(On Target) actual vs expected by shot type ────────────────────────
def plot_pontarget_by_type(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    if 'p_ontarget' not in shots.columns:
        print('    [skip] p_ontarget not available'); return

    type_cols = {'Open Play':'is_open_play','Header':'is_header','Set Piece':'is_set_piece',
                 'Corner':'is_from_corner','Fast Break':'is_fast_break',
                 'Free Kick':'is_free_kick','Volley':'is_volley','First Time':'is_first_time'}

    home_name = id_to_name.get(home_id,'Home')
    away_name = id_to_name.get(away_id,'Away')

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor=BG)
    fig.suptitle(f'{match_title}  —  P(On Target): Expected vs Actual by Shot Type',
                 color='white', fontsize=12, fontweight='bold')

    for ax, team_id, color, name in [(axes[0], home_id, HOME_COL, home_name),
                                      (axes[1], away_id, AWAY_COL, away_name)]:
        ax.set_facecolor(BG)
        labels, actual, expected = [], [], []
        t = shots[shots['contestant_id']==team_id]
        for lbl, col in type_cols.items():
            sub = t[t[col]==1] if col in t.columns else pd.DataFrame()
            if len(sub) < 1: continue
            labels.append(f'{lbl}\n(n={len(sub)})')
            actual.append(sub['is_on_target'].mean())
            expected.append(sub['p_ontarget'].mean())

        if not labels:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', color='white',
                    transform=ax.transAxes); continue

        x = np.arange(len(labels)); w = 0.35
        ax.bar(x - w/2, actual,   w, color=color, alpha=0.85, label='Actual OT%')
        ax.bar(x + w/2, expected, w, color=color, alpha=0.40, label='Mean P(OT)')
        ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8, color='white', rotation=15, ha='right')
        ax.set_ylabel('Rate', color='white', fontsize=9)
        ax.set_ylim(0, 1.05)
        ax.set_title(name, color=color, fontsize=11, fontweight='bold')
        ax.tick_params(colors='white')
        for spine in ax.spines.values(): spine.set_color(PITCH_LINE)
        ax.legend(fontsize=9, facecolor=BG, labelcolor='white', framealpha=0.4)
        ax.axhline(t['is_on_target'].mean(), color='white', lw=0.8, ls=':', alpha=0.4,
                   label='Overall OT%')

    fig.subplots_adjust(top=0.84, bottom=0.09, left=0.07, right=0.97)
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── V15: xGOT shot map (on-target shots only) ────────────────────────────────
def plot_xgot_map(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    if 'xgot' not in shots.columns:
        print('    [skip] xgot not available'); return

    on_target = shots[shots['is_on_target']==1].copy()
    if len(on_target) == 0:
        print('    [skip] no on-target shots'); return

    fig, axes = plt.subplots(1, 2, figsize=(18, 9), facecolor=BG)
    fig.suptitle(f'{match_title}  —  xGOT  (on-target shots only)',
                 color='white', fontsize=13, fontweight='bold', y=1.01)

    norm = Normalize(vmin=0, vmax=1)
    cmap = cm.get_cmap('plasma')

    for ax, team_id, color in [(axes[0], home_id, HOME_COL), (axes[1], away_id, AWAY_COL)]:
        name = id_to_name.get(team_id, str(team_id))
        t    = on_target[on_target['contestant_id']==team_id].copy()
        pitch = VerticalPitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                              half=True, pad_top=4, pad_bottom=2)
        pitch.draw(ax=ax)
        for _, row in t.iterrows():
            xgot_val = row.get('xgot', 0) or 0
            sz = max(60, xgot_val * 2000)
            ax.scatter(row['y'], row['x'], s=sz, color=cmap(norm(xgot_val)),
                       edgecolors='white' if row['outcome']=='Goal' else '#667788',
                       linewidth=2.5 if row['outcome']=='Goal' else 0.8,
                       zorder=5, alpha=0.92)
            if xgot_val >= 0.35 or row['outcome'] == 'Goal':
                player = str(row.get('player_name', '')).split()[-1] if row.get('player_name') else ''
                ax.annotate(f"{player}\n{xgot_val:.2f}",
                            xy=(row['y'], row['x']),
                            xytext=(row['y'], row['x'] + 5),
                            fontsize=6.0, color='white', ha='center', va='bottom',
                            zorder=8,
                            arrowprops=dict(arrowstyle='-', color='white', lw=0.4, alpha=0.4))

        mean_xgot = t['xgot'].mean() if len(t)>0 else 0
        goals      = int(t['is_goal'].sum())
        ax.set_title(f'{name}\n{len(t)} on-target  |  Goals: {goals}  |  Mean xGOT: {mean_xgot:.2f}',
                     color=color, fontsize=10, fontweight='bold')
        sm = cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
        plt.colorbar(sm, ax=ax, label='xGOT', shrink=0.55, pad=0.02)

    fig.subplots_adjust(top=0.84, bottom=0.09, left=0.07, right=0.97)
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── V16: Multi-outcome probability breakdown ──────────────────────────────────
def plot_multi_outcome(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    mo_cols = ['p_miss','p_post','p_save','p_goal_mo']
    mo_labels = ['Miss','Post','Save','Goal']
    mo_colors = [C_MUTED, '#ffffff', C_ORANGE, C_YELLOW]

    if not any(c in shots.columns for c in mo_cols):
        print('    [skip] multi-outcome model not available'); return

    fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor=BG)
    fig.suptitle(f'{match_title}  —  Multi-Outcome Probabilities per Shot',
                 color='white', fontsize=12, fontweight='bold')

    for ax, team_id, color in [(axes[0], home_id, HOME_COL), (axes[1], away_id, AWAY_COL)]:
        name = id_to_name.get(team_id, str(team_id))
        t = shots[shots['contestant_id']==team_id].copy().reset_index(drop=True)
        ax.set_facecolor(BG)

        avail = [c for c in mo_cols if c in t.columns]
        if not avail:
            ax.text(0.5,0.5,'No data',ha='center',va='center',color='white',transform=ax.transAxes)
            continue

        # stacked horizontal bar per shot, sorted by xG
        t_s = t.sort_values('xg', ascending=True).reset_index(drop=True)
        bottom = np.zeros(len(t_s))
        for col, lbl, c in zip(mo_cols, mo_labels, mo_colors):
            if col not in t_s.columns: continue
            vals = t_s[col].fillna(0).values
            ax.barh(range(len(t_s)), vals, left=bottom, color=c, alpha=0.85,
                    label=lbl, height=0.8)
            bottom += vals

        # mark goals
        for i, row in t_s.iterrows():
            if row.get('is_goal',0):
                ax.text(1.01, i, '★', va='center', fontsize=9, color=C_YELLOW,
                        transform=ax.get_yaxis_transform())

        ax.set_xlim(0, 1)
        ax.set_yticks(range(len(t_s)))
        ax.set_yticklabels([f"{r['player_name'].split(' ')[-1]} ({r['xg']:.2f})"
                            for _, r in t_s.iterrows()], fontsize=7, color='white')
        ax.set_xlabel('Probability', color='white', fontsize=9)
        ax.set_title(name, color=color, fontsize=11, fontweight='bold')
        ax.tick_params(colors='white')
        for spine in ax.spines.values(): spine.set_color(PITCH_LINE)

    handles = [mpatches.Patch(color=c, label=l) for c, l in zip(mo_colors, mo_labels)]
    axes[0].legend(handles=handles, fontsize=9, facecolor=BG, labelcolor='white',
                   framealpha=0.4, loc='lower right')
    fig.subplots_adjust(top=0.84, bottom=0.09, left=0.07, right=0.97)
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── V17: Danger vs P(On Target) bubble chart ─────────────────────────────────
def plot_danger_vs_pontarget(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    needed = {'situation_danger','p_ontarget'}
    if not needed.issubset(shots.columns):
        print('    [skip] situation_danger or p_ontarget not available'); return

    fig, ax = plt.subplots(figsize=(13, 8), facecolor=BG)
    ax.set_facecolor(BG)

    for team_id, color, marker in [(home_id, HOME_COL, 'o'), (away_id, AWAY_COL, 's')]:
        name = id_to_name.get(team_id, str(team_id))
        t = shots[shots['contestant_id']==team_id].copy()
        for _, row in t.iterrows():
            oc  = row.get('outcome','Missed')
            col = OUTCOME_COLORS.get(oc, C_MUTED)
            sz  = max(40, row['xg'] * 1800)
            ax.scatter(row['situation_danger'], row['p_ontarget'],
                       s=sz, color=col, marker=marker,
                       edgecolors=color, linewidth=1.5 if oc=='Goal' else 0.5,
                       alpha=0.85, zorder=4)
        ax.scatter([], [], color=color, marker=marker, s=80, label=name)

    ax.axvline(shots['situation_danger'].mean(), color='white', lw=0.7, ls=':', alpha=0.4)
    ax.axhline(shots['p_ontarget'].mean(),       color='white', lw=0.7, ls=':', alpha=0.4)
    ax.set_xlabel('Situation Danger', color='white', fontsize=10)
    ax.set_ylabel('P(On Target)', color='white', fontsize=10)
    ax.tick_params(colors='white')
    for spine in ax.spines.values(): spine.set_color(PITCH_LINE)

    outcome_handles = [mpatches.Patch(color=c, label=l) for l, c in OUTCOME_COLORS.items()]
    team_handles    = [mpatches.Patch(color=HOME_COL, label=id_to_name.get(home_id,'Home'),
                                      hatch='///'),
                       mpatches.Patch(color=AWAY_COL, label=id_to_name.get(away_id,'Away'))]
    leg1 = ax.legend(handles=outcome_handles, fontsize=8, facecolor=BG, labelcolor='white',
                     framealpha=0.4, loc='upper left', title='Outcome', title_fontsize=8)
    ax.add_artist(leg1)
    ax.legend(handles=team_handles, fontsize=8, facecolor=BG, labelcolor='white',
              framealpha=0.4, loc='lower right', title='Team  (○=Home  □=Away)', title_fontsize=8)

    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── V18: Shot zone grid (pitch thirds × width bands) ─────────────────────────
def plot_shot_zone_grid(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    """
    Divides the attacking half into a 3×3 zone grid and shows mean xG,
    P(OT), and Situation Danger per zone for each team.
    """
    def zone(x, y):
        col = 0 if y < 38 else (2 if y > 62 else 1)   # left / central / right
        row = 0 if x < 83 else (2 if x >= 94 else 1)   # long / mid / close
        return row, col

    zone_names_row = ['Long range\n(< 83)', 'Mid range\n(83–94)', 'Close range\n(≥ 94)']
    zone_names_col = ['Left', 'Central', 'Right']

    fig, axes = plt.subplots(2, 3, figsize=(18, 10), facecolor=BG)
    fig.suptitle(f'{match_title}  —  Shot Zone Grid  (mean xG | P(OT) | Danger)',
                 color='white', fontsize=13, fontweight='bold')

    metrics = ['xg']
    if 'p_ontarget'      in shots.columns: metrics.append('p_ontarget')
    if 'situation_danger' in shots.columns: metrics.append('situation_danger')

    for row_idx, team_id in enumerate([home_id, away_id]):
        color = HOME_COL if team_id == home_id else AWAY_COL
        name  = id_to_name.get(team_id, str(team_id))
        t     = shots[shots['contestant_id']==team_id].copy()

        for col_idx in range(3):
            ax = axes[row_idx][col_idx]
            ax.set_facecolor(BG)
            ax.set_xlim(-0.5, 2.5); ax.set_ylim(-0.5, 2.5)
            ax.set_xticks([0,1,2]); ax.set_yticks([0,1,2])
            ax.set_xticklabels(zone_names_col, fontsize=7, color='white')
            ax.set_yticklabels(zone_names_row, fontsize=7, color='white')
            for spine in ax.spines.values(): spine.set_color(PITCH_LINE)
            ax.tick_params(colors='white')
            if col_idx == 0:
                ax.set_ylabel(name, color=color, fontsize=10, fontweight='bold')
            if row_idx == 0:
                metric_label = ['xG', 'P(On Target)', 'Situation Danger'][col_idx] if col_idx < len(metrics) else '—'
                ax.set_title(metric_label, color='white', fontsize=10)

            metric = metrics[col_idx] if col_idx < len(metrics) else None
            if metric is None:
                ax.text(1, 1, 'N/A', ha='center', va='center', color='grey',
                        fontsize=12, transform=ax.transData)
                continue

            for r in range(3):
                for c in range(3):
                    mask = t.apply(lambda row: zone(row['x'], row['y'])==(r,c), axis=1)
                    sub  = t[mask]
                    n    = len(sub)
                    val  = sub[metric].mean() if n > 0 else 0
                    alpha = min(0.9, val * 4 + 0.15) if val > 0 else 0.05
                    rect = plt.Rectangle((c-0.5, r-0.5), 1, 1,
                                         facecolor=color, alpha=alpha, zorder=1)
                    ax.add_patch(rect)
                    ax.text(c, r, f'{val:.2f}\n(n={n})', ha='center', va='center',
                            fontsize=9, color='white', fontweight='bold', zorder=2)

    fig.subplots_adjust(top=0.84, bottom=0.09, left=0.07, right=0.97)
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── V19: Model radar / spider chart ──────────────────────────────────────────
def plot_model_radar(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    home_name = id_to_name.get(home_id,'Home')
    away_name = id_to_name.get(away_id,'Away')

    def team_vals(tid):
        t = shots[shots['contestant_id']==tid]
        n = max(len(t), 1)
        return {
            'xG / shot':            t['xg'].mean()               if 'xg'               in t else 0,
            'psxG / shot':          t['psxg'].mean()              if 'psxg'             in t else 0,
            'P(On Target)':         t['p_ontarget'].mean()        if 'p_ontarget'       in t else 0,
            'Situation Danger':     t['situation_danger'].mean()  if 'situation_danger' in t else 0,
            'xGOT / OT shot':       t[t['is_on_target']==1]['xgot'].mean()
                                    if 'xgot' in t and t['is_on_target'].sum()>0 else 0,
            'Big Chance %':         t['is_big_chance'].mean()     if 'is_big_chance'    in t else 0,
            'Actual OT %':          t['is_on_target'].mean(),
            'Conversion %':         t['is_goal'].mean(),
        }

    hv = team_vals(home_id)
    av = team_vals(away_id)
    labels = list(hv.keys())
    N = len(labels)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    # Normalise to 0–1 across both teams per dimension
    def norm_vals(d):
        all_max = {k: max(hv[k], av[k], 1e-9) for k in labels}
        return [d[k] / all_max[k] for k in labels]

    h_norm = norm_vals(hv) + [norm_vals(hv)[0]]
    a_norm = norm_vals(av) + [norm_vals(av)[0]]

    fig, ax = plt.subplots(figsize=(9, 9), facecolor=BG,
                           subplot_kw=dict(polar=True))
    ax.set_facecolor(BG)
    ax.spines['polar'].set_color(PITCH_LINE)
    ax.tick_params(colors='white')

    ax.plot(angles, h_norm, color=HOME_COL, lw=2.5, label=home_name)
    ax.fill(angles, h_norm, color=HOME_COL, alpha=0.18)
    ax.plot(angles, a_norm, color=AWAY_COL, lw=2.5, label=away_name)
    ax.fill(angles, a_norm, color=AWAY_COL, alpha=0.18)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(
        [f'{l}\nH:{hv[l]:.3f}  A:{av[l]:.3f}' for l in labels],
        fontsize=7.5, color='white'
    )
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['25%','50%','75%','100%'], color='#7a9ab0', fontsize=7)
    ax.set_ylim(0, 1)
    ax.grid(color=PITCH_LINE, alpha=0.5)

    ax.legend(fontsize=10, facecolor=BG, labelcolor='white', framealpha=0.4,
              loc='upper right', bbox_to_anchor=(1.28, 1.12))

    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── V20: Full model summary dashboard ────────────────────────────────────────
def plot_model_summary(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    home_name = id_to_name.get(home_id,'Home')
    away_name = id_to_name.get(away_id,'Away')

    def row_data(tid):
        t = shots[shots['contestant_id']==tid]
        ot = t[t['is_on_target']==1]
        return {
            'Shots':             len(t),
            'Goals':             int(t['is_goal'].sum()),
            'xG':                f"{t['xg'].sum():.2f}",
            'G − xG':            f"{t['is_goal'].sum()-t['xg'].sum():+.2f}",
            'psxG':              f"{t['psxg'].sum():.2f}",
            'G − psxG':          f"{t['is_goal'].sum()-t['psxg'].sum():+.2f}",
            'Actual OT':         int(t['is_on_target'].sum()),
            'Σ P(OT)':           f"{t['p_ontarget'].sum():.2f}" if 'p_ontarget' in t else '—',
            'OT − Σ P(OT)':      f"{t['is_on_target'].sum()-t['p_ontarget'].sum():+.2f}"
                                 if 'p_ontarget' in t else '—',
            'Σ Danger':          f"{t['situation_danger'].sum():.2f}" if 'situation_danger' in t else '—',
            'High-Danger (>0.15)': int((t['situation_danger']>0.15).sum()) if 'situation_danger' in t else '—',
            'xGOT (OT shots)':   f"{ot['xgot'].sum():.2f}" if 'xgot' in ot.columns and len(ot)>0 else '—',
            'G − xGOT':          f"{ot['is_goal'].sum()-ot['xgot'].sum():+.2f}"
                                 if 'xgot' in ot.columns and len(ot)>0 else '—',
            'Big Chances':       int(t['is_big_chance'].sum()),
            'Fast Breaks':       int(t['is_fast_break'].sum()),
            'Mean xG/shot':      f"{t['xg'].mean():.3f}",
            'Mean P(OT)':        f"{t['p_ontarget'].mean():.3f}" if 'p_ontarget' in t else '—',
            'Mean Danger':       f"{t['situation_danger'].mean():.3f}" if 'situation_danger' in t else '—',
        }

    hd = row_data(home_id)
    ad = row_data(away_id)
    metrics = list(hd.keys())

    fig, ax = plt.subplots(figsize=(13, 9), facecolor=BG)
    ax.set_facecolor(BG); ax.axis('off')
    ax.text(0.5, 0.97, match_title, ha='center', va='top', fontsize=16,
            color='white', fontweight='bold', transform=ax.transAxes)
    ax.text(0.5, 0.92, 'Full Model Summary — All Predictions vs Actuals',
            ha='center', va='top', fontsize=10, color='#7a9ab0', transform=ax.transAxes)

    n = len(metrics)
    row_h = 0.82 / n
    for i, metric in enumerate(metrics):
        y = 0.88 - i * row_h
        hv_raw = hd[metric]; av_raw = ad[metric]

        if i % 2 == 0:
            ax.add_patch(mpatches.FancyBboxPatch(
                (0.01, y - row_h*0.45), 0.98, row_h*0.9,
                boxstyle='round,pad=0.005', facecolor='white', alpha=0.04,
                transform=ax.transAxes, clip_on=False))

        def is_better(hv, av):
            try:
                return float(str(hv).replace('+','')) > float(str(av).replace('+',''))
            except: return False

        hcol = HOME_COL if is_better(hv_raw, av_raw) else 'white'
        acol = AWAY_COL if is_better(av_raw, hv_raw) else 'white'

        ax.text(0.14, y, str(hv_raw), ha='center', va='center', fontsize=11,
                color=hcol, fontweight='bold', transform=ax.transAxes)
        ax.text(0.50, y, metric,      ha='center', va='center', fontsize=8.5,
                color='#a0aab0',      transform=ax.transAxes)
        ax.text(0.86, y, str(av_raw), ha='center', va='center', fontsize=11,
                color=acol, fontweight='bold', transform=ax.transAxes)

    ax.text(0.14, 0.89, home_name, ha='center', va='center', fontsize=11,
            color=HOME_COL, fontweight='bold', transform=ax.transAxes)
    ax.text(0.86, 0.89, away_name, ha='center', va='center', fontsize=11,
            color=AWAY_COL, fontweight='bold', transform=ax.transAxes)

    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


print('V11 P(On Target) map ✓')
print('V13 Danger flow ✓')
print('V15 xGOT map ✓')
print('V17 Danger vs P(OT) bubble ✓')
print('V19 Model radar ✓')
print('\nAll 20 visual functions ready ✓')

# ── EVENT DATA LOADER ─────────────────────────────────────────────────────────
# Opta qualifier IDs used for event visuals
PASS_END_X   = 140   # pass end x
PASS_END_Y   = 141   # pass end y

# Event typeIds
TID_PASS    = 1
TID_TAKEON  = 3
TID_TACKLE  = 7
TID_INTERC  = 8
TID_CLEAR   = 12
TID_SHOT    = {13, 14, 15, 16}
TID_TOUCH   = 44
TID_RECOVERY= 49

POSSESSION_TYPES = {1, 3, 13, 14, 15, 16, 44, 49}   # events where a player has the ball
DEFENSIVE_TYPES  = {7, 8, 12, 49}                    # defensive actions

def load_opta_events(path):
    """Load ALL events from an Opta match JSON into a flat DataFrame."""
    with open(path, 'r', encoding='utf-8-sig') as f:
        data = json.load(f)
    events = data.get('liveData', {}).get('event', data.get('event', []))
    rows = []
    for e in events:
        tid = int(e.get('typeId', 0))
        x   = safe_float(e.get('x'))
        y   = safe_float(e.get('y'))
        rows.append({
            'type_id':       tid,
            'outcome':       int(e.get('outcome', 0) or 0),
            'x': x, 'y': y,
            'end_x':         safe_float(get_q(e, PASS_END_X)),
            'end_y':         safe_float(get_q(e, PASS_END_Y)),
            'player_id':     str(e.get('playerId',  '')),
            'player_name':   str(e.get('playerName','Unknown')),
            'contestant_id': str(e.get('contestantId','')),
            'period_id':     int(e.get('periodId', 0) or 0),
            'time_min':      int(e.get('timeMin',  0) or 0),
            'time_sec':      int(e.get('timeSec',  0) or 0),
        })
    df = pd.DataFrame(rows)
    if len(df):
        df['abs_min'] = df['time_min'] + df['time_sec'] / 60.0
    return df


def load_shirt_numbers(path):
    """Return {player_id: shirt_number_str} from matchInfo.contestant[].player[]."""
    with open(path, 'r', encoding='utf-8-sig') as f:
        data = json.load(f)
    lookup = {}
    for contestant in data.get('matchInfo', {}).get('contestant', []):
        for player in contestant.get('player', []):
            pid = str(player.get('id', player.get('playerId', '')))
            shirt = str(player.get('shirtNumber', player.get('shirt_number', '')))
            if pid and shirt:
                lookup[pid] = shirt
    return lookup


# ══════════════════════════════════════════════════════════════════════════════
# EVENT-BASED VISUALS
# ══════════════════════════════════════════════════════════════════════════════

# ── E1: Pass Map ──────────────────────────────────────────────────────────────
def plot_pass_map(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    name = id_to_name.get(team_id, str(team_id))
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    t = events[(events['contestant_id'] == team_id) & (events['type_id'] == TID_PASS)].copy()
    t = t.dropna(subset=['x','y','end_x','end_y'])
    completed = t[t['outcome'] == 1]
    incomplete = t[t['outcome'] == 0]

    fig = plt.figure(figsize=(14, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.9, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    # Incomplete passes — thin, very faded
    for _, row in incomplete.iterrows():
        ax.annotate('', xy=(row['end_x'], row['end_y']), xytext=(row['x'], row['y']),
                    arrowprops=dict(arrowstyle='->', color=color, alpha=0.15, lw=0.7))
    # Complete passes — bright
    for _, row in completed.iterrows():
        ax.annotate('', xy=(row['end_x'], row['end_y']), xytext=(row['x'], row['y']),
                    arrowprops=dict(arrowstyle='->', color=color, alpha=0.75, lw=0.9))

    fig.text(0.5, 0.028,
             f'{len(t)} passes  ·  {len(completed)} completed ({len(completed)/max(len(t),1)*100:.0f}%)'
             f'  ·  Bright=completed  ·  Faded=incomplete',
             ha='center', va='bottom', fontsize=8.5, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── E2: Pass Network ──────────────────────────────────────────────────────────
def plot_pass_network(events, home_id, away_id, id_to_name, match_title, subtitle='',
                      save_path=None, shirt_numbers=None):
    shirt_numbers = shirt_numbers or {}
    passes = events[(events['type_id'] == TID_PASS) & (events['outcome'] == 1)].copy()
    passes = passes.dropna(subset=['x','y','end_x','end_y'])

    fig, axes = plt.subplots(1, 2, figsize=(18, 8), facecolor=BG)
    fig.subplots_adjust(top=0.84, bottom=0.07, left=0.02, right=0.98, wspace=0.08)

    for ax, team_id, color in [(axes[0], home_id, HOME_COL), (axes[1], away_id, AWAY_COL)]:
        name = id_to_name.get(team_id, str(team_id))
        t = passes[passes['contestant_id'] == team_id].copy()
        pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                      linewidth=0.8, goal_type='box', goal_alpha=0.5)
        pitch.draw(ax=ax)

        if len(t) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', color='white',
                    transform=ax.transAxes)
            continue

        grp = t.groupby('player_id').agg(
            avg_x=('x', 'mean'), avg_y=('y', 'mean'),
            n=('x', 'count'), player_name=('player_name', 'first')
        ).reset_index()

        pid_to_xy = dict(zip(grp['player_id'], zip(grp['avg_x'], grp['avg_y'])))

        # Pass pair connections (thickness = pass volume)
        pair_counts = Counter()
        for _, row in t.iterrows():
            best_pid = None; best_d = 999
            for pid, (px, py) in pid_to_xy.items():
                if pid == row['player_id']: continue
                d = (px - row['end_x'])**2 + (py - row['end_y'])**2
                if d < best_d: best_d = d; best_pid = pid
            if best_pid:
                pair_counts[tuple(sorted([row['player_id'], best_pid]))] += 1

        max_pairs = max(pair_counts.values(), default=1)
        for (pid1, pid2), cnt in pair_counts.items():
            if pid1 not in pid_to_xy or pid2 not in pid_to_xy or cnt < 2: continue
            x1, y1 = pid_to_xy[pid1]; x2, y2 = pid_to_xy[pid2]
            ax.plot([x1, x2], [y1, y2], color=color,
                    lw=0.8 + 4.5 * cnt / max_pairs, alpha=0.50, zorder=2)

        # Node size = pass involvement; label = jersey number
        max_n = grp['n'].max()
        for _, row in grp.iterrows():
            sz    = 180 + 700 * row['n'] / max_n
            shirt = shirt_numbers.get(str(row['player_id']), '')
            short = row['player_name'].split(' ')[-1]
            ax.scatter(row['avg_x'], row['avg_y'], s=sz, color=color,
                       edgecolors='white', linewidth=1.5, zorder=4, alpha=0.92)
            # Jersey number inside node
            lbl = shirt if shirt else short[:2]
            ax.text(row['avg_x'], row['avg_y'], lbl,
                    ha='center', va='center', fontsize=7, color='white',
                    fontweight='bold', zorder=5)
            # Name below node
            ax.text(row['avg_x'], row['avg_y'] - 3.8, short,
                    ha='center', va='top', fontsize=5.5, color='white', alpha=0.75, zorder=5)

        ax.text(0.5, -0.02, name, ha='center', va='top', fontsize=10,
                color=color, fontweight='bold', transform=ax.transAxes)

    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── E2b: Pass Network — single team ───────────────────────────────────────────
def plot_pass_network_single(events, team_id, id_to_name, match_title, subtitle='', save_path=None, shirt_numbers=None):
    """Pass network for one team on a full pitch with player legend sidebar."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))
    passes = events[(events['type_id'] == TID_PASS) & (events['outcome'] == 1) &
                    (events['contestant_id'] == team_id)].copy()
    passes = passes.dropna(subset=['x','y','end_x','end_y'])

    fig = plt.figure(figsize=(14, 9), facecolor=BG)
    ax  = fig.add_axes([0.03, 0.06, 0.68, 0.78])
    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.8, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    grp = passes.groupby('player_id').agg(
        avg_x=('x','mean'), avg_y=('y','mean'),
        n=('x','count'), player_name=('player_name','first')
    ).reset_index()
    pid_to_xy = dict(zip(grp['player_id'], zip(grp['avg_x'], grp['avg_y'])))

    pair_counts = Counter()
    for _, row in passes.iterrows():
        best_pid = None; best_d = 999
        for pid, (px, py) in pid_to_xy.items():
            if pid == row['player_id']: continue
            d = (px - row['end_x'])**2 + (py - row['end_y'])**2
            if d < best_d: best_d = d; best_pid = pid
        if best_pid:
            pair_counts[tuple(sorted([row['player_id'], best_pid]))] += 1

    max_pairs = max(pair_counts.values(), default=1)
    for (pid1, pid2), cnt in pair_counts.items():
        if pid1 not in pid_to_xy or pid2 not in pid_to_xy or cnt < 2: continue
        x1, y1 = pid_to_xy[pid1]; x2, y2 = pid_to_xy[pid2]
        ax.plot([x1, x2], [y1, y2], color=color,
                lw=0.8 + 4.0 * cnt / max_pairs, alpha=0.55, zorder=2)

    max_n = grp['n'].max()
    for _, row in grp.iterrows():
        sz    = 180 + 700 * row['n'] / max_n
        shirt = shirt_numbers.get(str(row['player_id']), '') if shirt_numbers else ''
        short = row['player_name'].split(' ')[-1]
        ax.scatter(row['avg_x'], row['avg_y'], s=sz, color=color,
                   edgecolors='white', linewidth=1.5, zorder=4, alpha=0.92)
        lbl = shirt if shirt else short[:2]
        ax.text(row['avg_x'], row['avg_y'], lbl,
                ha='center', va='center', fontsize=7, color='white',
                fontweight='bold', zorder=5)

    # Legend sidebar
    ax_leg = fig.add_axes([0.73, 0.06, 0.25, 0.78])
    ax_leg.set_facecolor(BG); ax_leg.axis('off')
    ax_leg.text(0.05, 0.97, name, ha='left', va='top', fontsize=11,
                color=color, fontweight='bold', transform=ax_leg.transAxes)
    ax_leg.text(0.35, 0.97, '#  Player', ha='left', va='top', fontsize=8,
                color='#7a9ab0', transform=ax_leg.transAxes)
    sorted_grp = grp.sort_values('n', ascending=False).reset_index(drop=True)
    for i, row in sorted_grp.iterrows():
        y_pos = 0.90 - i * 0.055
        if y_pos < 0.02: break
        short = row['player_name']
        ax_leg.scatter([0.05], [y_pos], s=50, color=color,
                       edgecolors='white', linewidth=0.8, transform=ax_leg.transAxes,
                       zorder=4, clip_on=False)
        ax_leg.text(0.14, y_pos, short, ha='left', va='center', fontsize=7.5,
                    color='white', transform=ax_leg.transAxes)
        ax_leg.text(0.88, y_pos, str(int(row['n'])), ha='right', va='center',
                    fontsize=7.5, color=color, transform=ax_leg.transAxes)

    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── E3: Field Tilt ────────────────────────────────────────────────────────────
def plot_field_tilt(events, shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    """Share of final-third touches belonging to home team, 5-min rolling window."""
    # Use possession events (passes, shots, touches)
    poss = events[events['type_id'].isin(POSSESSION_TYPES)].copy()
    poss = poss.dropna(subset=['x','time_min'])

    # Final third = x > 67 (both teams attack toward x=100 in normalised Opta)
    final_third = poss[poss['x'] > 67].copy()
    final_third['home'] = (final_third['contestant_id'] == home_id).astype(int)

    fig, ax = plt.subplots(figsize=(14, 5.5), facecolor=BG)
    ax.set_facecolor(BG)

    max_min = max(int(events['time_min'].max()) + 3, 95)
    minutes = np.arange(0, max_min + 1)

    # 5-min rolling tilt per minute
    tilt_series = []
    for m in minutes:
        window = final_third[(final_third['time_min'] >= m - 5) & (final_third['time_min'] <= m)]
        n_total = len(window)
        n_home  = window['home'].sum()
        tilt_series.append((n_home / n_total * 100) if n_total >= 3 else np.nan)

    tilt_arr = np.array(tilt_series, dtype=float)

    ax.axhline(50, color='white', lw=0.8, alpha=0.20, ls='--')

    # Fill: above 50 = home (orange), below 50 = away (blue)
    ax.fill_between(minutes, tilt_arr, 50,
                    where=(tilt_arr >= 50), interpolate=True,
                    color=HOME_COL, alpha=0.35, zorder=3)
    ax.fill_between(minutes, tilt_arr, 50,
                    where=(tilt_arr <= 50), interpolate=True,
                    color=AWAY_COL, alpha=0.35, zorder=3)
    ax.plot(minutes, tilt_arr, color='white', lw=1.8, zorder=4)

    avg_home = np.nanmean(tilt_arr)
    avg_away = 100 - avg_home
    home_name = id_to_name.get(home_id, 'Home')
    away_name = id_to_name.get(away_id, 'Away')

    ax.text(0.01, 0.96, f'{home_name}  (avg. {avg_home:.0f}%)',
            ha='left', va='top', fontsize=10, color=HOME_COL, fontweight='bold',
            transform=ax.transAxes)
    ax.text(0.01, 0.84, f'{away_name}  (avg. {avg_away:.0f}%)',
            ha='left', va='top', fontsize=10, color=AWAY_COL, fontweight='bold',
            transform=ax.transAxes)

    # Goal markers
    goal_events = shots[shots['is_goal'] == 1].copy()
    for _, row in goal_events.iterrows():
        gc = HOME_COL if row['contestant_id'] == home_id else AWAY_COL
        ax.axvline(row['time_min'], color=gc, lw=1.2, alpha=0.7, zorder=5)
        ax.text(row['time_min'] + 0.5, 97,
                f"Goal {int(row['time_min'])}'",
                fontsize=7.5, color=gc, va='top', fontweight='bold')

    ax.axvline(45, color='white', lw=0.7, alpha=0.18, ls='--')
    ax.set_xlim(0, max_min); ax.set_ylim(0, 100)
    ax.set_xlabel('Minute', color='#8b9ab0', fontsize=9)
    ax.set_ylabel('Field Tilt %', color='#8b9ab0', fontsize=9)
    ax.tick_params(colors='#8b9ab0', labelsize=8)
    for spine in ax.spines.values(): spine.set_color(PITCH_LINE)
    ax.grid(axis='y', color=PITCH_LINE, lw=0.5, alpha=0.30)
    ax.text(0.5, -0.11,
            'Field Tilt = share of final-third touches in a rolling 5-min window',
            ha='center', va='bottom', fontsize=7.5, color='#7a9ab0',
            transform=ax.transAxes)

    fig.subplots_adjust(top=0.86, bottom=0.13, left=0.07, right=0.97)
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── E4: Zones of Control ──────────────────────────────────────────────────────
def plot_zones_of_control(events, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    """Grid of pitch zones coloured by whichever team has more touches."""
    poss = events[events['type_id'].isin(POSSESSION_TYPES)].copy()
    poss = poss.dropna(subset=['x','y'])
    home_name = id_to_name.get(home_id, 'Home')
    away_name = id_to_name.get(away_id, 'Away')

    COLS, ROWS = 5, 7
    x_edges = np.linspace(0, 100, COLS + 1)
    y_edges = np.linspace(0, 100, ROWS + 1)

    fig = plt.figure(figsize=(13, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.9, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    home_cmap = LinearSegmentedColormap.from_list('h', [BG, HOME_COL])
    away_cmap = LinearSegmentedColormap.from_list('a', [BG, AWAY_COL])

    for ci in range(COLS):
        for ri in range(ROWS):
            x0, x1 = x_edges[ci], x_edges[ci+1]
            y0, y1 = y_edges[ri], y_edges[ri+1]
            zone = poss[(poss['x'] >= x0) & (poss['x'] < x1) &
                        (poss['y'] >= y0) & (poss['y'] < y1)]
            nh = (zone['contestant_id'] == home_id).sum()
            na = (zone['contestant_id'] == away_id).sum()
            total = nh + na
            if total == 0: continue

            dom_color = HOME_COL if nh >= na else AWAY_COL
            dom_frac  = max(nh, na) / total
            alpha     = 0.12 + 0.62 * (dom_frac - 0.5) / 0.5 if dom_frac > 0.5 else 0.12
            rect = plt.Rectangle((x0, y0), x1-x0, y1-y0,
                                  facecolor=dom_color, alpha=alpha, zorder=1,
                                  edgecolor='none')
            ax.add_patch(rect)
            cx, cy = (x0+x1)/2, (y0+y1)/2
            ax.text(cx, cy, f'{nh}/{na}', ha='center', va='center',
                    fontsize=6.5, color='white', alpha=0.80, zorder=3)

    fig.text(0.5, 0.028,
             f'Colour = dominant team by touch count per zone  ·  Number = {home_name} / {away_name}',
             ha='center', va='bottom', fontsize=8.5, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── E5: Defensive Actions ─────────────────────────────────────────────────────
def plot_defensive_actions(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    DEF_TYPES = {
        TID_TACKLE:  ('Tackle',       C_BLUE),
        TID_INTERC:  ('Interception', C_GREEN),
        TID_CLEAR:   ('Clearance',    C_PURPLE),
        TID_RECOVERY:('Recovery',     C_YELLOW),
    }
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))

    fig = plt.figure(figsize=(14, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    fig.subplots_adjust(top=0.84, bottom=0.06)

    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.8, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    t = events[events['contestant_id'] == team_id].copy()
    total = 0
    for tid, (lbl, col) in DEF_TYPES.items():
        sub = t[(t['type_id'] == tid) & t['x'].notna() & t['y'].notna()]
        if len(sub) == 0: continue
        won  = sub[sub['outcome'] == 1]
        lost = sub[sub['outcome'] == 0]
        ax.scatter(won['x'],  won['y'],  s=55, color=col,
                   edgecolors='white', linewidth=0.9, alpha=0.88, zorder=4, label=lbl)
        ax.scatter(lost['x'], lost['y'], s=35, color=col,
                   edgecolors='none', linewidth=0.0, alpha=0.35, zorder=3)
        total += len(sub)

    won_total = t[t['type_id'].isin(DEF_TYPES) & (t['outcome'] == 1)].shape[0]
    fig.text(0.5, 0.025,
             f'{total} actions  ·  {won_total} won ({won_total/max(total,1)*100:.0f}%)'
             f'  ·  White border = won  ·  Faded = lost',
             ha='center', va='bottom', fontsize=8, color='#8b9ab0')

    handles = [mpatches.Patch(color=c, label=l) for _, (l, c) in DEF_TYPES.items()]
    ax.legend(handles=handles, fontsize=8.5, facecolor=BG, labelcolor='white',
              framealpha=0.4, edgecolor=PITCH_LINE, loc='lower left', ncol=2)

    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── E6: Defensive Line Height ─────────────────────────────────────────────────
def plot_line_height(events, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    """Rolling median x of each team's defensive actions = line height."""
    def_ev = events[events['type_id'].isin(DEFENSIVE_TYPES)].dropna(subset=['x','time_min'])
    fig, ax = plt.subplots(figsize=(14, 5.5), facecolor=BG)
    ax.set_facecolor(BG)
    max_min = max(int(events['time_min'].max()) + 3, 95)
    minutes = np.arange(0, max_min + 1)

    for team_id, color in [(home_id, HOME_COL), (away_id, AWAY_COL)]:
        name = id_to_name.get(team_id, str(team_id))
        t = def_ev[def_ev['contestant_id'] == team_id].copy()
        series = []
        for m in minutes:
            w = t[(t['time_min'] >= m - 5) & (t['time_min'] <= m)]
            series.append(w['x'].median() if len(w) >= 2 else np.nan)
        arr = np.array(series, dtype=float)
        # Smooth slightly
        from scipy.ndimage import uniform_filter1d
        valid = ~np.isnan(arr)
        if valid.sum() > 5:
            arr[valid] = uniform_filter1d(arr[valid], size=3)
        ax.plot(minutes, arr, color=color, lw=2.2, zorder=4, solid_capstyle='round')
        ax.fill_between(minutes, arr, alpha=0.15, color=color, zorder=3)
        avg = np.nanmean(arr)
        ax.text(0.01, 0.97 if team_id == home_id else 0.86,
                f'{name}  (avg: {avg:.0f})',
                ha='left', va='top', fontsize=10, color=color, fontweight='bold',
                transform=ax.transAxes)

    ax.axvline(45, color='white', lw=0.7, alpha=0.18, ls='--')
    ax.set_xlim(0, max_min)
    ax.set_xlabel('Minute', color='#8b9ab0', fontsize=9)
    ax.set_ylabel('Line Height (x)', color='#8b9ab0', fontsize=9)
    ax.tick_params(colors='#8b9ab0', labelsize=8)
    for spine in ax.spines.values(): spine.set_color(PITCH_LINE)
    ax.grid(axis='y', color=PITCH_LINE, lw=0.5, alpha=0.30)
    ax.text(0.5, -0.12,
            'Higher = defensive line pushed further up  ·  Based on median x of def. actions',
            ha='center', va='bottom', fontsize=7.5, color='#7a9ab0', transform=ax.transAxes)
    fig.subplots_adjust(top=0.86, bottom=0.13, left=0.07, right=0.97)
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── E7: Shape Comparison (in / out of possession) — per team ──────────────────
def plot_shape_comparison(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Average player positions when in vs out of possession for one team."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))

    in_poss  = events[(events['contestant_id'] == team_id) &
                       events['type_id'].isin(POSSESSION_TYPES)].dropna(subset=['x','y'])
    out_poss = events[(events['contestant_id'] != team_id) &
                       events['type_id'].isin(POSSESSION_TYPES)].dropna(subset=['x','y'])

    fig, axes = plt.subplots(1, 2, figsize=(18, 8), facecolor=BG)
    fig.subplots_adjust(top=0.84, bottom=0.05, left=0.02, right=0.98, wspace=0.06)

    for ax, df, lbl in [
        (axes[0], in_poss,  'In Possession'),
        (axes[1], out_poss, 'Out of Possession'),
    ]:
        pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                      linewidth=0.8, goal_type='box', goal_alpha=0.5)
        pitch.draw(ax=ax)
        if len(df) == 0:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                    color='white', transform=ax.transAxes)
            ax.text(0.5, 1.03, lbl, ha='center', va='bottom',
                    fontsize=10, color='white', transform=ax.transAxes)
            continue
        grp = df.groupby('player_id').agg(
            avg_x=('x','mean'), avg_y=('y','mean'),
            n=('x','count'), player_name=('player_name','first')
        ).reset_index()
        max_n = grp['n'].max()
        for _, row in grp.iterrows():
            sz = 60 + 300 * row['n'] / max_n
            ax.scatter(row['avg_x'], row['avg_y'], s=sz, color=color,
                       edgecolors='white', linewidth=1.2, zorder=4, alpha=0.88)
            short = row['player_name'].split(' ')[-1]
            ax.text(row['avg_x'], row['avg_y'] - 3.5, short,
                    ha='center', va='top', fontsize=6, color='white', alpha=0.85, zorder=5)
        ax.text(0.5, 1.03, lbl, ha='center', va='bottom',
                fontsize=10, color='white', transform=ax.transAxes)
        ax.text(0.5, -0.02, f'{len(df)} actions  ·  Node=avg player position',
                ha='center', va='top', fontsize=7.5, color='#7a9ab0', transform=ax.transAxes)

    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── E8: Progressive Passes ────────────────────────────────────────────────────
def plot_progressive_passes(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Successful passes that advance the ball ≥ 10 m toward goal (end_x > x + 10)."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))

    passes = events[(events['type_id'] == TID_PASS) & (events['outcome'] == 1) &
                    (events['contestant_id'] == team_id)].copy()
    passes = passes.dropna(subset=['x','y','end_x','end_y'])
    t = passes[(passes['end_x'] - passes['x']) >= 10].copy()

    fig = plt.figure(figsize=(14, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    fig.subplots_adjust(top=0.84, bottom=0.06)

    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.8, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    for _, row in t.iterrows():
        gain  = row['end_x'] - row['x']
        alpha = min(0.9, 0.35 + gain / 50)
        lw    = 0.5 + 2.0 * gain / 60
        ax.annotate('', xy=(row['end_x'], row['end_y']), xytext=(row['x'], row['y']),
                    arrowprops=dict(arrowstyle='->', color=color, alpha=alpha, lw=lw))

    fig.text(0.5, 0.025, f'{len(t)} progressive passes  (end_x > start_x + 10)',
             ha='center', va='bottom', fontsize=8.5, color='#8b9ab0')

    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── E9: Touches in Box ────────────────────────────────────────────────────────
def plot_touches_in_box(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Density map of all touches inside the team's attacking penalty box."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))

    poss = events[events['type_id'].isin(POSSESSION_TYPES) &
                  (events['contestant_id'] == team_id)].dropna(subset=['x','y'])
    t = poss[(poss['x'] >= 83) & (poss['y'].between(21, 79))].copy()

    fig, ax = plt.subplots(figsize=(10, 9), facecolor=BG)
    fig.subplots_adjust(top=0.84, bottom=0.06)

    pitch = VerticalPitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                          linewidth=0.9, half=True, pad_top=2, pad_bottom=0,
                          goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    if len(t) >= 4:
        try:
            kde = gaussian_kde(np.vstack([t['y'].values, t['x'].values]), bw_method=0.3)
            gx = np.linspace(21, 79, 60); gy = np.linspace(83, 100, 40)
            XX, YY = np.meshgrid(gx, gy)
            ZZ = kde(np.vstack([XX.ravel(), YY.ravel()])).reshape(XX.shape)
            cmap_name = 'Oranges' if color == HOME_COL else 'Blues'
            ax.contourf(XX, YY, ZZ, levels=10, cmap=cmap_name, alpha=0.72, zorder=2)
        except Exception:
            pass
    ax.scatter(t['y'], t['x'], s=20, color=color, alpha=0.45, zorder=3, edgecolors='none')

    fig.text(0.5, 0.025, f'{len(t)} touches in box',
             ha='center', va='bottom', fontsize=9, color='#8b9ab0')

    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── E10: Press Map ────────────────────────────────────────────────────────────
def plot_press_map(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Where the team wins the ball back after pressing (recoveries + interceptions)."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))

    press_types = {TID_INTERC, TID_RECOVERY}
    t = events[events['type_id'].isin(press_types) &
               (events['outcome'] == 1) &
               (events['contestant_id'] == team_id)].dropna(subset=['x','y']).copy()

    fig = plt.figure(figsize=(14, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    fig.subplots_adjust(top=0.84, bottom=0.06)

    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.8, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    if len(t) >= 4:
        try:
            kde = gaussian_kde(np.vstack([t['x'].values, t['y'].values]), bw_method=0.4)
            gx = np.linspace(0, 100, 80); gy = np.linspace(0, 100, 80)
            XX, YY = np.meshgrid(gx, gy)
            ZZ = kde(np.vstack([XX.ravel(), YY.ravel()])).reshape(XX.shape)
            cmap_name = 'Oranges' if color == HOME_COL else 'Blues'
            ax.contourf(XX, YY, ZZ, levels=10, cmap=cmap_name, alpha=0.55, zorder=2)
        except Exception:
            pass

    ax.scatter(t['x'], t['y'], s=40, color=color, alpha=0.72,
               edgecolors='white', linewidth=0.6, zorder=4)
    avg_x = t['x'].mean() if len(t) > 0 else 0
    ax.axvline(avg_x, color=color, lw=1.5, alpha=0.6, ls='--', zorder=5)
    ax.text(avg_x + 1, 5, f'Avg: {avg_x:.0f}', fontsize=8, color=color, va='bottom')

    fig.text(0.5, 0.025, f'{len(t)} press recoveries',
             ha='center', va='bottom', fontsize=8.5, color='#8b9ab0')

    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()

# ══════════════════════════════════════════════════════════════════════════════
# NEW EVENT VISUALS  (E12 – E21)
# ══════════════════════════════════════════════════════════════════════════════

# ── E12: Carry Map ─────────────────────────────────────────────────────────────
def plot_carry_map(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Ball carries / take-ons (typeId 3 = Take-On)."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))
    t = events[(events['contestant_id'] == team_id) &
               (events['type_id'] == TID_TAKEON)].dropna(subset=['x','y'])
    won  = t[t['outcome'] == 1]
    lost = t[t['outcome'] == 0]

    fig = plt.figure(figsize=(14, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.9, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    ax.scatter(won['x'],  won['y'],  s=65, color=color,
               edgecolors='white', linewidth=0.8, alpha=0.90, zorder=4)
    ax.scatter(lost['x'], lost['y'], s=45, color=PITCH_LINE,
               edgecolors='none', alpha=0.55, zorder=3)

    fig.text(0.5, 0.028,
             f'{len(t)} carries  ·  {len(won)} successful  ·  Bright=won  ·  Grey=lost',
             ha='center', va='bottom', fontsize=8.5, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── E13: Box Entry Map ─────────────────────────────────────────────────────────
def plot_box_entry(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Passes / carries where the ball enters the attacking penalty box."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))
    # Passes that end inside the box (end_x >= 83, end_y 21-79) starting outside
    passes = events[(events['contestant_id'] == team_id) &
                    (events['type_id'] == TID_PASS)].dropna(subset=['x','y','end_x','end_y'])
    entries = passes[
        (passes['end_x'] >= 83) & (passes['end_y'].between(21, 79)) &
        ~((passes['x'] >= 83) & (passes['y'].between(21, 79)))
    ].copy()
    completed = entries[entries['outcome'] == 1]
    incomplete = entries[entries['outcome'] == 0]
    crosses = entries[entries['end_y'].between(21, 35) | entries['end_y'].between(65, 79)]

    fig = plt.figure(figsize=(10, 11), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = VerticalPitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                          linewidth=0.9, half=True, pad_top=2, pad_bottom=4,
                          goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    for _, row in incomplete.iterrows():
        ax.annotate('', xy=(row['end_y'], row['end_x']), xytext=(row['y'], row['x']),
                    arrowprops=dict(arrowstyle='->', color=PITCH_LINE, alpha=0.55, lw=0.8))
    for _, row in completed.iterrows():
        ax.annotate('', xy=(row['end_y'], row['end_x']), xytext=(row['y'], row['x']),
                    arrowprops=dict(arrowstyle='->', color=color, alpha=0.80, lw=1.0))
    # Mark end-points
    ax.scatter(completed['end_y'], completed['end_x'],
               s=35, color=color, edgecolors='white', linewidth=0.6, zorder=5, alpha=0.90)

    fig.text(0.5, 0.028,
             f'{len(entries)} entries  ·  {len(completed)} successful ({len(completed)/max(len(entries),1)*100:.0f}%)'
             f'  ·  {len(crosses)} crosses  ·  Bright=completed  ·  Grey=incomplete',
             ha='center', va='bottom', fontsize=8, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── E14: Half Spaces ──────────────────────────────────────────────────────────
def plot_half_spaces(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Passes originating from left (y 17-33) and right (y 67-83) half-spaces."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))
    passes = events[(events['contestant_id'] == team_id) &
                    (events['type_id'] == TID_PASS)].dropna(subset=['x','y','end_x','end_y'])

    HS_LEFT  = passes[passes['y'].between(17, 33)].copy()
    HS_RIGHT = passes[passes['y'].between(67, 83)].copy()
    C_LEFT  = '#2ecc71'   # green
    C_RIGHT = C_PURPLE    # purple

    fig = plt.figure(figsize=(10, 11), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = VerticalPitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                          linewidth=0.9, half=True, pad_top=2, pad_bottom=4,
                          goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    # Half-space zone highlights
    for y0, y1, hc in [(17, 33, C_LEFT), (67, 83, C_RIGHT)]:
        rect = plt.Rectangle((y0, 50), y1-y0, 50, facecolor=hc, alpha=0.08, zorder=1)
        ax.add_patch(rect)
        ax.plot([y0,y0],[50,100], color=hc, lw=1.0, ls='--', alpha=0.35)
        ax.plot([y1,y1],[50,100], color=hc, lw=1.0, ls='--', alpha=0.35)

    for df, hc in [(HS_LEFT, C_LEFT), (HS_RIGHT, C_RIGHT)]:
        compl = df[df['outcome'] == 1]
        incompl = df[df['outcome'] == 0]
        for _, row in incompl.iterrows():
            ax.annotate('', xy=(row['end_y'], row['end_x']), xytext=(row['y'], row['x']),
                        arrowprops=dict(arrowstyle='->', color=PITCH_LINE, alpha=0.40, lw=0.7))
        for _, row in compl.iterrows():
            ax.annotate('', xy=(row['end_y'], row['end_x']), xytext=(row['y'], row['x']),
                        arrowprops=dict(arrowstyle='->', color=hc, alpha=0.75, lw=0.9))

    # Zone counts
    ax.text(25, 75, str(len(HS_LEFT)),  ha='center', va='center',
            fontsize=28, color=C_LEFT,  fontweight='bold', alpha=0.45, zorder=2)
    ax.text(75, 75, str(len(HS_RIGHT)), ha='center', va='center',
            fontsize=28, color=C_RIGHT, fontweight='bold', alpha=0.45, zorder=2)

    fig.text(0.5, 0.028,
             f'Left HS: {len(HS_LEFT)}  ·  Right HS: {len(HS_RIGHT)}'
             f'  ·  Bright=completed  ·  Grey=incomplete',
             ha='center', va='bottom', fontsize=8.5, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── E15: Zone 14 ──────────────────────────────────────────────────────────────
def plot_zone14(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """All passes from Zone 14 (x 68–83, y 33–67) — the central channel just outside box."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))
    passes = events[(events['contestant_id'] == team_id) &
                    (events['type_id'] == TID_PASS)].dropna(subset=['x','y','end_x','end_y'])
    z14 = passes[passes['x'].between(68, 83) & passes['y'].between(33, 67)].copy()
    touches = events[(events['contestant_id'] == team_id) &
                     events['type_id'].isin(POSSESSION_TYPES) &
                     events['x'].between(68, 83) & events['y'].between(33, 67)]
    completed = z14[z14['outcome'] == 1]
    incomplete = z14[z14['outcome'] == 0]

    fig = plt.figure(figsize=(10, 11), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = VerticalPitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                          linewidth=0.9, half=True, pad_top=2, pad_bottom=4,
                          goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    # Zone 14 box highlight
    rect = plt.Rectangle((33, 68), 34, 15, facecolor=C_YELLOW, alpha=0.08,
                          edgecolor=C_YELLOW, linewidth=1.0, linestyle='--', zorder=1)
    ax.add_patch(rect)

    for _, row in incomplete.iterrows():
        ax.annotate('', xy=(row['end_y'], row['end_x']), xytext=(row['y'], row['x']),
                    arrowprops=dict(arrowstyle='->', color=PITCH_LINE, alpha=0.45, lw=0.7))
    for _, row in completed.iterrows():
        ax.annotate('', xy=(row['end_y'], row['end_x']), xytext=(row['y'], row['x']),
                    arrowprops=dict(arrowstyle='->', color=C_YELLOW, alpha=0.82, lw=1.0))

    # Touch count label inside zone
    ax.text(50, 75, str(len(touches)), ha='center', va='center',
            fontsize=32, color=C_YELLOW, fontweight='bold', alpha=0.40, zorder=2)

    fig.text(0.5, 0.028,
             f'Zone 14 touches: {len(touches)}  ·  Passes: {len(z14)}'
             f'  ·  Completed: {len(completed)} ({len(completed)/max(len(z14),1)*100:.0f}%)',
             ha='center', va='bottom', fontsize=8.5, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── E16: Corner Delivery Map ──────────────────────────────────────────────────
def plot_corner_map(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Corner kicks — passes with qualifier FROM_CORNER (Q=25) from corner positions."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))
    # Reload raw JSON to check qualifiers properly — use what we have in events
    # Corner kicks: passes from corner positions (x < 5 or x > 95) AND (y < 5 or y > 95)
    # OR passes tagged with qualifier 25 (FROM_CORNER) — we stored qualifier lookup in load_opta_events
    # Fallback: filter by position
    passes = events[(events['contestant_id'] == team_id) &
                    (events['type_id'] == TID_PASS)].dropna(subset=['x','y','end_x','end_y'])
    corners = passes[
        ((passes['x'] > 95) & ((passes['y'] < 5) | (passes['y'] > 95))) |
        ((passes['x'] > 95) & passes['y'].between(0, 6)) |
        ((passes['x'] > 95) & passes['y'].between(94, 100))
    ].copy()
    # Also capture from qualifier column if available
    if 'is_corner' in events.columns:
        corners = passes[passes['is_corner'] == 1].copy()

    completed  = corners[corners['outcome'] == 1]
    incomplete = corners[corners['outcome'] == 0]
    left_corners  = corners[corners['y'] < 50]
    right_corners = corners[corners['y'] >= 50]

    fig = plt.figure(figsize=(10, 11), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = VerticalPitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                          linewidth=0.9, half=True, pad_top=2, pad_bottom=8,
                          goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    for _, row in incomplete.iterrows():
        ax.annotate('', xy=(row['end_y'], row['end_x']), xytext=(row['y'], row['x']),
                    arrowprops=dict(arrowstyle='->', color=PITCH_LINE, alpha=0.55, lw=0.8))
    for _, row in completed.iterrows():
        ax.annotate('', xy=(row['end_y'], row['end_x']), xytext=(row['y'], row['x']),
                    arrowprops=dict(arrowstyle='->', color=color, alpha=0.85, lw=1.1))
    # Dot at landing
    if len(completed):
        ax.scatter(completed['end_y'], completed['end_x'],
                   s=40, color=color, edgecolors='white', linewidth=0.6, zorder=5)

    if len(corners) == 0:
        ax.text(50, 75, 'No corner data\n(adjust qualifier IDs if needed)',
                ha='center', va='center', fontsize=10, color='#7a9ab0', transform=ax.transData)

    fig.text(0.5, 0.028,
             f'{len(corners)} corners  ·  Left: {len(left_corners)}  ·  Right: {len(right_corners)}'
             f'  ·  Completed: {len(completed)}  ·  Bright=completed  ·  Grey=incomplete',
             ha='center', va='bottom', fontsize=8, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── E17: Duels Map ────────────────────────────────────────────────────────────
def plot_duels_map(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Ground duels (●) and aerial duels (▲) — tackles split by header qualifier."""
    name  = id_to_name.get(team_id, str(team_id))
    t = events[(events['contestant_id'] == team_id) &
               (events['type_id'] == TID_TACKLE)].dropna(subset=['x','y'])

    # We need to know if aerial — stored in events? Use 'is_aerial' col if available
    if 'is_aerial' in t.columns:
        aerial = t[t['is_aerial'] == 1]
        ground = t[t['is_aerial'] == 0]
    else:
        # Heuristic: if end_x is NaN and y is near pitch edges, aerial — else treat all as ground
        aerial = t.iloc[0:0]   # empty fallback
        ground = t

    won_g  = ground[ground['outcome'] == 1]
    lost_g = ground[ground['outcome'] == 0]
    won_a  = aerial[aerial['outcome'] == 1]
    lost_a = aerial[aerial['outcome'] == 0]

    fig = plt.figure(figsize=(14, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.9, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    # Ground duels — circles
    ax.scatter(won_g['x'],  won_g['y'],  s=70, color=C_YELLOW, marker='o',
               edgecolors='white', linewidth=0.9, alpha=0.90, zorder=4, label='Ground won')
    ax.scatter(lost_g['x'], lost_g['y'], s=50, color=C_YELLOW, marker='o',
               edgecolors='none', alpha=0.30, zorder=3, label='Ground lost')
    # Aerial duels — triangles
    ax.scatter(won_a['x'],  won_a['y'],  s=70, color=C_PURPLE, marker='^',
               edgecolors='white', linewidth=0.9, alpha=0.90, zorder=4, label='Aerial won')
    ax.scatter(lost_a['x'], lost_a['y'], s=50, color=C_PURPLE, marker='^',
               edgecolors='none', alpha=0.35, zorder=3, label='Aerial lost')

    won_total = len(won_g) + len(won_a)
    total     = len(t)
    fig.text(0.5, 0.028,
             f'Aerial: {len(aerial)}  ·  Ground: {len(ground)}'
             f'  ·  Won: {won_total}/{total} ({won_total/max(total,1)*100:.0f}%)'
             f'  ·  ▲ Aerial  ·  ● Ground  ·  White border=won',
             ha='center', va='bottom', fontsize=8.5, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── E18: Transition Triggers ──────────────────────────────────────────────────
def plot_transition_triggers(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Ball recoveries coloured by what followed: shot (★), danger (●), or safe (□)."""
    name  = id_to_name.get(team_id, str(team_id))
    # Recoveries = typeId 49 OR interceptions (8) with outcome=1
    rec = events[(events['contestant_id'] == team_id) &
                 (events['type_id'].isin({TID_INTERC, TID_RECOVERY})) &
                 (events['outcome'] == 1)].dropna(subset=['x','y']).copy()
    rec = rec.sort_values('abs_min').reset_index(drop=True)

    ev_sorted = events.sort_values('abs_min').reset_index(drop=True)
    WINDOW = 15 / 60   # 15 seconds

    cats = []
    for _, row in rec.iterrows():
        t0 = row['abs_min']
        nxt = ev_sorted[(ev_sorted['abs_min'] > t0) &
                        (ev_sorted['abs_min'] <= t0 + WINDOW) &
                        (ev_sorted['contestant_id'] == team_id)]
        if nxt['type_id'].isin(TID_SHOT).any():
            cats.append('shot')
        elif (nxt['x'] > 67).any():
            cats.append('danger')
        else:
            cats.append('safe')
    rec['cat'] = cats

    shots_r   = rec[rec['cat'] == 'shot']
    danger_r  = rec[rec['cat'] == 'danger']
    safe_r    = rec[rec['cat'] == 'safe']

    fig = plt.figure(figsize=(14, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.9, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    ax.scatter(safe_r['x'],   safe_r['y'],   s=40, color=PITCH_LINE, marker='s',
               alpha=0.60, zorder=3)
    ax.scatter(danger_r['x'], danger_r['y'], s=65, color=HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL,
               marker='o', edgecolors='none', alpha=0.88, zorder=4)
    ax.scatter(shots_r['x'],  shots_r['y'],  s=150, color=C_YELLOW,
               marker='*', edgecolors='white', linewidth=0.5, alpha=0.95, zorder=5)

    fig.text(0.5, 0.028,
             f'{len(rec)} recoveries  ·  {len(danger_r)+len(shots_r)} led to danger ({(len(danger_r)+len(shots_r))/max(len(rec),1)*100:.0f}%)'
             f'  ·  {len(shots_r)} led to shot  ·  ★=shot  ·  ●=dangerous  ·  □=safe',
             ha='center', va='bottom', fontsize=8.5, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── E19: Player Heatmaps ──────────────────────────────────────────────────────
def plot_player_heatmap(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """6-panel grid of KDE heatmaps for the top-6 players by touch count."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    poss = events[(events['contestant_id'] == team_id) &
                  events['type_id'].isin(POSSESSION_TYPES)].dropna(subset=['x','y'])

    top6 = (poss.groupby(['player_id','player_name'])
               .size().reset_index(name='n')
               .sort_values('n', ascending=False).head(6))

    fig, axes = plt.subplots(2, 3, figsize=(18, 11), facecolor=BG)
    fig.subplots_adjust(top=0.88, bottom=0.04, left=0.02, right=0.98,
                        hspace=0.05, wspace=0.03)

    cmap_name = 'Oranges' if color == HOME_COL else 'Blues'

    for idx, (_, prow) in enumerate(top6.iterrows()):
        ax = axes[idx // 3][idx % 3]
        t  = poss[poss['player_id'] == prow['player_id']]
        pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                      linewidth=0.6, goal_type='box', goal_alpha=0.4)
        pitch.draw(ax=ax)
        if len(t) >= 5:
            try:
                kde = gaussian_kde(np.vstack([t['x'].values, t['y'].values]), bw_method=0.25)
                gx = np.linspace(0,100,80); gy = np.linspace(0,100,80)
                XX, YY = np.meshgrid(gx, gy)
                ZZ = kde(np.vstack([XX.ravel(), YY.ravel()])).reshape(XX.shape)
                ax.contourf(XX, YY, ZZ, levels=14, cmap=cmap_name, alpha=0.85, zorder=2)
            except Exception:
                pass
        short = prow['player_name'].split(' ')[-1] if ' ' in prow['player_name'] else prow['player_name']
        ax.text(0.02, 0.97, f'{short}  ({int(prow["n"])})',
                ha='left', va='top', fontsize=8.5, color='white', fontweight='bold',
                transform=ax.transAxes, zorder=6)

    # Blank unused panels
    for idx in range(len(top6), 6):
        axes[idx // 3][idx % 3].axis('off')

    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── E20: Box Defending ────────────────────────────────────────────────────────
def plot_box_defending(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Defensive actions inside own penalty box with a Box Defending Score gauge."""
    name  = id_to_name.get(team_id, str(team_id))
    BOX_TYPES = {
        7:  ('Tackle',    C_YELLOW),
        15: ('Block',     '#e74c3c'),
        12: ('Clearance', C_GREEN),
        4:  ('Aerial',    C_PURPLE),
        66: ('Blk Pass',  '#1abc9c'),
        49: ('Recovery',  AWAY_COL),
    }
    box_def = events[(events['contestant_id'] == team_id) &
                     events['type_id'].isin(BOX_TYPES.keys()) &
                     (events['x'] <= 17) &            # own defensive box
                     events['y'].between(21, 79)].dropna(subset=['x','y']).copy()

    total = len(box_def)
    won   = (box_def['outcome'] == 1).sum()
    score = round(won / max(total, 1) * 10, 1)

    fig = plt.figure(figsize=(12, 10), facecolor=BG)
    # Left: pitch panel
    ax_pitch = fig.add_axes([0.03, 0.10, 0.62, 0.72])
    pitch = VerticalPitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                          linewidth=0.9, half=True, pad_top=2, pad_bottom=8,
                          goal_type='box', goal_alpha=0.5)
    pitch.draw(ax_pitch)

    for _, row in box_def.iterrows():
        _, col = BOX_TYPES.get(int(row['type_id']), ('?', C_MUTED))
        is_won = row['outcome'] == 1
        ax_pitch.scatter(row['y'], row['x'], s=70, color=col,
                         edgecolors='white' if is_won else col,
                         linewidth=1.4 if is_won else 0.0,
                         alpha=0.92 if is_won else 0.35, zorder=4)

    # Legend row
    for lx, (tid, (lbl, lc)) in zip(np.linspace(0.05, 0.90, len(BOX_TYPES)), BOX_TYPES.items()):
        fig.text(lx, 0.055, '●', ha='center', fontsize=11, color=lc)
        fig.text(lx + 0.02, 0.055, lbl, ha='left', va='center', fontsize=7.5, color='#9ab0c0')

    fig.text(0.05, 0.030,
             f'{total} actions  ·  {won} won ({won/max(total,1)*100:.0f}%)'
             f'  ·  White border=won  ·  Faded=lost',
             ha='left', va='bottom', fontsize=8.5, color='#8b9ab0')

    # Right: gauge
    ax_g = fig.add_axes([0.67, 0.30, 0.30, 0.42])
    ax_g.set_facecolor(BG); ax_g.axis('off')
    ax_g.set_xlim(-1.1, 1.1); ax_g.set_ylim(-0.1, 1.1)

    import matplotlib.patches as mp
    theta_start, theta_end = 180, 0
    # Background arc
    bg_arc = mp.Arc((0, 0), 2.0, 2.0, angle=0, theta1=theta_start, theta2=theta_end,
                    color=PITCH_LINE, lw=12, zorder=1)
    ax_g.add_patch(bg_arc)
    # Score arc
    score_theta = theta_start - (score / 10) * 180
    fg_col = C_GREEN if score >= 7 else (C_YELLOW if score >= 5 else '#e74c3c')
    fg_arc = mp.Arc((0, 0), 2.0, 2.0, angle=0, theta1=score_theta, theta2=theta_end + (theta_start - score_theta),
                    color=fg_col, lw=12, zorder=2)
    # Simpler: draw arc from 0 to score
    fg_arc2 = mp.Arc((0, 0), 2.0, 2.0, angle=0,
                     theta1=180 - (score/10)*180, theta2=180,
                     color=fg_col, lw=12, zorder=2)
    ax_g.add_patch(fg_arc2)
    ax_g.text(0, 0.10, f'{score}', ha='center', va='center',
              fontsize=36, color='white', fontweight='bold')
    ax_g.text(0, -0.05, '/ 10', ha='center', va='center',
              fontsize=14, color='#7a9ab0')
    ax_g.text(0, 0.80, 'Box Defending Score', ha='center', va='center',
              fontsize=10, color='white')

    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── E21: Free Kick Map ────────────────────────────────────────────────────────
def plot_free_kick_map(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Free kick shots (★=goal ●=shot) and deliveries (arrows) in attacking half."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))

    # FK passes (deliveries): passes with qualifier FREE_KICK (Q=26) stored via is_fk col
    # or heuristic: passes in attacking half where x < 83 (outside box) marked as set piece
    if 'is_fk' in events.columns:
        fk_pass = events[(events['contestant_id'] == team_id) &
                         (events['type_id'] == TID_PASS) &
                         (events['is_fk'] == 1)].dropna(subset=['x','y','end_x','end_y'])
    else:
        # Fallback: can't distinguish FK passes without qualifier flag
        fk_pass = events.iloc[0:0]

    # FK shots: from shots data we can identify is_free_kick
    if 'is_free_kick' in events.columns:
        fk_shots = events[(events['contestant_id'] == team_id) &
                          events['type_id'].isin(TID_SHOT) &
                          (events['is_free_kick'] == 1)].dropna(subset=['x','y'])
    else:
        fk_shots = events.iloc[0:0]

    goals    = fk_shots[fk_shots['type_id'] == 16]
    on_tgt   = fk_shots[fk_shots['type_id'].isin({15, 16})]
    compl    = fk_pass[fk_pass['outcome'] == 1]
    incompl  = fk_pass[fk_pass['outcome'] == 0]

    fig = plt.figure(figsize=(10, 11), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = VerticalPitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                          linewidth=0.9, half=True, pad_top=2, pad_bottom=8,
                          goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    for _, row in incompl.iterrows():
        ax.annotate('', xy=(row['end_y'], row['end_x']), xytext=(row['y'], row['x']),
                    arrowprops=dict(arrowstyle='->', color=PITCH_LINE, alpha=0.50, lw=0.8))
    for _, row in compl.iterrows():
        ax.annotate('', xy=(row['end_y'], row['end_x']), xytext=(row['y'], row['x']),
                    arrowprops=dict(arrowstyle='->', color=color, alpha=0.82, lw=1.0))
    ax.scatter(on_tgt['y'], on_tgt['x'],  s=70, color=color,
               edgecolors='white', linewidth=0.8, alpha=0.88, zorder=5, label='Shot')
    ax.scatter(goals['y'],  goals['x'],   s=160, color=C_YELLOW,
               marker='*', edgecolors='white', linewidth=0.5, zorder=6, label='Goal')

    if len(fk_pass) == 0 and len(fk_shots) == 0:
        ax.text(50, 75, 'No free kick data\n(adjust qualifier IDs if needed)',
                ha='center', va='center', fontsize=10, color='#7a9ab0')

    fig.text(0.5, 0.028,
             f'Direct shots: {len(fk_shots)}  ·  Goals: {len(goals)}'
             f'  ·  Deliveries: {len(fk_pass)}  ·  ★=Goal  ·  ●=Shot  ·  Arrow=Delivery',
             ha='center', va='bottom', fontsize=8, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ══════════════════════════════════════════════════════════════════════════════
# ADDITIONAL VISUALS  (A1-A8)
# ══════════════════════════════════════════════════════════════════════════════

# ── A1: Momentum Chart (shared) ───────────────────────────────────────────────
def plot_momentum(events, shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    home_name = id_to_name.get(home_id, 'Home')
    away_name = id_to_name.get(away_id, 'Away')
    poss = events[events['type_id'].isin(POSSESSION_TYPES)].dropna(subset=['time_min'])
    max_min = max(int(events['time_min'].max()) + 3, 95)
    minutes = np.arange(0, max_min + 1)
    momentum = []
    for m in minutes:
        w = poss[(poss['time_min'] >= m - 5) & (poss['time_min'] <= m)]
        h = (w['contestant_id'] == home_id).sum()
        a = (w['contestant_id'] == away_id).sum()
        tot = h + a
        momentum.append((h - a) / tot if tot >= 5 else 0.0)
    momentum = np.array(momentum)
    from scipy.ndimage import uniform_filter1d
    smooth = uniform_filter1d(momentum, size=7)
    fig, ax = plt.subplots(figsize=(14, 5), facecolor=BG)
    ax.set_facecolor(BG)
    ax.axhline(0, color='white', lw=0.7, alpha=0.25)
    pos = np.where(smooth >= 0, smooth, 0)
    neg = np.where(smooth <  0, smooth, 0)
    ax.fill_between(minutes, pos, 0, alpha=0.45, color=HOME_COL, label=home_name)
    ax.fill_between(minutes, neg, 0, alpha=0.45, color=AWAY_COL, label=away_name)
    ax.plot(minutes, smooth, color='white', lw=0.6, alpha=0.35)
    for tid, col in [(home_id, HOME_COL), (away_id, AWAY_COL)]:
        for _, row in shots[(shots['contestant_id'] == tid) & (shots['is_goal'] == 1)].iterrows():
            ax.axvline(row['time_min'], color=col, lw=1.0, alpha=0.6, ls='--')
    ax.axvline(45, color='white', lw=0.6, alpha=0.15, ls='--')
    ax.text(45.5, ax.get_ylim()[0] * 0.9 if ax.get_ylim()[0] < 0 else 0.02,
            'HT', fontsize=7.5, color='#7a9ab0', va='bottom')
    ax.set_xlabel('Minute', color='#8b9ab0', fontsize=9)
    ax.set_ylabel('Touch Share \u0394  (home \u2212 away)', color='#8b9ab0', fontsize=9)
    ax.tick_params(colors='#6a7a8a'); ax.set_xlim(0, max_min)
    for spine in ax.spines.values(): spine.set_color(PITCH_LINE)
    ax.legend(fontsize=9, facecolor=BG, labelcolor='white',
              framealpha=0.3, edgecolor=PITCH_LINE, loc='upper right')
    fig.subplots_adjust(top=0.84, bottom=0.12)
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── A2: PPDA by Third (shared) ────────────────────────────────────────────────
def plot_ppda(events, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    home_name = id_to_name.get(home_id, 'Home')
    away_name = id_to_name.get(away_id, 'Away')
    thirds = [('Defensive Third', 0, 33), ('Middle Third', 33, 67), ('Attacking Third', 67, 100)]
    fig, axes = plt.subplots(1, 3, figsize=(15, 6), facecolor=BG)
    fig.subplots_adjust(top=0.84, bottom=0.1, left=0.06, right=0.97, wspace=0.38)
    def_actions = events[events['type_id'].isin(DEFENSIVE_TYPES) & (events['outcome'] == 1)]
    passes_all  = events[(events['type_id'] == TID_PASS) & (events['outcome'] == 1)]
    for ax, (label, x0, x1) in zip(axes, thirds):
        ax.set_facecolor(BG)
        for spine in ax.spines.values(): spine.set_color(PITCH_LINE)
        ax.tick_params(colors='#6a7a8a')
        vals, names, colors = [], [], []
        for tid, tname, tcol in [(home_id, home_name, HOME_COL), (away_id, away_name, AWAY_COL)]:
            opp = away_id if tid == home_id else home_id
            opp_passes = passes_all[(passes_all['contestant_id'] == opp) &
                                    (passes_all['x'].between(x0, x1))]
            own_def = def_actions[(def_actions['contestant_id'] == tid) &
                                  (def_actions['x'].between(x0, x1))]
            ppda = len(opp_passes) / max(len(own_def), 1)
            vals.append(ppda); names.append(tname); colors.append(tcol)
        bars = ax.barh(names, vals, color=colors, alpha=0.75, height=0.5)
        for bar, v in zip(bars, vals):
            ax.text(v + 0.2, bar.get_y() + bar.get_height() / 2,
                    f'{v:.1f}', va='center', fontsize=10, color='white', fontweight='bold')
        ax.set_xlim(0, max(vals) * 1.45 + 1)
        ax.set_title(label, color='#9ab0c0', fontsize=9, pad=6)
        ax.set_xlabel('PPDA  (lower = more intense press)', color='#7a9ab0', fontsize=7.5)
    fig.text(0.5, 0.025, 'PPDA = opponent passes allowed \u00f7 own defensive actions in zone',
             ha='center', va='bottom', fontsize=7.5, color='#4a6070')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── A3: Cross Map (per team) ──────────────────────────────────────────────────
def plot_cross_map(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))
    passes = events[(events['type_id'] == TID_PASS) &
                    (events['contestant_id'] == team_id)].dropna(subset=['x','y','end_x','end_y']).copy()
    is_cross = (
        ((passes['y'] < 25) | (passes['y'] > 75)) &
        (passes['end_x'] >= 83) & (passes['end_y'].between(21, 79))
    ) | (
        (passes['x'] >= 83) & ((passes['y'] < 25) | (passes['y'] > 75))
    )
    crosses = passes[is_cross].copy()
    fig = plt.figure(figsize=(14, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.8, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)
    compl   = crosses[crosses['outcome'] == 1]
    incompl = crosses[crosses['outcome'] == 0]
    for _, row in incompl.iterrows():
        ax.annotate('', xy=(row['end_x'], row['end_y']), xytext=(row['x'], row['y']),
                    arrowprops=dict(arrowstyle='->', color=PITCH_LINE, alpha=0.55, lw=0.9))
    for _, row in compl.iterrows():
        ax.annotate('', xy=(row['end_x'], row['end_y']), xytext=(row['x'], row['y']),
                    arrowprops=dict(arrowstyle='->', color=color, alpha=0.85, lw=1.2))
    ax.scatter(crosses['x'], crosses['y'], s=30, color=color,
               edgecolors='white', linewidth=0.6, alpha=0.75, zorder=5)
    acc = len(compl) / max(len(crosses), 1) * 100
    fig.text(0.5, 0.025,
             f'{len(crosses)} crosses  \u00b7  {len(compl)} completed ({acc:.0f}%)'
             f'  \u00b7  Bright = completed  \u00b7  Muted = incomplete',
             ha='center', va='bottom', fontsize=8, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()



# ── xCrossAcc helper ──────────────────────────────────────────────────────────
def _get_crosses(events, team_id):
    """Return DataFrame of cross events for a team with xCrossAcc scored."""
    passes = events[
        (events['type_id'] == TID_PASS) &
        (events['contestant_id'] == team_id)
    ].dropna(subset=['x','y','end_x','end_y']).copy()

    is_cross = (
        ((passes['y'] < 25) | (passes['y'] > 75)) &
        (passes['end_x'] >= 83) & (passes['end_y'].between(21, 79))
    ) | (
        (passes['x'] >= 83) & ((passes['y'] < 25) | (passes['y'] > 75))
    )
    c = passes[is_cross].copy()
    if len(c) == 0:
        return c

    # Feature engineering matching xcross_acc_model.pkl
    c['dist_to_goal']        = np.sqrt((100 - c['x'])**2 + (50 - c['y'])**2)
    c['cross_length']        = np.sqrt((c['end_x'] - c['x'])**2 + (c['end_y'] - c['y'])**2)
    c['end_dist_to_center']  = (c['end_y'] - 50).abs()
    c['angle_to_goal']       = np.arctan2((c['y'] - 50).abs(), 100 - c['x'])
    c['right_side']          = (c['y'] < 50).astype(int)
    c['byline']              = (c['x'] >= 94).astype(int)
    c['wide_zone']           = ((c['x'] >= 83) & (c['x'] < 94)).astype(int)
    c['deep_zone']           = ((c['x'] >= 66) & (c['x'] < 83)).astype(int)
    c['long_cross']          = (c['x'] < 66).astype(int)
    c['dist_from_touchline'] = np.minimum(c['y'], 100 - c['y'])
    # Approximate into_box: end lands in box (x>83, y 21-79)
    c['into_box']            = (
        (c['end_x'] > 83) & (c['end_y'].between(21, 79))
    ).astype(int)

    if xcross_model is not None and len(xcross_features):
        Xf = c[xcross_features].values
        c['xCrossAcc'] = xcross_model.predict_proba(Xf)[:, 1]
    else:
        c['xCrossAcc'] = np.nan

    # Zone label
    def _zone(x, y):
        if x < 66:   return 'Deep'
        byline = x >= 94
        if y < 15:   lat = 'R Byline' if byline else ('R Flank' if x >= 83 else 'R Flank Deep')
        elif y < 35: lat = 'R Half-Space'
        elif y < 65: lat = 'Central'
        elif y < 85: lat = 'L Half-Space'
        else:        lat = 'L Byline' if byline else ('L Flank' if x >= 83 else 'L Flank Deep')
        return lat
    c['zone'] = [_zone(row['x'], row['y']) for _, row in c.iterrows()]
    return c


# ── C1: xCrossAcc Map ─────────────────────────────────────────────────────────
def plot_xcross_map(events, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    """Pitch map: each cross arrow coloured by xCrossAcc (cool=low, warm=high)."""
    fig, axes = plt.subplots(1, 2, figsize=(20, 9.5), facecolor=BG)
    fig.subplots_adjust(top=0.84, bottom=0.08, left=0.02, right=0.98, wspace=0.06)

    cmap = plt.cm.RdYlGn
    norm = plt.Normalize(vmin=0.10, vmax=0.45)

    for ax, tid in zip(axes, [home_id, away_id]):
        color = HOME_COL if tid == home_id else AWAY_COL
        name  = id_to_name.get(tid, str(tid))
        c = _get_crosses(events, tid)

        pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                      linewidth=0.8, goal_type='box', goal_alpha=0.5)
        pitch.draw(ax=ax)

        if len(c) == 0:
            ax.set_title(f'{name}\nNo crosses recorded', color=TEXT, fontsize=11, pad=8)
            continue

        for _, row in c.iterrows():
            xca = row['xCrossAcc'] if not pd.isna(row['xCrossAcc']) else 0.24
            arrow_col = cmap(norm(xca))
            alpha = 0.85 if row['outcome'] == 1 else 0.45
            lw    = 1.4  if row['outcome'] == 1 else 0.8
            ax.annotate('', xy=(row['end_x'], row['end_y']),
                         xytext=(row['x'], row['y']),
                         arrowprops=dict(arrowstyle='->', color=arrow_col,
                                         alpha=alpha, lw=lw))
        # Origin dots
        dot_cols = [cmap(norm(r['xCrossAcc'] if not pd.isna(r['xCrossAcc']) else 0.24))
                    for _, r in c.iterrows()]
        ax.scatter(c['x'], c['y'], c=dot_cols, s=40, edgecolors='white',
                   linewidth=0.6, zorder=5, norm=norm)

        n  = len(c)
        acc = c['outcome'].mean() * 100
        avg_xca = c['xCrossAcc'].mean() * 100 if not c['xCrossAcc'].isna().all() else float('nan')
        oe  = acc - avg_xca if not pd.isna(avg_xca) else float('nan')
        ax.set_title(
            f'{name}\n'
            f'{n} crosses  ·  Actual {acc:.0f}%  ·  xCrossAcc {avg_xca:.0f}%  ·  O/E {oe:+.1f}%',
            color=TEXT, fontsize=10.5, pad=8
        )

    # Colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes, orientation='vertical', fraction=0.012, pad=0.01)
    cbar.set_label('xCrossAcc  (expected accuracy)', color=TEXT, fontsize=8)
    cbar.ax.yaxis.set_tick_params(color=TEXT)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT, fontsize=7)

    fig.text(0.5, 0.025,
             'Arrow colour = xCrossAcc  ·  Bright = completed  ·  Faded = incomplete  '
             '·  Green = high expected accuracy  ·  Red = low expected accuracy',
             ha='center', fontsize=8, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── C2: Cross Zone Breakdown ──────────────────────────────────────────────────
def plot_xcross_zone(events, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    """Side-by-side: actual vs expected accuracy by zone for each team."""
    ZONE_ORDER = ['R Byline','R Flank','R Flank Deep','R Half-Space',
                  'L Half-Space','L Flank Deep','L Flank','L Byline','Deep']
    ZONE_COLORS = {
        'R Byline':'#4CAF50','R Flank':'#8BC34A','R Flank Deep':'#CDDC39',
        'R Half-Space':'#FFC107',
        'L Half-Space':'#FF9800',
        'L Flank Deep':'#FF7043','L Flank':'#F44336','L Byline':'#E91E63',
        'Deep':'#2196F3',
    }

    fig, axes = plt.subplots(1, 2, figsize=(20, 8), facecolor=BG)
    fig.subplots_adjust(top=0.84, bottom=0.14, left=0.06, right=0.97, wspace=0.30)

    for ax, tid in zip(axes, [home_id, away_id]):
        name  = id_to_name.get(tid, str(tid))
        c = _get_crosses(events, tid)

        if len(c) == 0:
            ax.text(0.5, 0.5, 'No crosses', ha='center', va='center',
                    transform=ax.transAxes, color=TEXT, fontsize=12)
            ax.set_facecolor(BG); ax.set_title(name, color=TEXT, fontsize=12)
            continue

        stats = c.groupby('zone').agg(
            n=('outcome','count'),
            actual=('outcome','mean'),
            xca=('xCrossAcc','mean'),
        ).reindex(ZONE_ORDER).dropna(subset=['n'])
        stats = stats[stats['n'] >= 1]

        x_pos = np.arange(len(stats))
        w = 0.35

        bars_act = ax.bar(x_pos - w/2, stats['actual']*100, w,
                          color=[ZONE_COLORS.get(z,'#888') for z in stats.index],
                          alpha=0.9, label='Actual', zorder=3)
        bars_xca = ax.bar(x_pos + w/2, stats['xca']*100, w,
                          color=[ZONE_COLORS.get(z,'#888') for z in stats.index],
                          alpha=0.4, label='xCrossAcc', zorder=3, hatch='///',
                          edgecolor='white', linewidth=0.5)

        # O/E labels
        for i, (zone, row) in enumerate(stats.iterrows()):
            oe = (row['actual'] - row['xca']) * 100
            col = '#4CAF50' if oe >= 0 else '#F44336'
            ax.text(x_pos[i], max(row['actual'], row['xca'])*100 + 1.2,
                    f'{oe:+.0f}%', ha='center', fontsize=7, color=col, fontweight='bold')
            ax.text(x_pos[i], -3.5, f'n={int(row["n"])}',
                    ha='center', fontsize=6.5, color='#8b9ab0')

        ax.axhline(c['outcome'].mean()*100, ls='--', lw=0.9, color='white', alpha=0.4,
                   label=f'Team avg {c["outcome"].mean()*100:.0f}%')
        ax.set_xticks(x_pos)
        ax.set_xticklabels(stats.index, rotation=35, ha='right', fontsize=8, color=TEXT)
        ax.set_ylabel('Accuracy %', color=TEXT, fontsize=9)
        ax.set_ylim(-6, 65)
        ax.set_facecolor(BG)
        ax.tick_params(colors=TEXT)
        ax.spines[:].set_color(PITCH_LINE)
        ax.yaxis.label.set_color(TEXT)
        ax.legend(fontsize=8, facecolor=BG, labelcolor=TEXT, framealpha=0.4, loc='upper right')
        n  = len(c); acc = c['outcome'].mean()*100
        avg_xca = c['xCrossAcc'].mean()*100
        oe = acc - avg_xca
        ax.set_title(
            f'{name}\n{n} crosses  ·  Actual {acc:.0f}%  ·  xCrossAcc {avg_xca:.0f}%  ·  O/E {oe:+.1f}%',
            color=TEXT, fontsize=10.5
        )

    fig.text(0.5, 0.04,
             'Solid = actual accuracy  ·  Hatched = xCrossAcc (expected)  ·  Label = O/E (actual − expected)',
             ha='center', fontsize=8.5, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── C3: xCrossAcc Player O/E ──────────────────────────────────────────────────
def plot_xcross_player_oe(events, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    """Horizontal bar chart: per-player Actual% vs xCrossAcc% (O/E) for both teams."""
    fig, axes = plt.subplots(1, 2, figsize=(20, 9), facecolor=BG)
    fig.subplots_adjust(top=0.84, bottom=0.08, left=0.04, right=0.97, wspace=0.38)

    for ax, tid in zip(axes, [home_id, away_id]):
        color = HOME_COL if tid == home_id else AWAY_COL
        name  = id_to_name.get(tid, str(tid))
        c = _get_crosses(events, tid)

        if len(c) == 0:
            ax.text(0.5, 0.5, 'No crosses', ha='center', va='center',
                    transform=ax.transAxes, color=TEXT, fontsize=12)
            ax.set_facecolor(BG); ax.set_title(name, color=TEXT, fontsize=12); continue

        p = c.groupby('player_name').agg(
            n=('outcome','count'),
            actual=('outcome','mean'),
            xca=('xCrossAcc','mean'),
        ).reset_index()
        p = p[p['n'] >= 2].copy()
        p['oe'] = (p['actual'] - p['xca']) * 100
        p = p.sort_values('oe')

        if len(p) == 0:
            ax.text(0.5, 0.5, 'No players with ≥2 crosses', ha='center', va='center',
                    transform=ax.transAxes, color=TEXT, fontsize=11)
            ax.set_facecolor(BG); continue

        y = np.arange(len(p))
        bar_cols = ['#4CAF50' if v >= 0 else '#F44336' for v in p['oe']]
        ax.barh(y, p['oe'], color=bar_cols, alpha=0.85, height=0.6, zorder=3)
        ax.axvline(0, color='white', lw=0.8, alpha=0.5)

        # Actual & xCrossAcc labels
        for i, row in p.reset_index(drop=True).iterrows():
            x_off = 0.4 if row['oe'] >= 0 else -0.4
            ha    = 'left' if row['oe'] >= 0 else 'right'
            ax.text(row['oe'] + x_off, i,
                    f'{row["actual"]*100:.0f}% vs xCA {row["xca"]*100:.0f}%  (n={int(row["n"])})',
                    va='center', ha=ha, fontsize=7, color=TEXT)

        ax.set_yticks(y)
        ax.set_yticklabels(p['player_name'], fontsize=8.5, color=TEXT)
        ax.set_xlabel('Accuracy O/E  (actual % − xCrossAcc %)', color=TEXT, fontsize=9)
        ax.set_facecolor(BG)
        ax.tick_params(colors=TEXT)
        ax.spines[:].set_color(PITCH_LINE)
        ax.xaxis.label.set_color(TEXT)
        n  = len(c); acc = c['outcome'].mean()*100
        avg_xca = c['xCrossAcc'].mean()*100
        ax.set_title(
            f'{name}  —  Player Cross Accuracy vs Expected\n'
            f'{n} crosses total  ·  Team avg {acc:.0f}%  ·  xCrossAcc {avg_xca:.0f}%',
            color=TEXT, fontsize=10.5
        )

    fig.text(0.5, 0.025,
             'Green = outperforming location-adjusted expectation  ·  '
             'Red = underperforming  ·  Min 2 crosses',
             ha='center', fontsize=8.5, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── A4: Key Pass Map (per team) ───────────────────────────────────────────────
def plot_key_pass_map(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))
    team_ev = events[events['contestant_id'] == team_id].sort_values('abs_min').reset_index(drop=True)
    shots_idx = team_ev[team_ev['type_id'].isin(TID_SHOT)].index
    key_rows = []
    for si in shots_idx:
        if si == 0: continue
        prev = team_ev.iloc[si - 1]
        if prev['type_id'] == TID_PASS and not pd.isna(prev.get('x')):
            key_rows.append(prev)
    kp = pd.DataFrame(key_rows).drop_duplicates() if key_rows else pd.DataFrame()
    fig = plt.figure(figsize=(14, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.8, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)
    if len(kp) > 0:
        kp = kp.dropna(subset=['x','y','end_x','end_y'])
        for _, row in kp.iterrows():
            ax.annotate('', xy=(row['end_x'], row['end_y']), xytext=(row['x'], row['y']),
                        arrowprops=dict(arrowstyle='->', color=color, alpha=0.80, lw=1.3))
        ax.scatter(kp['x'], kp['y'], s=55, color=color,
                   edgecolors='white', linewidth=0.8, alpha=0.90, zorder=5)
    fig.text(0.5, 0.025, f'{len(kp)} key passes  (pass immediately before a shot)',
             ha='center', va='bottom', fontsize=8.5, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── A5: Long Ball Map (per team) ──────────────────────────────────────────────
def plot_long_ball_map(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))
    passes = events[(events['type_id'] == TID_PASS) &
                    (events['contestant_id'] == team_id)].dropna(subset=['x','y','end_x','end_y']).copy()
    passes['dist'] = np.sqrt((passes['end_x'] - passes['x'])**2 + (passes['end_y'] - passes['y'])**2)
    long = passes[passes['dist'] >= 32].copy()
    fig = plt.figure(figsize=(14, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.8, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)
    compl   = long[long['outcome'] == 1]
    incompl = long[long['outcome'] == 0]
    for _, row in incompl.iterrows():
        ax.annotate('', xy=(row['end_x'], row['end_y']), xytext=(row['x'], row['y']),
                    arrowprops=dict(arrowstyle='->', color=PITCH_LINE, alpha=0.40, lw=0.7))
    for _, row in compl.iterrows():
        ax.annotate('', xy=(row['end_x'], row['end_y']), xytext=(row['x'], row['y']),
                    arrowprops=dict(arrowstyle='->', color=color, alpha=0.75, lw=1.0))
    acc = len(compl) / max(len(long), 1) * 100
    fig.text(0.5, 0.025,
             f'{len(long)} long balls (dist \u226532)  \u00b7  {len(compl)} accurate ({acc:.0f}%)',
             ha='center', va='bottom', fontsize=8, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── A6: Attacking Third Entries (per team) ────────────────────────────────────
def plot_attacking_entries(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))
    team_ev = events[events['contestant_id'] == team_id].dropna(subset=['x','y','end_x','end_y'])
    entries = team_ev[
        (team_ev['type_id'].isin({TID_PASS, TID_TAKEON})) &
        (team_ev['x'] < 67) & (team_ev['end_x'] >= 67)
    ].copy()
    fig = plt.figure(figsize=(14, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.8, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)
    ax.axvspan(67, 100, alpha=0.06, color=color, zorder=0)
    ax.axvline(67, color=color, lw=1.0, alpha=0.35, ls='--')
    compl   = entries[entries['outcome'] == 1]
    incompl = entries[entries['outcome'] == 0]
    for _, row in incompl.iterrows():
        ax.annotate('', xy=(row['end_x'], row['end_y']), xytext=(row['x'], row['y']),
                    arrowprops=dict(arrowstyle='->', color=PITCH_LINE, alpha=0.45, lw=0.8))
    for _, row in compl.iterrows():
        ax.annotate('', xy=(row['end_x'], row['end_y']), xytext=(row['x'], row['y']),
                    arrowprops=dict(arrowstyle='->', color=color, alpha=0.82, lw=1.1))
    ax.scatter(entries['x'], entries['y'], s=28, color=color,
               edgecolors='none', alpha=0.65, zorder=5)
    acc = len(compl) / max(len(entries), 1) * 100
    fig.text(0.5, 0.025,
             f'{len(entries)} final-third entries  \u00b7  {len(compl)} retained ({acc:.0f}%)',
             ha='center', va='bottom', fontsize=8.5, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── A7: Width of Play (per team) ──────────────────────────────────────────────
def plot_width_of_play(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))
    poss = events[events['type_id'].isin(POSSESSION_TYPES) &
                  (events['contestant_id'] == team_id)].dropna(subset=['x','y']).copy()
    fig = plt.figure(figsize=(14, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.8, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)
    if len(poss) >= 10:
        try:
            kde = gaussian_kde(np.vstack([poss['x'].values, poss['y'].values]), bw_method=0.18)
            gx = np.linspace(0, 100, 100); gy = np.linspace(0, 100, 100)
            XX, YY = np.meshgrid(gx, gy)
            ZZ = kde(np.vstack([XX.ravel(), YY.ravel()])).reshape(XX.shape)
            cmap_name = 'Oranges' if color == HOME_COL else 'Blues'
            ax.contourf(XX, YY, ZZ, levels=14, cmap=cmap_name, alpha=0.62, zorder=2)
        except Exception:
            pass
    for y_line in [33, 67]:
        ax.axhline(y_line, color='white', lw=0.5, alpha=0.18, ls='--')
    ax.text(50, 16, 'Left',   ha='center', va='center', fontsize=8, color='#5a7a90', alpha=0.6)
    ax.text(50, 50, 'Centre', ha='center', va='center', fontsize=8, color='#5a7a90', alpha=0.6)
    ax.text(50, 84, 'Right',  ha='center', va='center', fontsize=8, color='#5a7a90', alpha=0.6)
    left   = poss[poss['y'] < 33]
    centre = poss[poss['y'].between(33, 67)]
    right  = poss[poss['y'] > 67]
    tot    = max(len(poss), 1)
    fig.text(0.5, 0.025,
             f'Left {len(left)/tot*100:.0f}%  \u00b7  Centre {len(centre)/tot*100:.0f}%'
             f'  \u00b7  Right {len(right)/tot*100:.0f}%  \u00b7  ({len(poss)} touches)',
             ha='center', va='bottom', fontsize=8.5, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── A8: Switch of Play Map (per team) ─────────────────────────────────────────
def plot_switch_map(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))
    passes = events[(events['type_id'] == TID_PASS) &
                    (events['contestant_id'] == team_id)].dropna(subset=['x','y','end_x','end_y']).copy()
    lateral = (passes['end_y'] - passes['y']).abs()
    crosses_centre = (
        ((passes['y'] < 50) & (passes['end_y'] > 50)) |
        ((passes['y'] > 50) & (passes['end_y'] < 50))
    )
    switches = passes[crosses_centre & (lateral >= 33)].copy()
    fig = plt.figure(figsize=(14, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.8, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)
    ax.axhline(50, color='white', lw=0.5, alpha=0.15, ls='--')
    compl   = switches[switches['outcome'] == 1]
    incompl = switches[switches['outcome'] == 0]
    for _, row in incompl.iterrows():
        ax.annotate('', xy=(row['end_x'], row['end_y']), xytext=(row['x'], row['y']),
                    arrowprops=dict(arrowstyle='->', color=PITCH_LINE, alpha=0.45, lw=0.8,
                                   connectionstyle='arc3,rad=0.15'))
    for _, row in compl.iterrows():
        ax.annotate('', xy=(row['end_x'], row['end_y']), xytext=(row['x'], row['y']),
                    arrowprops=dict(arrowstyle='->', color=color, alpha=0.82, lw=1.2,
                                   connectionstyle='arc3,rad=0.15'))
    acc = len(compl) / max(len(switches), 1) * 100
    fig.text(0.5, 0.025,
             f'{len(switches)} switches of play  \u00b7  {len(compl)} accurate ({acc:.0f}%)',
             ha='center', va='bottom', fontsize=8, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()




# ── Score Outcome Probability (shared) ────────────────────────────────────────
def plot_score_probability(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    """Poisson scoreline probability matrix + win/draw/loss bar from xG totals."""
    from scipy.stats import poisson

    home_name = id_to_name.get(home_id, 'Home')
    away_name = id_to_name.get(away_id, 'Away')

    home_xg = shots[shots['contestant_id'] == home_id]['xg'].sum()
    away_xg = shots[shots['contestant_id'] == away_id]['xg'].sum()
    MAX_G   = 6  # 0..5

    # P(home=i, away=j)
    prob_matrix = np.zeros((MAX_G, MAX_G))
    for i in range(MAX_G):
        for j in range(MAX_G):
            prob_matrix[i, j] = poisson.pmf(i, home_xg) * poisson.pmf(j, away_xg)

    p_home_win = prob_matrix[np.tril_indices(MAX_G, -1)].sum()
    p_draw     = np.trace(prob_matrix)
    p_away_win = prob_matrix[np.triu_indices(MAX_G, 1)].sum()

    fig = plt.figure(figsize=(15, 9), facecolor=BG)
    ax_mat  = fig.add_axes([0.10, 0.12, 0.50, 0.70])
    ax_bar  = fig.add_axes([0.66, 0.20, 0.30, 0.55])

    # ── probability matrix ────────────────────────────────────────────────────
    cmap_mat = LinearSegmentedColormap.from_list('prob', [BG, '#1a3a5a', HOME_COL])
    im = ax_mat.imshow(prob_matrix, cmap=cmap_mat, vmin=0, vmax=prob_matrix.max(),
                        origin='upper', aspect='auto')

    for i in range(MAX_G):
        for j in range(MAX_G):
            p = prob_matrix[i, j]
            txt_col = 'white' if p > prob_matrix.max() * 0.4 else '#8b9ab0'
            ax_mat.text(j, i, f'{p*100:.1f}%',
                        ha='center', va='center', fontsize=9,
                        color=txt_col, fontweight='bold')

    # Diagonal = draw (highlight)
    for k in range(MAX_G):
        ax_mat.add_patch(plt.Rectangle((k-0.5, k-0.5), 1, 1,
                                        fill=False, edgecolor='#ffd700',
                                        lw=1.5, linestyle='--', zorder=3))

    ax_mat.set_xticks(range(MAX_G))
    ax_mat.set_yticks(range(MAX_G))
    ax_mat.set_xticklabels(range(MAX_G), color='white', fontsize=10)
    ax_mat.set_yticklabels(range(MAX_G), color='white', fontsize=10)
    ax_mat.set_xlabel(f'{away_name} goals', color='#8b9ab0', fontsize=11)
    ax_mat.set_ylabel(f'{home_name} goals', color='#8b9ab0', fontsize=11)
    ax_mat.tick_params(colors='#6a7a8a')
    for spine in ax_mat.spines.values(): spine.set_color(PITCH_LINE)

    # Team xG labels
    ax_mat.text(-0.12, -0.07, f'{home_name}  xG: {home_xg:.2f}',
                ha='left', va='top', fontsize=9.5, color=HOME_COL,
                fontweight='bold', transform=ax_mat.transAxes)
    ax_mat.text(1.0, -0.07, f'{away_name}  xG: {away_xg:.2f}',
                ha='right', va='top', fontsize=9.5, color=AWAY_COL,
                fontweight='bold', transform=ax_mat.transAxes)
    ax_mat.text(0.5, -0.07, 'Yellow dashes = draw scorelines',
                ha='center', va='top', fontsize=7.5, color='#ffd700',
                transform=ax_mat.transAxes)

    # ── win/draw/loss bar ─────────────────────────────────────────────────────
    ax_bar.set_facecolor(BG)
    for spine in ax_bar.spines.values(): spine.set_color(PITCH_LINE)
    ax_bar.tick_params(colors='#6a7a8a')

    categories = [home_name + ' Win', 'Draw', away_name + ' Win']
    probs      = [p_home_win, p_draw, p_away_win]
    colors_bar = [HOME_COL, '#7a9ab0', AWAY_COL]
    bars = ax_bar.bar(categories, [p*100 for p in probs],
                       color=colors_bar, alpha=0.82, width=0.55,
                       edgecolor='white', linewidth=0.6)
    for bar, p in zip(bars, probs):
        ax_bar.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.8,
                    f'{p*100:.1f}%',
                    ha='center', va='bottom', fontsize=13,
                    color='white', fontweight='bold')

    ax_bar.set_ylim(0, max(probs)*100 * 1.25)
    ax_bar.set_ylabel('Probability (%)', color='#8b9ab0', fontsize=9)
    ax_bar.set_title('Match Outcome Probability', color='#9ab0c0', fontsize=10, pad=8)
    ax_bar.tick_params(axis='x', colors='white', labelsize=9)

    fig.text(0.5, 0.025,
             'Poisson distribution model using in-match xG totals  ·  Matrix cell = P(exact scoreline)',
             ha='center', va='bottom', fontsize=8, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()




# ══════════════════════════════════════════════════════════════════════════════
# CANNON-STATS STYLE VISUALS
# ══════════════════════════════════════════════════════════════════════════════

# ── CS1: Field Tilt Mirror Chart (shared) ─────────────────────────────────────
def plot_field_tilt_mirror(events, shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    """Field tilt: above 50 = home dominant (filled), below = away dominant.
    Goals shown as dashed verticals + markers; shots as small ticks."""
    home_name = id_to_name.get(home_id, 'Home')
    away_name = id_to_name.get(away_id, 'Away')

    poss = events[events['type_id'].isin(POSSESSION_TYPES)].dropna(subset=['x','time_min']).copy()
    final_third = poss[poss['x'] > 67].copy()
    final_third['home'] = (final_third['contestant_id'] == home_id).astype(int)

    max_min = max(int(events['time_min'].max()) + 3, 95)
    minutes = np.arange(0, max_min + 1)

    WIN = 10  # 10-min rolling window
    tilt = []
    for m in minutes:
        w = final_third[(final_third['time_min'] >= m - WIN) & (final_third['time_min'] <= m)]
        tilt.append((w['home'].sum() / len(w) * 100) if len(w) >= 4 else np.nan)
    tilt = np.array(tilt, dtype=float)

    avg_h = float(np.nanmean(tilt))
    avg_a = 100 - avg_h

    fig, ax = plt.subplots(figsize=(16, 7), facecolor=BG)
    ax.set_facecolor(BG)
    fig.subplots_adjust(top=0.84, bottom=0.10, left=0.07, right=0.97)

    # Mirror axis: flip so home goes up, away goes down from 50
    # Plot home tilt above 50, away tilt (100-tilt) below 50 mirrored
    ax.axhline(50, color='white', lw=1.2, alpha=0.35, ls='--', zorder=3)

    # Home fill (above 50)
    ax.fill_between(minutes, tilt, 50,
                    where=(tilt >= 50), interpolate=True,
                    color=HOME_COL, alpha=0.80, zorder=2)
    # Away fill (below 50)
    ax.fill_between(minutes, tilt, 50,
                    where=(tilt <= 50), interpolate=True,
                    color=AWAY_COL, alpha=0.80, zorder=2)

    # Smoothed line
    from scipy.ndimage import uniform_filter1d
    valid = ~np.isnan(tilt)
    if valid.sum() > 5:
        smooth = tilt.copy()
        smooth[~valid] = np.interp(minutes[~valid], minutes[valid], tilt[valid])
        smooth = uniform_filter1d(smooth, size=5)
        ax.plot(minutes, smooth, color='white', lw=1.0, alpha=0.50, zorder=4)

    # Goal markers
    for tid, col, sign in [(home_id, HOME_COL, 1), (away_id, AWAY_COL, -1)]:
        gs = shots[(shots['contestant_id'] == tid) & (shots['is_goal'] == 1)]
        for _, row in gs.iterrows():
            m = int(row['time_min'])
            tval = tilt[min(m, len(tilt)-1)] if not np.isnan(tilt[min(m, len(tilt)-1)]) else 50
            ax.axvline(m, color=col, lw=1.2, alpha=0.70, ls='--', zorder=5)
            marker_y = 98 if sign == 1 else 2
            ax.scatter(m, marker_y, s=120, color=col, marker='*',
                       edgecolors='white', linewidth=0.8, zorder=7)

        # Shot ticks
        non_goal = shots[(shots['contestant_id'] == tid) & (shots['is_goal'] == 0)]
        for _, row in non_goal.iterrows():
            m = int(row['time_min'])
            tick_y = 96 if sign == 1 else 4
            ax.scatter(m, tick_y, s=30, color=col, marker='|',
                       linewidth=1.5, alpha=0.70, zorder=6)

    # HT line
    ax.axvline(45, color='white', lw=0.8, alpha=0.20, ls='--')
    ax.text(45.5, 51, 'HT', fontsize=7.5, color='#7a9ab0', va='bottom')

    # Y-axis: show 0,25,50,75,100 but label 50 as equilibrium
    ax.set_ylim(0, 100)
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.set_yticklabels(['100%', '75%', '50%', '75%', '100%'], color='#6a7a8a', fontsize=8)
    ax.set_xlabel('Minute', color='#8b9ab0', fontsize=9)
    ax.set_ylabel('Field Tilt  (final third touches)', color='#8b9ab0', fontsize=9, labelpad=8)
    ax.set_xlim(0, max_min)
    ax.tick_params(colors='#6a7a8a')
    for spine in ax.spines.values(): spine.set_color(PITCH_LINE)

    # Team labels
    ax.text(0.01, 0.97, f'{home_name}:  {avg_h:.1f}%',
            ha='left', va='top', fontsize=11, color=HOME_COL,
            fontweight='bold', transform=ax.transAxes)
    ax.text(0.01, 0.87, f'{away_name}:  {avg_a:.1f}%',
            ha='left', va='top', fontsize=11, color=AWAY_COL,
            fontweight='bold', transform=ax.transAxes)

    fig.text(0.5, 0.025,
             f'{WIN}-minute moving average  ·  Final-third touch share  ·  ★ = goal  ·  | = shot',
             ha='center', va='bottom', fontsize=8, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── CS2: Goal Probability Added per Player (per team) ─────────────────────────
def plot_gpa(events, shots, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """GPA approx: shots credited xG; key passes credited xG of the assisted shot."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))

    team_ev = events[events['contestant_id'] == team_id].sort_values('abs_min').reset_index(drop=True)
    team_shots_idx = team_ev[team_ev['type_id'].isin(TID_SHOT)].index

    gpa = {}  # player_id -> (player_name, gpa_value)

    # Credit shooters with xG of their shot
    team_shots_df = shots[shots['contestant_id'] == team_id].copy()
    for _, row in team_shots_df.iterrows():
        pid  = str(row['player_id'])
        pname = str(row['player_name'])
        xg_val = float(row.get('xg', 0) or 0)
        gpa[pid] = (pname, gpa.get(pid, (pname, 0))[1] + xg_val)

    # Credit the last-pass player with xG of the resulting shot
    for si in team_shots_idx:
        shot_row_ev = team_ev.iloc[si]
        # Find xG for this shot from the shots DataFrame
        shot_match = team_shots_df[
            (team_shots_df['time_min'] == shot_row_ev['time_min']) &
            (abs(team_shots_df['x'] - shot_row_ev['x']) < 3)
        ]
        if len(shot_match) == 0: continue
        xg_val = float(shot_match.iloc[0].get('xg', 0) or 0)
        for k in range(1, 4):
            if si - k < 0: break
            prev = team_ev.iloc[si - k]
            if prev['type_id'] == TID_PASS:
                pid   = str(prev['player_id'])
                pname = str(prev['player_name'])
                # Don't double-count if same player took the shot
                if pid not in gpa or gpa[pid][0] != str(shot_row_ev.get('player_name','')):
                    gpa[pid] = (pname, gpa.get(pid, (pname, 0))[1] + xg_val * 0.5)
                break

    if not gpa:
        return

    df = pd.DataFrame([(pid, n, v) for pid,(n,v) in gpa.items()],
                      columns=['pid','player','gpa'])
    df = df.sort_values('gpa', ascending=True)

    fig, ax = plt.subplots(figsize=(12, max(6, len(df) * 0.52)), facecolor=BG)
    ax.set_facecolor(BG)
    fig.subplots_adjust(top=0.88, bottom=0.08, left=0.32, right=0.88)

    vmax = df['gpa'].abs().max()
    cmap_pos = LinearSegmentedColormap.from_list('gpa_pos', ['#c0a0a0', color])
    cmap_neg = LinearSegmentedColormap.from_list('gpa_neg', [AWAY_COL if color == HOME_COL else HOME_COL, '#3a3a5a'])

    for i, (_, row) in enumerate(df.iterrows()):
        v = row['gpa']
        bar_col = cmap_pos(min(abs(v) / max(vmax, 0.01), 1.0)) if v >= 0                   else cmap_neg(min(abs(v) / max(vmax, 0.01), 1.0))
        ax.barh(i, v, color=bar_col, height=0.65, zorder=3)
        x_txt = v + 0.005 if v >= 0 else v - 0.005
        ha = 'left' if v >= 0 else 'right'
        ax.text(x_txt, i, f'{v:.2f}', va='center', fontsize=8.5,
                color='white', fontweight='bold', ha=ha)

    ax.set_yticks(range(len(df)))
    ax.set_yticklabels(df['player'], fontsize=8.5, color='white')
    ax.axvline(0, color='white', lw=0.8, alpha=0.35)
    ax.set_xlabel('Goal Probability Added', color='#8b9ab0', fontsize=10, fontweight='bold')
    ax.tick_params(colors='#6a7a8a')
    for spine in ax.spines.values(): spine.set_color(PITCH_LINE)

    fig.text(0.5, 0.01,
             'GPA = xG from shots + 0.5 × xG from assisted shots',
             ha='center', va='bottom', fontsize=7.5, color='#4a6070')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── CS3: xG Buildup per Player (per team) ─────────────────────────────────────
def plot_xg_buildup_player(events, shots, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Sum of xG for shots in sequences each player was involved in (last N actions)."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))

    team_ev    = events[events['contestant_id'] == team_id].sort_values('abs_min').reset_index(drop=True)
    team_shots = shots[shots['contestant_id'] == team_id].copy()
    shot_idxs  = team_ev[team_ev['type_id'].isin(TID_SHOT)].index

    buildup_xg = {}  # player_id -> (name, total_xg)

    for si in shot_idxs:
        ev_row = team_ev.iloc[si]
        shot_m = team_shots[
            (team_shots['time_min'] == ev_row['time_min']) &
            (abs(team_shots['x'] - ev_row['x']) < 3)
        ]
        if len(shot_m) == 0: continue
        xg_val = float(shot_m.iloc[0].get('xg', 0) or 0)

        # Collect players in last 5 actions before this shot
        involved = set()
        for k in range(0, min(6, si + 1)):
            prev = team_ev.iloc[si - k]
            pid  = str(prev['player_id'])
            pname = str(prev['player_name'])
            if pid and pname and pname != 'Unknown':
                involved.add((pid, pname))
        for pid, pname in involved:
            buildup_xg[pid] = (pname, buildup_xg.get(pid, (pname, 0.0))[1] + xg_val)

    if not buildup_xg:
        return

    df = pd.DataFrame([(pid, n, v) for pid,(n,v) in buildup_xg.items()],
                      columns=['pid','player','xg_buildup'])
    df = df.sort_values('xg_buildup', ascending=True)

    cmap = LinearSegmentedColormap.from_list('bu', ['#2a3a6a', '#6a3a6a', color])

    fig, ax = plt.subplots(figsize=(12, max(6, len(df) * 0.52)), facecolor=BG)
    ax.set_facecolor(BG)
    fig.subplots_adjust(top=0.88, bottom=0.08, left=0.28, right=0.90)

    vmax = df['xg_buildup'].max()
    for i, (_, row) in enumerate(df.iterrows()):
        v = row['xg_buildup']
        bar_col = cmap(v / max(vmax, 0.01))
        ax.barh(i, v, color=bar_col, height=0.65, zorder=3)
        ax.text(v + vmax * 0.01, i, f'{v:.2f}', va='center',
                fontsize=8.5, color='white', fontweight='bold')

    ax.set_yticks(range(len(df)))
    ax.set_yticklabels(df['player'], fontsize=8.5, color='white')
    ax.set_xlabel('xG Buildup', color='#8b9ab0', fontsize=10, fontweight='bold')
    ax.set_xlim(0, vmax * 1.18)
    ax.tick_params(colors='#6a7a8a')
    for spine in ax.spines.values(): spine.set_color(PITCH_LINE)

    fig.text(0.5, 0.01,
             'Sum of xG for all shots in sequences each player was involved in (last 5 actions before shot)',
             ha='center', va='bottom', fontsize=7.5, color='#4a6070')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── CS4: Attacking Participation (per team) ───────────────────────────────────
def plot_attacking_participation(events, shots, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Stacked bar: Buildup actions / Shot assisted / Shots per player."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))

    team_ev    = events[events['contestant_id'] == team_id].sort_values('abs_min').reset_index(drop=True)
    team_shots = shots[shots['contestant_id'] == team_id]
    shot_idxs  = team_ev[team_ev['type_id'].isin(TID_SHOT)].index

    # Shots per player
    shot_counts = team_ev[team_ev['type_id'].isin(TID_SHOT)].groupby('player_id').agg(
        player_name=('player_name','first'), shots=('type_id','count')
    )

    # Shot-assisted (last pass before a shot)
    assist_counts = {}
    for si in shot_idxs:
        for k in range(1, 4):
            if si - k < 0: break
            prev = team_ev.iloc[si - k]
            if prev['type_id'] == TID_PASS:
                pid = str(prev['player_id'])
                assist_counts[pid] = (str(prev['player_name']),
                                      assist_counts.get(pid, (str(prev['player_name']), 0))[1] + 1)
                break

    # Buildup actions: passes + carries in final third (x > 67)
    buildup = team_ev[
        team_ev['type_id'].isin({TID_PASS, TID_TAKEON}) & (team_ev['x'] > 67)
    ].groupby('player_id').agg(
        player_name=('player_name','first'), buildup=('type_id','count')
    )

    # Merge
    all_players = set(shot_counts.index) | set(assist_counts.keys()) | set(buildup.index)
    rows = []
    for pid in all_players:
        pname = (shot_counts.loc[pid,'player_name'] if pid in shot_counts.index
                 else (assist_counts[pid][0] if pid in assist_counts
                       else buildup.loc[pid,'player_name'] if pid in buildup.index else 'Unknown'))
        s = int(shot_counts.loc[pid,'shots']) if pid in shot_counts.index else 0
        a = assist_counts.get(pid, ('',0))[1]
        b = int(buildup.loc[pid,'buildup']) if pid in buildup.index else 0
        total = s + a + b
        if total > 0:
            rows.append({'player': pname, 'shots': s, 'assisted': a, 'buildup': b, 'total': total})

    if not rows:
        return

    df = pd.DataFrame(rows).sort_values('total', ascending=True)

    fig, ax = plt.subplots(figsize=(12, max(6, len(df) * 0.55)), facecolor=BG)
    ax.set_facecolor(BG)
    fig.subplots_adjust(top=0.88, bottom=0.12, left=0.32, right=0.91)

    COL_BUILD  = '#2a3a6a'
    COL_ASSIST = '#7a7a8a'
    COL_SHOT   = color

    for i, (_, row) in enumerate(df.iterrows()):
        x = 0
        for val, col, lbl in [
            (row['buildup'],  COL_BUILD,  'Buildup Action'),
            (row['assisted'], COL_ASSIST, 'Shot Assisted'),
            (row['shots'],    COL_SHOT,   'Shot'),
        ]:
            if val > 0:
                ax.barh(i, val, left=x, color=col, height=0.65, zorder=3,
                        edgecolor=BG, linewidth=0.5)
                ax.text(x + val / 2, i, str(val), ha='center', va='center',
                        fontsize=8.5, color='white', fontweight='bold', zorder=4)
                x += val

    ax.set_yticks(range(len(df)))
    ax.set_yticklabels(df['player'], fontsize=8.5, color='white')
    ax.set_xlabel('Attacking Actions', color='#8b9ab0', fontsize=10, fontweight='bold')
    ax.tick_params(colors='#6a7a8a')
    for spine in ax.spines.values(): spine.set_color(PITCH_LINE)

    from matplotlib.patches import Patch
    legend_items = [Patch(color=COL_BUILD, label='Buildup Action'),
                    Patch(color=COL_ASSIST, label='Shot Assisted'),
                    Patch(color=COL_SHOT,  label='Shot')]
    ax.legend(handles=legend_items, fontsize=8.5, facecolor=BG, labelcolor='white',
              framealpha=0.35, edgecolor=PITCH_LINE, loc='lower right')

    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── CS5: Team Metrics Comparison Dot Chart (shared) ───────────────────────────
def plot_team_metrics(events, shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    """League-style dot chart comparing key match metrics for both teams."""
    home_name = id_to_name.get(home_id, 'Home')
    away_name = id_to_name.get(away_id, 'Away')

    passes_all = events[events['type_id'] == TID_PASS].copy()
    poss_all   = events[events['type_id'].isin(POSSESSION_TYPES)].copy()
    def_all    = events[events['type_id'].isin(DEFENSIVE_TYPES)].copy()

    def team_metric(tid, key):
        h  = shots[shots['contestant_id'] == tid]
        p  = passes_all[passes_all['contestant_id'] == tid]
        po = poss_all[poss_all['contestant_id'] == tid]
        pf = p[(p['end_x'] - p['x']) >= 10].dropna(subset=['end_x','x'])
        pc = p.dropna(subset=['x','y','end_x','end_y'])
        pc['dist'] = np.sqrt((pc['end_x']-pc['x'])**2 + (pc['end_y']-pc['y'])**2)
        long_p = pc[pc['dist'] >= 32]
        ft     = po[po['x'] > 67]
        att_po = po[po['x'] > 50]
        metrics = {
            'Shots':               len(h),
            'Shots on Target':     int(h['is_on_target'].sum()),
            'xG':                  round(float(h['xg'].sum()), 2),
            'Avg xG / Shot':       round(float(h['xg'].mean()) if len(h) else 0, 3),
            'Big Chances':         int(h['is_big_chance'].sum()) if 'is_big_chance' in h else 0,
            'Passes':              len(p),
            'Pass Acc %':          round(p['outcome'].eq(1).mean() * 100, 1) if len(p) else 0,
            'Progressive Passes':  len(pf),
            'Long Balls':          len(long_p),
            'Long Ball Acc %':     round(long_p['outcome'].eq(1).mean() * 100, 1) if len(long_p) else 0,
            'Att. Third Touches':  len(ft),
            'Def. Actions':        len(def_all[def_all['contestant_id'] == tid]),
            'Att. Possessions':    len(att_po),
        }
        return metrics[key]

    metric_keys = [
        'Shots', 'Shots on Target', 'xG', 'Avg xG / Shot',
        'Big Chances', 'Passes', 'Pass Acc %', 'Progressive Passes',
        'Long Balls', 'Long Ball Acc %', 'Att. Third Touches',
        'Def. Actions', 'Att. Possessions',
    ]

    home_vals = [team_metric(home_id, k) for k in metric_keys]
    away_vals = [team_metric(away_id, k) for k in metric_keys]

    n = len(metric_keys)
    fig, ax = plt.subplots(figsize=(12, n * 0.75 + 1.5), facecolor=BG)
    ax.set_facecolor(BG)
    fig.subplots_adjust(top=0.86, bottom=0.06, left=0.30, right=0.70)

    for i, (key, hv, av) in enumerate(zip(metric_keys, home_vals, away_vals)):
        y = n - 1 - i
        # divider line
        ax.axhline(y, color=PITCH_LINE, lw=0.5, alpha=0.4, zorder=1)
        ax.text(0.5, y, key, ha='center', va='center',
                fontsize=8.5, color='#9ab0c0', transform=ax.get_yaxis_transform(),
                zorder=5)
        # Home value left, away value right
        ax.text(-0.04, y, f'{hv}', ha='right', va='center',
                fontsize=11, color=HOME_COL, fontweight='bold',
                transform=ax.get_yaxis_transform())
        ax.text(1.04, y, f'{av}', ha='left', va='center',
                fontsize=11, color=AWAY_COL, fontweight='bold',
                transform=ax.get_yaxis_transform())

        # Comparison bar (home left of centre, away right)
        total = max(hv + av, 0.01)
        h_w = hv / total * 0.8
        a_w = av / total * 0.8
        ax.barh(y, -h_w, left=0, height=0.55, color=HOME_COL, alpha=0.75, zorder=3)
        ax.barh(y,  a_w, left=0, height=0.55, color=AWAY_COL, alpha=0.75, zorder=3)

    ax.set_xlim(-1, 1)
    ax.set_ylim(-0.5, n - 0.5)
    ax.axvline(0, color='white', lw=0.8, alpha=0.25)
    ax.axis('off')

    # Team name headers
    ax.text(-0.25, 1.04, home_name, ha='center', va='bottom',
            fontsize=13, color=HOME_COL, fontweight='bold', transform=ax.transAxes)
    ax.text(1.25, 1.04, away_name, ha='center', va='bottom',
            fontsize=13, color=AWAY_COL, fontweight='bold', transform=ax.transAxes)

    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ══════════════════════════════════════════════════════════════════════════════
# SHOT DANGER PITCH MAPS  (D-series)
# ══════════════════════════════════════════════════════════════════════════════

# ── D1: Danger Weighted Heatmap (per team) ────────────────────────────────────
def plot_danger_heatmap(shots, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """KDE heatmap where each shot location is weighted by its situation_danger score."""
    if 'situation_danger' not in shots.columns:
        print('    [skip] situation_danger not available'); return
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))
    t = shots[shots['contestant_id'] == team_id].dropna(subset=['x','y','situation_danger']).copy()

    fig = plt.figure(figsize=(14, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = VerticalPitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                          linewidth=0.9, half=True, pad_top=4, pad_bottom=2,
                          goal_type='box', goal_alpha=0.6)
    pitch.draw(ax=ax)

    if len(t) >= 3:
        try:
            weights = t['situation_danger'].values.astype(float)
            weights = weights / weights.sum()
            # Expand points by weight for KDE
            repeat_idx = np.repeat(np.arange(len(t)), np.maximum(1, (weights * 300).astype(int)))
            pts = np.vstack([t['y'].values[repeat_idx], t['x'].values[repeat_idx]])
            kde = gaussian_kde(pts, bw_method=0.30)
            gy_ = np.linspace(0, 100, 80); gx_ = np.linspace(50, 100, 60)
            XX, YY = np.meshgrid(gy_, gx_)
            ZZ = kde(np.vstack([XX.ravel(), YY.ravel()])).reshape(XX.shape)
            cmap_name = 'Oranges' if color == HOME_COL else 'Blues'
            ax.contourf(XX, YY, ZZ, levels=14, cmap=cmap_name, alpha=0.70, zorder=2)
        except Exception:
            pass

    # All shots as scatter
    other = t[t['is_goal'] == 0]
    goals = t[t['is_goal'] == 1]
    ax.scatter(other['y'], other['x'],
               s=other['situation_danger'] * 800 + 30,
               color=color, edgecolors='white', linewidth=0.7, alpha=0.75, zorder=4)
    ax.scatter(goals['y'], goals['x'],
               s=goals['situation_danger'] * 800 + 80, marker='*',
               color=C_YELLOW, edgecolors='white', linewidth=0.7, alpha=0.95, zorder=6)

    mean_d = t['situation_danger'].mean() if len(t) else 0
    fig.text(0.5, 0.025,
             f'{len(t)} shots  ·  Mean danger: {mean_d:.3f}  ·  Heat weighted by danger score  ·  ★ = goal',
             ha='center', va='bottom', fontsize=8.5, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── D2: Danger Zone Grid (shared) ─────────────────────────────────────────────
def plot_danger_zone_grid(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    """Pitch grid coloured by avg situation_danger of shots from each zone."""
    if 'situation_danger' not in shots.columns:
        print('    [skip] situation_danger not available'); return
    home_name = id_to_name.get(home_id, 'Home')
    away_name = id_to_name.get(away_id, 'Away')

    COLS, ROWS = 6, 8
    x_edges = np.linspace(0, 100, COLS + 1)
    y_edges = np.linspace(0, 100, ROWS + 1)

    fig, axes = plt.subplots(1, 2, figsize=(18, 9), facecolor=BG)
    fig.subplots_adjust(top=0.84, bottom=0.06, left=0.03, right=0.97, wspace=0.06)

    for ax, team_id, color, tname in [
        (axes[0], home_id, HOME_COL, home_name),
        (axes[1], away_id, AWAY_COL, away_name),
    ]:
        t = shots[shots['contestant_id'] == team_id].dropna(subset=['x','y','situation_danger'])
        pitch = VerticalPitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                              linewidth=0.9, half=True, pad_top=6, pad_bottom=4,
                              goal_type='box', goal_alpha=0.6)
        pitch.draw(ax=ax)
        cmap = LinearSegmentedColormap.from_list('dg', [BG, '#3a0a6e', '#c0392b', '#ffd700'])
        all_vals = []
        for i in range(COLS):
            for j in range(ROWS):
                x0,x1 = x_edges[i], x_edges[i+1]
                y0,y1 = y_edges[j], y_edges[j+1]
                zone = t[(t['x']>=x0)&(t['x']<x1)&(t['y']>=y0)&(t['y']<y1)]
                if len(zone): all_vals.append(zone['situation_danger'].mean())
        vmax = max(all_vals) if all_vals else 0.3
        for i in range(COLS):
            for j in range(ROWS):
                x0,x1 = x_edges[i], x_edges[i+1]
                y0,y1 = y_edges[j], y_edges[j+1]
                # VerticalPitch: x on vertical axis, y on horizontal
                if x0 < 50: continue  # attacking half only
                zone = t[(t['x']>=x0)&(t['x']<x1)&(t['y']>=y0)&(t['y']<y1)]
                if len(zone) == 0: continue
                avg_d = zone['situation_danger'].mean()
                cx, cy = (x0+x1)/2, (y0+y1)/2
                rect = plt.Rectangle((y0, x0), y1-y0, x1-x0,
                                     color=cmap(avg_d / max(vmax, 0.01)),
                                     alpha=0.75, zorder=2)
                ax.add_patch(rect)
                ax.text(cy, cx, f'{avg_d:.3f}\n({len(zone)})',
                        ha='center', va='center', fontsize=7,
                        color='white', fontweight='bold', zorder=5)
        ax.text(0.5, 1.03, f'{tname}  ({len(t)} shots)',
                ha='center', va='bottom', fontsize=11, color=color,
                fontweight='bold', transform=ax.transAxes)

    fig.text(0.5, 0.025,
             'Avg situation danger per zone  ·  Count in brackets  ·  Attacking half only  ·  Gold = highest danger',
             ha='center', va='bottom', fontsize=8, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── D3: High Danger Shots Only (shared) ───────────────────────────────────────
def plot_high_danger_shots(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    """Only shots above the 60th-percentile danger threshold; sized by xG."""
    if 'situation_danger' not in shots.columns:
        print('    [skip] situation_danger not available'); return
    home_name = id_to_name.get(home_id, 'Home')
    away_name = id_to_name.get(away_id, 'Away')

    threshold = float(shots['situation_danger'].quantile(0.60))
    high = shots[shots['situation_danger'] >= threshold].copy()

    fig, axes = plt.subplots(1, 2, figsize=(18, 9), facecolor=BG)
    fig.subplots_adjust(top=0.84, bottom=0.06, left=0.03, right=0.97, wspace=0.06)

    for ax, team_id, color, tname in [
        (axes[0], home_id, HOME_COL, home_name),
        (axes[1], away_id, AWAY_COL, away_name),
    ]:
        t = high[high['contestant_id'] == team_id].copy()
        pitch = VerticalPitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                              linewidth=0.9, half=True, pad_top=4, pad_bottom=2,
                              goal_type='box', goal_alpha=0.6)
        pitch.draw(ax=ax)

        if len(t) >= 3:
            try:
                kde = gaussian_kde(np.vstack([t['y'].values, t['x'].values]), bw_method=0.35)
                gy_ = np.linspace(0, 100, 80); gx_ = np.linspace(50, 100, 60)
                XX, YY = np.meshgrid(gy_, gx_)
                ZZ = kde(np.vstack([XX.ravel(), YY.ravel()])).reshape(XX.shape)
                cmap_k = 'Oranges' if color == HOME_COL else 'Blues'
                ax.contourf(XX, YY, ZZ, levels=10, cmap=cmap_k, alpha=0.50, zorder=2)
            except Exception:
                pass

        goals = t[t['is_goal'] == 1]
        other = t[t['is_goal'] == 0]
        ax.scatter(other['y'], other['x'],
                   s=other['xg'] * 900 + 50,
                   color=color, edgecolors='white', linewidth=0.8, alpha=0.82, zorder=4)
        ax.scatter(goals['y'], goals['x'],
                   s=goals['xg'] * 900 + 120, marker='*',
                   color=C_YELLOW, edgecolors='white', linewidth=0.8, alpha=0.95, zorder=6)

        total_all = len(shots[shots['contestant_id'] == team_id])
        ax.text(0.5, 1.03,
                f'{tname}   {len(t)}/{total_all} high-danger shots  ·  Goals: {len(goals)}  ·  xG: {t["xg"].sum():.2f}',
                ha='center', va='bottom', fontsize=9.5, color=color,
                fontweight='bold', transform=ax.transAxes)

    fig.text(0.5, 0.025,
             f'High-danger threshold: top 40% by situation_danger (≥{threshold:.3f})  ·  Size = xG  ·  ★ = goal',
             ha='center', va='bottom', fontsize=8, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── D4: Danger by Situation Type — Pitch (shared) ────────────────────────────
def plot_danger_situation_pitch(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    """All shots on pitch colour-coded by situation type (open play / corner / FK / etc.)."""
    home_name = id_to_name.get(home_id, 'Home')
    away_name = id_to_name.get(away_id, 'Away')

    SIT_MAP = {
        'is_from_corner': ('Corner',    C_YELLOW),
        'is_free_kick':   ('Free Kick', C_PURPLE),
        'is_fast_break':  ('Fast Break','#00e5ff'),
        'is_header':      ('Header',    C_GREEN),
        'is_big_chance':  ('Big Chance',C_BLUE),
    }
    # fallback colour for open play
    OPEN_COL = HOME_COL

    fig, axes = plt.subplots(1, 2, figsize=(18, 9), facecolor=BG)
    fig.subplots_adjust(top=0.84, bottom=0.08, left=0.03, right=0.97, wspace=0.06)

    for ax, team_id, base_col, tname in [
        (axes[0], home_id, HOME_COL, home_name),
        (axes[1], away_id, AWAY_COL, away_name),
    ]:
        t = shots[shots['contestant_id'] == team_id].copy()
        pitch = VerticalPitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                              linewidth=0.9, half=True, pad_top=4, pad_bottom=2,
                              goal_type='box', goal_alpha=0.6)
        pitch.draw(ax=ax)

        for _, row in t.iterrows():
            col = base_col  # default open play
            for col_key, (lbl, c) in SIT_MAP.items():
                if col_key in row and row[col_key] == 1:
                    col = c; break
            is_goal = (row.get('outcome') == 'Goal')
            xg_val  = float(row.get('xg', 0.1) or 0.1)
            sz = 60 + xg_val * 700
            ax.scatter(row['y'], row['x'],
                       s=sz * 2 if is_goal else sz,
                       marker='*' if is_goal else 'o',
                       color=col, edgecolors='white' if is_goal else col,
                       linewidth=1.2 if is_goal else 0.4,
                       alpha=0.92, zorder=5)

        ax.text(0.5, 1.03,
                f'{tname}   {len(t)} shots  ·  Goals: {int(t["is_goal"].sum())}  ·  xG: {t["xg"].sum():.2f}',
                ha='center', va='bottom', fontsize=9.5, color=base_col,
                fontweight='bold', transform=ax.transAxes)

    # Legend
    legend_items = [(lbl, c) for _, (lbl, c) in SIT_MAP.items()]
    legend_items.insert(0, ('Open Play', HOME_COL))
    handles = [plt.scatter([], [], s=55, color=c, label=lbl, edgecolors='none')
               for lbl, c in legend_items]
    fig.legend(handles=handles, loc='lower center', ncol=len(legend_items),
               fontsize=8.5, facecolor=BG, labelcolor='white',
               framealpha=0.3, edgecolor=PITCH_LINE,
               bbox_to_anchor=(0.5, 0.00))
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── D5: Shot Buildup — Last Pass Before Shot (per team) ───────────────────────
def plot_shot_buildup(events, shots, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """The final pass before each shot — reveals where danger is created from."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))

    team_ev = events[events['contestant_id'] == team_id].sort_values('abs_min').reset_index(drop=True)
    team_shots_idx = team_ev[team_ev['type_id'].isin(TID_SHOT)].index

    buildup_rows = []
    for si in team_shots_idx:
        shot_row = team_ev.iloc[si]
        # Walk back up to 3 events to find the last pass
        for k in range(1, 4):
            if si - k < 0: break
            prev = team_ev.iloc[si - k]
            if prev['type_id'] == TID_PASS and not pd.isna(prev.get('x')):
                buildup_rows.append({
                    'pass_x': prev['x'], 'pass_y': prev['y'],
                    'pass_ex': prev.get('end_x', shot_row['x']),
                    'pass_ey': prev.get('end_y', shot_row['y']),
                    'shot_x': shot_row['x'], 'shot_y': shot_row['y'],
                    'outcome': prev['outcome'],
                })
                break
    bp = pd.DataFrame(buildup_rows).dropna()

    fig = plt.figure(figsize=(14, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.8, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    if len(bp) >= 3:
        try:
            kde = gaussian_kde(np.vstack([bp['pass_x'].values, bp['pass_y'].values]), bw_method=0.30)
            gx = np.linspace(0,100,80); gy = np.linspace(0,100,80)
            XX, YY = np.meshgrid(gx, gy)
            ZZ = kde(np.vstack([XX.ravel(), YY.ravel()])).reshape(XX.shape)
            cmap_k = 'Oranges' if color == HOME_COL else 'Blues'
            ax.contourf(XX, YY, ZZ, levels=10, cmap=cmap_k, alpha=0.40, zorder=2)
        except Exception:
            pass

    for _, row in bp.iterrows():
        alpha = 0.80 if row['outcome'] == 1 else 0.35
        lw    = 1.3  if row['outcome'] == 1 else 0.7
        ax.annotate('', xy=(row['shot_x'], row['shot_y']),
                    xytext=(row['pass_x'], row['pass_y']),
                    arrowprops=dict(arrowstyle='->', color=color, alpha=alpha, lw=lw))
    ax.scatter(bp['pass_x'], bp['pass_y'], s=30, color=color,
               edgecolors='none', alpha=0.65, zorder=4)
    ax.scatter(bp['shot_x'], bp['shot_y'], s=45, color='white',
               edgecolors=color, linewidth=1.0, alpha=0.55, zorder=5, marker='o')

    # KDE of pass origins
    fig.text(0.5, 0.025,
             f'{len(bp)} shot-preceding passes  ·  Bright arrow = pass completed  ·  Heat = pass origin density',
             ha='center', va='bottom', fontsize=8.5, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ══════════════════════════════════════════════════════════════════════════════
# MATRIX / GRID VISUALS  (M-series)
# ══════════════════════════════════════════════════════════════════════════════

# ── M1: xG Probability Matrix (shared) ────────────────────────────────────────
def plot_xg_matrix(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    """Pitch divided into zones; each cell shows avg xG of shots taken from it."""
    COLS, ROWS = 8, 10
    x_edges = np.linspace(0, 100, COLS + 1)
    y_edges = np.linspace(0, 100, ROWS + 1)

    fig, axes = plt.subplots(1, 2, figsize=(18, 9), facecolor=BG)
    fig.subplots_adjust(top=0.84, bottom=0.06, left=0.03, right=0.97, wspace=0.10)

    for ax, team_id, color in [(axes[0], home_id, HOME_COL), (axes[1], away_id, AWAY_COL)]:
        name = id_to_name.get(team_id, str(team_id))
        t    = shots[shots['contestant_id'] == team_id].dropna(subset=['x','y','xg'])

        pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                      linewidth=0.8, goal_type='box', goal_alpha=0.5)
        pitch.draw(ax=ax)

        cmap = LinearSegmentedColormap.from_list('xgmap', [BG, color])
        all_xg = []

        for i in range(COLS):
            for j in range(ROWS):
                x0, x1 = x_edges[i], x_edges[i+1]
                y0, y1 = y_edges[j], y_edges[j+1]
                zone = t[(t['x'] >= x0) & (t['x'] < x1) &
                         (t['y'] >= y0) & (t['y'] < y1)]
                if len(zone) > 0:
                    all_xg.append(zone['xg'].mean())

        vmax = max(all_xg) if all_xg else 0.5

        for i in range(COLS):
            for j in range(ROWS):
                x0, x1 = x_edges[i], x_edges[i+1]
                y0, y1 = y_edges[j], y_edges[j+1]
                zone = t[(t['x'] >= x0) & (t['x'] < x1) &
                         (t['y'] >= y0) & (t['y'] < y1)]
                cx, cy = (x0+x1)/2, (y0+y1)/2
                if len(zone) == 0:
                    rect = plt.Rectangle((y0, x0), y1-y0, x1-x0,
                                         color=BG, alpha=0.0, zorder=2)
                    ax.add_patch(rect)
                    continue
                avg_xg = zone['xg'].mean()
                intensity = avg_xg / max(vmax, 0.01)
                cell_col = cmap(min(intensity, 1.0))
                rect = plt.Rectangle((y0, x0), y1-y0, x1-x0,
                                     color=cell_col, alpha=0.75, zorder=2)
                ax.add_patch(rect)
                ax.text(cy, cx, f'{avg_xg:.2f}\n({len(zone)})',
                        ha='center', va='center', fontsize=6.5,
                        color='white', fontweight='bold', zorder=5)

        ax.text(0.5, 1.03, f'{name}  ({len(t)} shots)',
                ha='center', va='bottom', fontsize=11, color=color,
                fontweight='bold', transform=ax.transAxes)

    fig.text(0.5, 0.025, 'Avg xG per zone  ·  Number in brackets = shot count  ·  Brighter = higher danger',
             ha='center', va='bottom', fontsize=8, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── M2: Pass Completion Matrix (per team) ─────────────────────────────────────
def plot_pass_completion_matrix(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Grid showing pass completion % per zone with count; colour = completion rate."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))

    passes = events[(events['type_id'] == TID_PASS) &
                    (events['contestant_id'] == team_id)].dropna(subset=['x','y']).copy()

    COLS, ROWS = 8, 10
    x_edges = np.linspace(0, 100, COLS + 1)
    y_edges = np.linspace(0, 100, ROWS + 1)

    fig = plt.figure(figsize=(14, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.8, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    cmap_good = LinearSegmentedColormap.from_list('comp', ['#8B0000', '#FFD700', color])

    for i in range(COLS):
        for j in range(ROWS):
            x0, x1 = x_edges[i], x_edges[i+1]
            y0, y1 = y_edges[j], y_edges[j+1]
            zone = passes[(passes['x'] >= x0) & (passes['x'] < x1) &
                          (passes['y'] >= y0) & (passes['y'] < y1)]
            if len(zone) < 2:
                continue
            comp_rate = zone['outcome'].eq(1).mean()
            cx, cy = (x0+x1)/2, (y0+y1)/2
            cell_col = cmap_good(comp_rate)
            rect = plt.Rectangle((y0, x0), y1-y0, x1-x0,
                                 color=cell_col, alpha=0.72, zorder=2)
            ax.add_patch(rect)
            ax.text(cy, cx, f'{comp_rate*100:.0f}%\n({len(zone)})',
                    ha='center', va='center', fontsize=6.5,
                    color='white', fontweight='bold', zorder=5)

    fig.text(0.5, 0.025,
             f'{len(passes)} passes  ·  Cell = completion % (count)  ·  Green = higher accuracy',
             ha='center', va='bottom', fontsize=8.5, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── M3: Touch Density Matrix (per team) ───────────────────────────────────────
def plot_touch_matrix(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Numbered zone grid showing touch count; identifies areas of dominance/absence."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))

    touches = events[events['type_id'].isin(POSSESSION_TYPES) &
                     (events['contestant_id'] == team_id)].dropna(subset=['x','y']).copy()

    COLS, ROWS = 8, 10
    x_edges = np.linspace(0, 100, COLS + 1)
    y_edges = np.linspace(0, 100, ROWS + 1)

    fig = plt.figure(figsize=(14, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.8, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    counts = np.zeros((COLS, ROWS))
    for i in range(COLS):
        for j in range(ROWS):
            x0, x1 = x_edges[i], x_edges[i+1]
            y0, y1 = y_edges[j], y_edges[j+1]
            n = ((touches['x'] >= x0) & (touches['x'] < x1) &
                 (touches['y'] >= y0) & (touches['y'] < y1)).sum()
            counts[i, j] = n
    vmax = counts.max() or 1
    cmap = LinearSegmentedColormap.from_list('td', [BG, color])

    for i in range(COLS):
        for j in range(ROWS):
            x0, x1 = x_edges[i], x_edges[i+1]
            y0, y1 = y_edges[j], y_edges[j+1]
            n = int(counts[i, j])
            if n == 0:
                continue
            intensity = n / vmax
            cell_col  = cmap(intensity ** 0.6)
            rect = plt.Rectangle((y0, x0), y1-y0, x1-x0,
                                 color=cell_col, alpha=0.80, zorder=2)
            ax.add_patch(rect)
            cx, cy = (x0+x1)/2, (y0+y1)/2
            ax.text(cy, cx, str(n),
                    ha='center', va='center', fontsize=8,
                    color='white', fontweight='bold', zorder=5,
                    alpha=0.5 + 0.5 * intensity)

    fig.text(0.5, 0.025,
             f'{len(touches)} total touches  ·  Number = touch count per zone  ·  Brighter = more active',
             ha='center', va='bottom', fontsize=8.5, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── M4: Ball Loss Map (per team) ──────────────────────────────────────────────
def plot_ball_loss_map(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Where the team loses possession: failed passes + failed carries."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))

    losses = events[
        (events['contestant_id'] == team_id) &
        (events['type_id'].isin({TID_PASS, TID_TAKEON})) &
        (events['outcome'] == 0)
    ].dropna(subset=['x','y']).copy()

    fig = plt.figure(figsize=(14, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.8, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    if len(losses) >= 6:
        try:
            kde = gaussian_kde(np.vstack([losses['x'].values, losses['y'].values]), bw_method=0.25)
            gx = np.linspace(0, 100, 80); gy = np.linspace(0, 100, 80)
            XX, YY = np.meshgrid(gx, gy)
            ZZ = kde(np.vstack([XX.ravel(), YY.ravel()])).reshape(XX.shape)
            ax.contourf(XX, YY, ZZ, levels=10, cmap='Reds', alpha=0.55, zorder=2)
        except Exception:
            pass

    ax.scatter(losses['x'], losses['y'], s=35, color='#e05050',
               edgecolors='none', alpha=0.65, zorder=4)

    # Zone counts in thirds
    d3 = losses[losses['x'] < 33]
    m3 = losses[losses['x'].between(33, 67)]
    a3 = losses[losses['x'] > 67]
    for xc, lbl, n in [(16,'Def\nThird',len(d3)), (50,'Mid\nThird',len(m3)), (84,'Att\nThird',len(a3))]:
        ax.text(xc, 95, f'{n}', ha='center', va='center',
                fontsize=13, color='white', fontweight='bold', zorder=6)
        ax.text(xc, 88, lbl, ha='center', va='center',
                fontsize=7, color='#7a9ab0', zorder=6)

    fig.text(0.5, 0.025,
             f'{len(losses)} possession losses  ·  Failed passes + turnovers  ·  Red heat = high-loss zone',
             ha='center', va='bottom', fontsize=8.5, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── M5: xG Conceded Map (per team) ────────────────────────────────────────────
def plot_xg_conceded_map(shots, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Where the opponent shot from against this team — defensive vulnerability map."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))

    # Shots conceded = shots by the opponent
    opp_shots = shots[shots['contestant_id'] != team_id].dropna(subset=['x','y','xg']).copy()

    fig = plt.figure(figsize=(14, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = VerticalPitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                          linewidth=0.9, half=True, pad_top=4, pad_bottom=2,
                          goal_type='box', goal_alpha=0.6)
    pitch.draw(ax=ax)

    if len(opp_shots) >= 3:
        try:
            kde = gaussian_kde(np.vstack([opp_shots['y'].values, opp_shots['x'].values]), bw_method=0.35)
            gy_ = np.linspace(0, 100, 80); gx_ = np.linspace(50, 100, 60)
            XX, YY = np.meshgrid(gy_, gx_)
            ZZ = kde(np.vstack([XX.ravel(), YY.ravel()])).reshape(XX.shape)
            ax.contourf(XX, YY, ZZ, levels=10, cmap='Reds', alpha=0.55, zorder=2)
        except Exception:
            pass

    goals = opp_shots[opp_shots['is_goal'] == 1]
    other = opp_shots[opp_shots['is_goal'] == 0]
    ax.scatter(other['y'], other['x'], s=opp_shots.loc[other.index,'xg']*600 + 20,
               color='#e05050', edgecolors='white', linewidth=0.7, alpha=0.75, zorder=4)
    ax.scatter(goals['y'],  goals['x'],  s=180, marker='*',
               color=C_YELLOW, edgecolors='white', linewidth=0.5, alpha=0.95, zorder=6)

    total_xg = opp_shots['xg'].sum()
    fig.text(0.5, 0.025,
             f'{len(opp_shots)} shots conceded  ·  {len(goals)} goals  ·  '
             f'xG against: {total_xg:.2f}  ·  Size = xG  ·  \u2605 = goal scored',
             ha='center', va='bottom', fontsize=8.5, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── M6: Recovery Zones Matrix (per team) ──────────────────────────────────────
def plot_recovery_matrix(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Grid showing where ball is won back, split by defensive action type."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))

    TYPE_LABELS = {
        TID_TACKLE:   'Tackle',
        TID_INTERC:   'Intercept',
        TID_CLEAR:    'Clearance',
        TID_RECOVERY: 'Recovery',
    }
    TYPE_COLS = {
        TID_TACKLE:   C_BLUE,
        TID_INTERC:   C_GREEN,
        TID_CLEAR:    C_PURPLE,
        TID_RECOVERY: C_YELLOW,
    }
    COLS, ROWS = 6, 8
    x_edges = np.linspace(0, 100, COLS + 1)
    y_edges = np.linspace(0, 100, ROWS + 1)

    fig = plt.figure(figsize=(14, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.8, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    def_ev = events[(events['contestant_id'] == team_id) &
                    (events['type_id'].isin(TYPE_LABELS)) &
                    (events['outcome'] == 1)].dropna(subset=['x','y'])

    cell_totals = np.zeros((COLS, ROWS))
    for i in range(COLS):
        for j in range(ROWS):
            x0,x1 = x_edges[i], x_edges[i+1]
            y0,y1 = y_edges[j], y_edges[j+1]
            n = ((def_ev['x']>=x0)&(def_ev['x']<x1)&(def_ev['y']>=y0)&(def_ev['y']<y1)).sum()
            cell_totals[i,j] = n
    vmax = cell_totals.max() or 1
    cmap = LinearSegmentedColormap.from_list('rec', [BG, color])

    for i in range(COLS):
        for j in range(ROWS):
            x0,x1 = x_edges[i], x_edges[i+1]
            y0,y1 = y_edges[j], y_edges[j+1]
            n = int(cell_totals[i,j])
            if n == 0: continue
            rect = plt.Rectangle((y0,x0), y1-y0, x1-x0,
                                  color=cmap(n/vmax)**1, alpha=0.70, zorder=2)
            ax.add_patch(rect)
            cx, cy = (x0+x1)/2, (y0+y1)/2
            ax.text(cy, cx, str(n), ha='center', va='center',
                    fontsize=9, color='white', fontweight='bold', zorder=5)

    # scatter individual events coloured by type
    for tid, col in TYPE_COLS.items():
        sub = def_ev[def_ev['type_id'] == tid]
        ax.scatter(sub['x'], sub['y'], s=18, color=col, alpha=0.55,
                   edgecolors='none', zorder=4, label=TYPE_LABELS[tid])
    ax.legend(fontsize=8, facecolor=BG, labelcolor='white',
              framealpha=0.35, edgecolor=PITCH_LINE, loc='lower left', ncol=2)

    fig.text(0.5, 0.025,
             f'{len(def_ev)} successful recoveries  ·  Number = count per zone  ·  Dots = individual events',
             ha='center', va='bottom', fontsize=8.5, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── M7: Forward Pass Ratio Matrix (per team) ──────────────────────────────────
def plot_pass_direction_matrix(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Each zone: % of passes that go forward (end_x > x + 5). Identifies build-up intent."""
    color = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL
    name  = id_to_name.get(team_id, str(team_id))

    passes = events[(events['type_id'] == TID_PASS) &
                    (events['contestant_id'] == team_id)].dropna(subset=['x','y','end_x']).copy()
    passes['forward'] = (passes['end_x'] - passes['x']) > 5

    COLS, ROWS = 8, 10
    x_edges = np.linspace(0, 100, COLS + 1)
    y_edges = np.linspace(0, 100, ROWS + 1)

    fig = plt.figure(figsize=(14, 9.5), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.06, 0.92, 0.78])
    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.8, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    cmap = LinearSegmentedColormap.from_list('fwd', ['#8B0000', '#888800', color])

    for i in range(COLS):
        for j in range(ROWS):
            x0, x1 = x_edges[i], x_edges[i+1]
            y0, y1 = y_edges[j], y_edges[j+1]
            zone = passes[(passes['x']>=x0)&(passes['x']<x1)&
                          (passes['y']>=y0)&(passes['y']<y1)]
            if len(zone) < 2: continue
            fwd_pct = zone['forward'].mean()
            cx, cy  = (x0+x1)/2, (y0+y1)/2
            rect = plt.Rectangle((y0,x0), y1-y0, x1-x0,
                                  color=cmap(fwd_pct), alpha=0.75, zorder=2)
            ax.add_patch(rect)
            ax.text(cy, cx, f'{fwd_pct*100:.0f}%\n({len(zone)})',
                    ha='center', va='center', fontsize=6.5,
                    color='white', fontweight='bold', zorder=5)

    fig.text(0.5, 0.025,
             '% of passes going forward (end_x > start_x + 5)  ·  Brighter = more direct/vertical play',
             ha='center', va='bottom', fontsize=8.5, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ── M8: Danger Zone Matrix (shared) ───────────────────────────────────────────
def plot_danger_zone_matrix(shots, home_id, away_id, id_to_name, match_title, subtitle='', save_path=None):
    """Combined shot count + xG sum per zone, both teams on same pitch (home left / away right)."""
    home_name = id_to_name.get(home_id, 'Home')
    away_name = id_to_name.get(away_id, 'Away')

    COLS, ROWS = 6, 8
    x_edges = np.linspace(0, 100, COLS + 1)
    y_edges = np.linspace(0, 100, ROWS + 1)

    fig = plt.figure(figsize=(13, 10), facecolor=BG)
    ax  = fig.add_axes([0.04, 0.07, 0.92, 0.77])
    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.8, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)

    for team_id, tcol, side in [(home_id, HOME_COL, 'home'), (away_id, AWAY_COL, 'away')]:
        t = shots[shots['contestant_id'] == team_id].dropna(subset=['x','y','xg'])
        cmap = LinearSegmentedColormap.from_list('dz', [BG, tcol])
        all_xg_sum = [
            t[(t['x']>=x_edges[i])&(t['x']<x_edges[i+1])&
              (t['y']>=y_edges[j])&(t['y']<y_edges[j+1])]['xg'].sum()
            for i in range(COLS) for j in range(ROWS)
        ]
        vmax = max(all_xg_sum) if all_xg_sum else 0.1

        for i in range(COLS):
            for j in range(ROWS):
                x0,x1 = x_edges[i], x_edges[i+1]
                y0,y1 = y_edges[j], y_edges[j+1]
                zone = t[(t['x']>=x0)&(t['x']<x1)&(t['y']>=y0)&(t['y']<y1)]
                if len(zone) == 0: continue
                xg_sum = zone['xg'].sum()
                cx, cy = (x0+x1)/2, (y0+y1)/2
                # only label on the attacking half for clarity
                if x0 >= 50:
                    rect = plt.Rectangle((y0,x0), y1-y0, x1-x0,
                                          color=cmap(xg_sum/max(vmax,0.01)),
                                          alpha=0.72, zorder=2)
                    ax.add_patch(rect)
                    ax.text(cy, cx, f'{xg_sum:.2f}\n({len(zone)})',
                            ha='center', va='center', fontsize=6.5,
                            color='white', fontweight='bold', zorder=5)

    # team labels
    fig.text(0.22, 0.88, home_name, ha='center', va='bottom',
             fontsize=11, color=HOME_COL, fontweight='bold')
    fig.text(0.78, 0.88, away_name, ha='center', va='bottom',
             fontsize=11, color=AWAY_COL, fontweight='bold')
    fig.text(0.5, 0.025,
             'xG sum per attacking-half zone  ·  Cell value = total xG (shot count)  ·  Attacking half only',
             ha='center', va='bottom', fontsize=8, color='#8b9ab0')
    _title_footer(fig, match_title, subtitle)
    if save_path: plt.savefig(save_path, dpi=DPI, bbox_inches='tight', facecolor=BG)
    plt.show(); plt.close()


# ══════════════════════════════════════════════════════════════════════════════
# ──────────────────────────────────────────────────────────────────────────────
# CannonStats-style dual shot map with central stats box + xPts/win-prob bar
# ──────────────────────────────────────────────────────────────────────────────
def plot_cannon_shotmap(shots, home_id, away_id, id_to_name, match_title,
                        subtitle='', save_path=None):
    from scipy.stats import poisson
    home_name = id_to_name.get(home_id, str(home_id))
    away_name = id_to_name.get(away_id, str(away_id))

    ht = shots[shots['contestant_id'] == home_id]
    at = shots[shots['contestant_id'] == away_id]

    h_goals = int(ht['is_goal'].sum())
    a_goals = int(at['is_goal'].sum())
    h_xg    = ht['xg'].sum()
    a_xg    = at['xg'].sum()
    h_shots = len(ht)
    a_shots = len(at)
    h_ot    = int(ht['on_target'].sum()) if 'on_target' in ht else int((ht['outcome'] == 1).sum())
    a_ot    = int(at['on_target'].sum()) if 'on_target' in at else int((at['outcome'] == 1).sum())
    h_psxg  = ht['psxg'].sum() if 'psxg' in ht.columns else h_xg
    a_psxg  = at['psxg'].sum() if 'psxg' in at.columns else a_xg

    # Poisson win probabilities
    MAX_G = 8
    p_hw = p_draw = p_aw = 0.0
    for i in range(MAX_G):
        for j in range(MAX_G):
            p = poisson.pmf(i, max(h_xg, 0.01)) * poisson.pmf(j, max(a_xg, 0.01))
            if i > j:   p_hw   += p
            elif i == j: p_draw += p
            else:        p_aw   += p
    h_xpts = round(p_hw * 3 + p_draw * 1, 1)
    a_xpts = round(p_aw * 3 + p_draw * 1, 1)

    fig = plt.figure(figsize=(20, 9), facecolor=BG)
    # Left pitch (home) | central col | right pitch (away)
    ax_h   = fig.add_axes([0.01, 0.14, 0.38, 0.68])
    ax_a   = fig.add_axes([0.61, 0.14, 0.38, 0.68])
    ax_cen = fig.add_axes([0.39, 0.14, 0.22, 0.68])
    ax_bar = fig.add_axes([0.01, 0.07, 0.98, 0.05])
    ax_xpt = fig.add_axes([0.01, 0.12, 0.98, 0.02])

    pitch_h = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                    linewidth=0.9, goal_type='box', goal_alpha=0.7)
    pitch_a = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                    linewidth=0.9, goal_type='box', goal_alpha=0.7)
    pitch_h.draw(ax=ax_h)
    pitch_a.draw(ax=ax_a)

    def _scatter_team(ax, df, color, flip=False):
        for _, row in df.iterrows():
            x, y = (float(row['x']), float(row['y']))
            if flip:
                x = 100 - x
                y = 100 - y
            size  = max(50, min(600, float(row['xg']) * 1800 + 30))
            is_g  = bool(row.get('is_goal', False))
            on_t  = bool(row.get('on_target', row.get('outcome', 0) == 1))
            marker = '*' if is_g else ('o' if on_t else 'o')
            fc     = color if is_g else ('none' if not on_t else color)
            ec     = color
            alpha  = 0.95 if is_g else (0.75 if on_t else 0.45)
            lw     = 1.2
            if fc == 'none':
                ax.scatter(x, y, s=size, facecolors='none',
                           edgecolors=ec, marker=marker, alpha=alpha,
                           linewidths=lw, zorder=5)
            else:
                ax.scatter(x, y, s=size, c=[fc],
                           edgecolors=ec, marker=marker, alpha=alpha,
                           linewidths=lw, zorder=5)

            # Annotate goals with player last-name + minute
            if is_g:
                pname = str(row.get('player_name', '')).split()[-1] if row.get('player_name') else ''
                minute = int(row.get('time_min', 0)) if not pd.isna(row.get('time_min', 0)) else 0
                ax.annotate(f"{pname} {minute}\u2019",
                            xy=(x, y), xytext=(x, y + 5),
                            fontsize=6.8, color='white', fontweight='bold',
                            ha='center', va='bottom', zorder=9,
                            arrowprops=dict(arrowstyle='-', color=ec, lw=0.5, alpha=0.5))

    _scatter_team(ax_h, ht, HOME_COL, flip=False)
    _scatter_team(ax_a, at, AWAY_COL, flip=True)

    # Axis titles
    ax_h.set_title(f'{home_name} Shots\n{h_xg:.1f} xG', color=HOME_COL,
                   fontsize=13, fontweight='bold', pad=6)
    ax_a.set_title(f'{away_name} Shots\n{a_xg:.1f} xG', color=AWAY_COL,
                   fontsize=13, fontweight='bold', pad=6)

    # Central stats table
    ax_cen.set_facecolor(BG)
    ax_cen.set_xlim(0, 1); ax_cen.set_ylim(0, 1)
    ax_cen.axis('off')

    rows = [
        ('Goals',     h_goals, a_goals),
        ('xG',        f'{h_xg:.2f}', f'{a_xg:.2f}'),
        ('Shots',     h_shots, a_shots),
        ('On Target', h_ot, a_ot),
        ('PSxG',      f'{h_psxg:.2f}', f'{a_psxg:.2f}'),
    ]
    n = len(rows)
    row_h = 0.16
    y0 = 0.92
    for i, (label, hv, av) in enumerate(rows):
        yc = y0 - i * row_h
        yb = yc - row_h * 0.85
        bg_rect_h = plt.Rectangle((0, yb), 0.46, row_h * 0.82,
                                   facecolor=HOME_COL, alpha=0.85, transform=ax_cen.transAxes)
        bg_rect_a = plt.Rectangle((0.54, yb), 0.46, row_h * 0.82,
                                   facecolor=AWAY_COL, alpha=0.85, transform=ax_cen.transAxes)
        ax_cen.add_patch(bg_rect_h)
        ax_cen.add_patch(bg_rect_a)
        ax_cen.text(0.23, yc - row_h * 0.42, str(hv), ha='center', va='center',
                    color='white', fontsize=13, fontweight='bold',
                    transform=ax_cen.transAxes)
        ax_cen.text(0.77, yc - row_h * 0.42, str(av), ha='center', va='center',
                    color='white', fontsize=13, fontweight='bold',
                    transform=ax_cen.transAxes)
        ax_cen.text(0.50, yc - row_h * 0.42, label, ha='center', va='center',
                    color='white', fontsize=9.5, fontweight='bold',
                    transform=ax_cen.transAxes)

    # xPts labels
    ax_xpt.axis('off')
    ax_xpt.set_xlim(0, 1)
    ax_xpt.text(0.20, 0.5, f'{h_xpts} xpts', ha='center', va='center',
                color='#8b9ab0', fontsize=9, transform=ax_xpt.transAxes)
    ax_xpt.text(0.80, 0.5, f'{a_xpts} xpts', ha='center', va='center',
                color='#8b9ab0', fontsize=9, transform=ax_xpt.transAxes)

    # Win probability bar
    ax_bar.set_xlim(0, 1); ax_bar.set_ylim(0, 1)
    ax_bar.axis('off')
    total = p_hw + p_draw + p_aw
    w_h = p_hw / total; w_d = p_draw / total; w_a = p_aw / total
    ax_bar.barh(0.5, w_h, left=0, height=0.8, color=HOME_COL, align='center')
    ax_bar.barh(0.5, w_d, left=w_h, height=0.8, color='#3a4a5a', align='center')
    ax_bar.barh(0.5, w_a, left=w_h + w_d, height=0.8, color=AWAY_COL, align='center')
    ax_bar.text(w_h / 2, 0.5, f'{p_hw*100:.1f}%', ha='center', va='center',
                color='white', fontsize=9.5, fontweight='bold')
    ax_bar.text(w_h + w_d / 2, 0.5, f'{p_draw*100:.1f}%', ha='center', va='center',
                color='white', fontsize=9.5, fontweight='bold')
    ax_bar.text(w_h + w_d + w_a / 2, 0.5, f'{p_aw*100:.1f}%', ha='center', va='center',
                color='white', fontsize=9.5, fontweight='bold')

    _title_footer(fig, match_title, subtitle)
    _save_or_show(fig, save_path)


# ──────────────────────────────────────────────────────────────────────────────
# Progressive passing table — vertical pitch + per-player breakdown
# ──────────────────────────────────────────────────────────────────────────────
def _is_progressive(x, y, ex, ey):
    """True if pass moves ball toward goal by >10 opta units and forward."""
    if pd.isna(ex) or pd.isna(ey):
        return False
    goal_x = 100.0
    dist_before = abs(goal_x - x)
    dist_after  = abs(goal_x - ex)
    return (dist_after < dist_before - 10) and (ex > x)

def _is_box_completion(ex, ey):
    if pd.isna(ex) or pd.isna(ey):
        return False
    return ex >= 83 and 21 <= ey <= 79

def plot_progressive_passing_table(events, shots, team_id, id_to_name,
                                   match_title, subtitle='', save_path=None):
    tname = id_to_name.get(team_id, str(team_id))
    passes = events[(events['type_id'] == 1) &
                    (events['contestant_id'] == team_id) &
                    (events['outcome'] == 1)].copy()

    # Get pass endpoints from qualifiers
    # load_opta_events stores end_x / end_y directly as snake_case
    if 'end_x' in passes.columns:
        passes['pass_end_x'] = passes['end_x']
        passes['pass_end_y'] = passes['end_y']
    else:
        passes['pass_end_x'] = np.nan
        passes['pass_end_y'] = np.nan

    passes['is_prog'] = passes.apply(
        lambda r: _is_progressive(r['x'], r['y'], r.get('pass_end_x', np.nan), r.get('pass_end_y', np.nan)), axis=1)
    passes['is_box']  = passes.apply(
        lambda r: _is_box_completion(r.get('pass_end_x', np.nan), r.get('pass_end_y', np.nan)), axis=1)
    # Key pass heuristic: pass that ends close to a shot start location
    shot_team = shots[shots['contestant_id'] == team_id]
    def _near_shot(row):
        if pd.isna(row.get('pass_end_x')): return False
        return any((abs(shot_team['x'] - row['pass_end_x']) < 5) &
                   (abs(shot_team['y'] - row['pass_end_y']) < 5))
    passes['is_key'] = passes.apply(_near_shot, axis=1)

    # xA: credit passer with xG of the shot near their pass endpoint
    player_xa = {}
    for _, kp in passes[passes['is_key']].iterrows():
        pid   = str(kp.get('player_id', ''))
        pname = str(kp.get('player_name', 'Unknown'))
        ex, ey = kp.get('pass_end_x', np.nan), kp.get('pass_end_y', np.nan)
        if pd.isna(ex): continue
        nearby = shot_team[(abs(shot_team['x'] - ex) < 5) & (abs(shot_team['y'] - ey) < 5)]
        if len(nearby) > 0:
            xa = float(nearby.iloc[0]['xg'])
            player_xa[pid] = player_xa.get(pid, 0.0) + xa

    # Per-player aggregation using snake_case columns
    grp = passes.groupby('player_name').agg(
        prog  =('is_prog', 'sum'),
        box   =('is_box',  'sum'),
        kp    =('is_key',  'sum'),
    ).astype(int)

    pid_map = passes.groupby('player_name')['player_id'].first().astype(str)
    grp['xa'] = grp.index.map(lambda n: player_xa.get(str(pid_map.get(n, '')), 0.0))

    grp = grp[(grp['prog'] > 0) | (grp['kp'] > 0)].sort_values('prog', ascending=False)

    # Totals
    total_prog = int(passes['is_prog'].sum())
    total_box  = int(passes['is_box'].sum())
    total_kp   = int(passes['is_key'].sum())
    total_xa   = grp['xa'].sum()

    fig = plt.figure(figsize=(20, 11), facecolor=BG)
    ax_pitch = fig.add_axes([0.02, 0.10, 0.36, 0.78])
    ax_table = fig.add_axes([0.40, 0.10, 0.57, 0.78])

    # Vertical pitch (attacking direction = up)
    pitch = VerticalPitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                          linewidth=0.9, goal_type='box', goal_alpha=0.7)
    pitch.draw(ax=ax_pitch)

    prog_col  = '#3ecf8e'   # teal-green
    box_col   = '#48b0e0'   # light blue
    key_col   = '#e05a5a'   # red

    PROG_COL  = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL

    for _, row in passes.iterrows():
        ex = row.get('pass_end_x', np.nan)
        ey = row.get('pass_end_y', np.nan)
        if pd.isna(ex) or pd.isna(ey):
            continue
        is_p = bool(row['is_prog'])
        is_b = bool(row['is_box'])
        is_k = bool(row['is_key'])
        if not (is_p or is_b or is_k):
            continue
        c = key_col if is_k else (box_col if is_b else prog_col)
        alpha = 0.85 if (is_k or is_b) else 0.60
        ax_pitch.annotate('', xy=(ey, ex), xytext=(float(row['y']), float(row['x'])),
                          arrowprops=dict(arrowstyle='->', color=c, lw=1.0, alpha=alpha))

    # Scatter start points
    for _, row in passes[passes['is_prog'] | passes['is_box'] | passes['is_key']].iterrows():
        is_k = bool(row['is_key']); is_b = bool(row['is_box'])
        c = key_col if is_k else (box_col if is_b else prog_col)
        ax_pitch.scatter(float(row['y']), float(row['x']), s=14, c=c, zorder=5, alpha=0.9)

    # Legend
    from matplotlib.lines import Line2D
    leg_els = [Line2D([0],[0], color=prog_col, lw=1.5, label=f'Progressive ({total_prog})'),
               Line2D([0],[0], color=box_col,  lw=1.5, label=f'Box Completion ({total_box})'),
               Line2D([0],[0], color=key_col,  lw=1.5, label=f'Key Pass ({total_kp})')]
    ax_pitch.legend(handles=leg_els, loc='lower center', fontsize=8,
                    facecolor=BG, edgecolor='#3a4a5a', labelcolor='white',
                    framealpha=0.8, ncol=1)

    # Totals header above pitch
    ax_pitch.set_title(
        f'PROG {total_prog}   BOX {total_box}   KP {total_kp}   xA {total_xa:.2f}',
        color='white', fontsize=10, pad=8)

    # Table
    ax_table.set_facecolor(BG)
    ax_table.set_xlim(0, 1); ax_table.set_ylim(0, 1)
    ax_table.axis('off')

    cols = ['PLAYER', 'PROG', 'BOX', 'KP', 'XA']
    col_x = [0.00, 0.62, 0.72, 0.82, 0.92]
    col_align = ['left','center','center','center','center']
    col_colors= ['white', prog_col, box_col, key_col, '#f0c040']

    # Header
    hdr_y = 0.97
    for ci, (cx, ca, cc, ch) in enumerate(zip(col_x, col_align, col_colors, cols)):
        ax_table.text(cx, hdr_y, ch, ha=ca, va='top', color=cc,
                      fontsize=9.5, fontweight='bold', transform=ax_table.transAxes)

    ax_table.axhline(hdr_y - 0.035, color='#3a4a5a', lw=0.8, transform=ax_table.transAxes)

    row_h = min(0.058, 0.90 / max(len(grp), 1))
    for ri, (pname, row) in enumerate(grp.iterrows()):
        y = hdr_y - 0.045 - ri * row_h
        if y < 0.02:
            break
        vals = [pname, str(int(row['prog'])) if row['prog'] > 0 else '-',
                str(int(row['box'])) if row['box'] > 0 else '-',
                str(int(row['kp']))  if row['kp']  > 0 else '-',
                f"{row['xa']:.2f}"   if row['xa'] > 0 else '-']
        bg_alpha = 0.06 if ri % 2 == 0 else 0.0
        if bg_alpha > 0:
            ax_table.add_patch(plt.Rectangle((0, y - row_h * 0.05), 1, row_h * 0.92,
                                             facecolor='white', alpha=bg_alpha,
                                             transform=ax_table.transAxes))
        for ci, (cx, ca, vt) in enumerate(zip(col_x, col_align, vals)):
            fc = 'white' if ci == 0 else col_colors[ci]
            fw = 'normal' if ci == 0 else 'bold'
            ax_table.text(cx, y, vt, ha=ca, va='top', color=fc, fontsize=9,
                          fontweight=fw, transform=ax_table.transAxes)
        ax_table.axhline(y - row_h * 0.08, color='#2a3a4a', lw=0.4,
                         transform=ax_table.transAxes)

    _title_footer(fig, match_title, subtitle)
    _save_or_show(fig, save_path)


# ──────────────────────────────────────────────────────────────────────────────
# Player spotlight — top player per team: 3-panel (passes, received heatmap,
# defensive actions)
# ──────────────────────────────────────────────────────────────────────────────
def plot_player_spotlight(events, shots, team_id, id_to_name,
                          match_title, subtitle='', save_path=None):
    if len(events) == 0:
        return
    tname  = id_to_name.get(team_id, str(team_id))
    TCOLOR = HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL

    team_ev = events[events['contestant_id'] == team_id]
    if 'player_name' not in team_ev.columns or len(team_ev) == 0:
        return

    # Pick player with most events
    top_player = team_ev['player_name'].value_counts().idxmax()
    pev = team_ev[team_ev['player_name'] == top_player]

    passes_made = pev[pev['type_id'] == 1].copy()
    player_xs = pev['x'].dropna().values
    player_ys = pev['y'].dropna().values

    DEF_TYPES = {7, 8, 12, 49}
    def_actions = pev[pev['type_id'].isin(DEF_TYPES)]

    # Pass endpoints from end_x/end_y (snake_case from load_opta_events)
    pm = passes_made.copy()
    if 'end_x' in pm.columns:
        pm['pass_end_x'] = pm['end_x']
        pm['pass_end_y'] = pm['end_y']
    else:
        pm['pass_end_x'] = np.nan
        pm['pass_end_y'] = np.nan

    pm['is_prog'] = pm.apply(lambda r: _is_progressive(r['x'], r['y'],
                              r.get('pass_end_x', np.nan), r.get('pass_end_y', np.nan)), axis=1)
    # Key pass: ends near a shot location
    shot_team_sp = shots[shots['contestant_id'] == team_id]
    def _near_shot_sp(row):
        if pd.isna(row.get('pass_end_x')): return False
        return any((abs(shot_team_sp['x'] - row['pass_end_x']) < 5) &
                   (abs(shot_team_sp['y'] - row['pass_end_y']) < 5))
    pm['is_key'] = pm.apply(_near_shot_sp, axis=1)

    n_passes = len(pm)
    n_prog   = int(pm['is_prog'].sum())
    n_key    = int(pm['is_key'].sum())
    n_comp   = int((pm.get('outcome', pm.get('outcome', pd.Series(dtype=int))) == 1).sum()) if 'outcome' in pm.columns else n_passes

    fig = plt.figure(figsize=(22, 9), facecolor=BG)
    fig.suptitle(top_player, x=0.02, y=0.97, ha='left', va='top',
                 fontsize=22, fontweight='bold', color='white')

    ax1 = fig.add_axes([0.01, 0.12, 0.30, 0.72])
    ax2 = fig.add_axes([0.35, 0.12, 0.30, 0.72])
    ax3 = fig.add_axes([0.69, 0.12, 0.30, 0.72])

    pitch_kw = dict(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                    linewidth=0.8, goal_type='box', goal_alpha=0.6)
    for ax, title in [(ax1, 'OPEN PLAY PASSES'),
                      (ax2, 'PASSES RECEIVED'),
                      (ax3, 'DEFENSIVE ACTIONS')]:
        Pitch(**pitch_kw).draw(ax=ax)
        ax.set_title(title, color='#8b9ab0', fontsize=8.5, pad=5, fontweight='bold')

    # Panel 1 — passes made
    prog_col = '#3ecf8e'; key_col = '#e05a5a'
    for _, row in pm.iterrows():
        ex = row.get('pass_end_x', np.nan)
        ey = row.get('pass_end_y', np.nan)
        is_k = bool(row['is_key']); is_p = bool(row['is_prog'])
        oc   = int(row.get('outcome', 1))
        if pd.isna(ex) or pd.isna(ey):
            ax1.scatter(float(row['x']), float(row['y']), s=12,
                        c=key_col if is_k else (prog_col if is_p else TCOLOR),
                        alpha=0.7, zorder=4)
            continue
        c     = key_col if is_k else (prog_col if is_p else TCOLOR)
        alpha = 0.85 if (is_k or is_p) else (0.50 if oc == 1 else 0.25)
        lw    = 1.2 if (is_k or is_p) else 0.7
        ax1.annotate('', xy=(float(ex), float(ey)),
                     xytext=(float(row['x']), float(row['y'])),
                     arrowprops=dict(arrowstyle='->', color=c, lw=lw, alpha=alpha))
        ax1.scatter(float(row['x']), float(row['y']), s=12, c=c, alpha=alpha, zorder=4)

    # Panel 2 — KDE of all player touch positions (simulate "received passes" heatmap)
    if len(player_xs) > 4:
        try:
            from scipy.stats import gaussian_kde
            kde = gaussian_kde(np.vstack([player_xs, player_ys]), bw_method=0.25)
            gx, gy = np.mgrid[0:100:80j, 0:100:80j]
            z = kde(np.vstack([gx.ravel(), gy.ravel()])).reshape(gx.shape)
            ax2.contourf(gx, gy, z, levels=12, cmap='Reds', alpha=0.7, zorder=3)
        except Exception:
            pass
    ax2.scatter(player_xs, player_ys, s=10, c='white', alpha=0.25, zorder=4)

    # Panel 3 — defensive actions
    def_markers = {7: ('s', '#e07030', 'Tackle'),
                   8: ('D', TCOLOR,    'Interception'),
                   12:('v', '#30a0e0', 'Clearance'),
                   49:('o', '#a060d0', 'Recovery')}
    for tid, (mkr, col, lbl) in def_markers.items():
        sub_d = def_actions[def_actions['type_id'] == tid]
        if len(sub_d) == 0: continue
        ax3.scatter(sub_d['x'], sub_d['y'], s=40, marker=mkr,
                    c=col, alpha=0.85, zorder=5, label=f'{lbl} ({len(sub_d)})')
    if len(def_actions) > 0:
        ax3.legend(loc='lower left', fontsize=7.5, facecolor=BG,
                   edgecolor='#3a4a5a', labelcolor='white', framealpha=0.8)

    # Bottom labels
    fig.text(0.155, 0.09, f'Passes: {n_passes} ({int(n_comp/max(n_passes,1)*100)}%)  '
             f'Progressive: {n_prog}  Key Passes: {n_key}',
             ha='center', va='top', color='#8b9ab0', fontsize=8.5)
    fig.text(0.50, 0.09, f'Touch positions: {len(player_xs)}',
             ha='center', va='top', color='#8b9ab0', fontsize=8.5)
    fig.text(0.845, 0.09, f'Defensive actions: {len(def_actions)}',
             ha='center', va='top', color='#8b9ab0', fontsize=8.5)

    _title_footer(fig, match_title, subtitle)
    _save_or_show(fig, save_path)


# MAIN: generate_game_report
# ══════════════════════════════════════════════════════════════════════════════
def generate_game_report(json_path, out_dir=None, show=True):
    print(f'\n{"─"*60}')
    print(f'Processing: {os.path.basename(json_path)}')

    shots_raw = load_opta_json(json_path)
    if len(shots_raw) == 0:
        print('  ⚠ No shots found — skipping'); return None

    shots = run_models(shots_raw)
    shots, home_id, away_id, home_name, away_name, match_title, fname, competition, match_date = \
        parse_match_meta(json_path, shots)
    id_to_name = {home_id: home_name, away_id: away_name}
    sub = (f'{competition} · ' if competition else '') + match_date

    print(f'  {match_title}')
    for tid, name in id_to_name.items():
        t = shots[shots['contestant_id'] == tid]
        print(f'    {name:20s}  shots={len(t):2d}  goals={int(t["is_goal"].sum())}  '
              f'xG={t["xg"].sum():.2f}  psxG={t["psxg"].sum():.2f}')

    print('  Loading full event data...')
    try:
        events = load_opta_events(json_path)
    except Exception as exc:
        print(f'  ⚠ Event load failed: {exc}'); events = pd.DataFrame()
    try:
        shirt_numbers = load_shirt_numbers(json_path)
    except Exception:
        shirt_numbers = {}

    _SUBFOLDER_RANGES = [
        (3,   '01_ShotMaps'),
        (21,  '02_xGAnalysis'),
        (26,  '03_DangerMaps'),
        (36,  '04_EventsShared'),
        (42,  '05_Defensive'),
        (48,  '06_Attacking'),
        (999, '07_Players'),
    ]

    def sp(suffix):
        if out_dir and SAVE_INDIVIDUAL:
            num_str = suffix.split('_')[0]
            num = int(num_str) if num_str.isdigit() else 999
            folder = '07_Players'
            for threshold, fname_folder in _SUBFOLDER_RANGES:
                if num <= threshold:
                    folder = fname_folder
                    break
            game_dir = os.path.join(out_dir, fname, folder)
            os.makedirs(game_dir, exist_ok=True)
            return os.path.join(game_dir, f'{suffix}.png')
        return None

    def _safe(fn, *args, **kwargs):
        """Call fn(*args, **kwargs), print traceback on failure, never crash the report."""
        try:
            fn(*args, **kwargs)
        except Exception as _e:
            import traceback as _tb
            print(f'    ⚠ {fn.__name__} failed: {_e}')
            _tb.print_exc()
            plt.close('all')

    # ── xG / model visuals ───────────────────────────────────────────────────
    print('  Generating xG visuals  [01–21]...')
    _safe(plot_shot_map, shots, home_id, id_to_name, match_title,
          subtitle=f'{sub} · {home_name} — xG Shot Map', metric='xg',
          save_path=sp('01_xGShotMap_' + home_name))
    _safe(plot_shot_map, shots, away_id, id_to_name, match_title,
          subtitle=f'{sub} · {away_name} — xG Shot Map', metric='xg',
          save_path=sp('02_xGShotMap_' + away_name))
    _safe(plot_xg_flow, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · xG Flow', save_path=sp('03_xGFlow'))
    _safe(plot_cannon_shotmap, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · Shot Map Overview', save_path=sp('03b_CannonShotMap'))
    _safe(plot_score_probability, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · Score Outcome Probability', save_path=sp('04_ScoreProb'))
    _safe(plot_shot_timeline, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · Shot Timeline', save_path=sp('05_ShotTimeline'))
    _safe(plot_goal_frame, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · Goal Frame Placement', save_path=sp('06_GoalFrame'))
    _safe(plot_stats_dashboard, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · Match Statistics', save_path=sp('07_Stats'))
    _safe(plot_shot_scatter, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · Shot Quality', save_path=sp('08_ShotQuality'))
    _safe(plot_pressure_analysis, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · Pressure Analysis', save_path=sp('09_Pressure'))
    _safe(plot_shot_types, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · Shot Type Breakdown', save_path=sp('10_ShotTypes'))
    _safe(plot_placement_quality, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · Placement Quality', save_path=sp('11_Placement'))
    _safe(plot_pontarget_map, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · P(On Target) Map', save_path=sp('12_POnTarget'))
    _safe(plot_situation_danger, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · Situation Danger', save_path=sp('13_Danger'))
    _safe(plot_danger_flow, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · Danger Flow', save_path=sp('14_DangerFlow'))
    _safe(plot_pontarget_by_type, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · P(On Target) by Type', save_path=sp('15_POnTarget_ByType'))
    _safe(plot_xgot_map, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · xGOT Map', save_path=sp('16_xGOT'))
    _safe(plot_multi_outcome, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · Multi-Outcome', save_path=sp('17_MultiOutcome'))
    _safe(plot_danger_vs_pontarget, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · Danger vs P(OT)', save_path=sp('18_DangerVsPOT'))
    _safe(plot_shot_zone_grid, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · Shot Zone Grid', save_path=sp('19_ZoneGrid'))
    _safe(plot_model_radar, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · Model Radar', save_path=sp('20_Radar'))
    _safe(plot_model_summary, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · Model Summary', save_path=sp('21_ModelSummary'))

    # ── Shot Danger Pitch Maps ───────────────────────────────────────────────
    print('  Generating shot danger maps  [22–26]...')
    _safe(plot_danger_zone_grid, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · Danger Zone Grid', save_path=sp('22_DangerZoneGrid'))
    _safe(plot_high_danger_shots, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · High Danger Shots', save_path=sp('23_HighDangerShots'))
    _safe(plot_danger_situation_pitch, shots, home_id, away_id, id_to_name, match_title,
          subtitle=f'{sub} · Danger by Situation', save_path=sp('24_DangerSituation'))
    for _tid, _tname in id_to_name.items():
        _safe(plot_danger_heatmap, shots, _tid, id_to_name, match_title,
              subtitle=f'{sub} · {_tname} — Danger Heatmap',
              save_path=sp(f'25_DangerHeatmap_{_tname}'))
        _safe(plot_shot_buildup, events if len(events) > 0 else pd.DataFrame(),
              shots, _tid, id_to_name, match_title,
              subtitle=f'{sub} · {_tname} — Shot Buildup',
              save_path=sp(f'26_ShotBuildup_{_tname}'))

    if len(events) == 0:
        print('  ⚠ No event data — skipping event visuals')
    else:
        # ── Shared event visuals ─────────────────────────────────────────────
        print('  Generating shared event visuals  [27–36]...')
        _safe(plot_pass_network, events, home_id, away_id, id_to_name, match_title,
              subtitle=f'{sub} · Pass Network', save_path=sp('27_PassNetwork'))
        _safe(plot_field_tilt, events, shots, home_id, away_id, id_to_name, match_title,
              subtitle=f'{sub} · Field Tilt', save_path=sp('28_FieldTilt'))
        _safe(plot_field_tilt_mirror, events, shots, home_id, away_id, id_to_name, match_title,
              subtitle=f'{sub} · Field Tilt Mirror', save_path=sp('29_FieldTiltMirror'))
        _safe(plot_zones_of_control, events, home_id, away_id, id_to_name, match_title,
              subtitle=f'{sub} · Zones of Control', save_path=sp('30_ZonesOfControl'))
        _safe(plot_line_height, events, home_id, away_id, id_to_name, match_title,
              subtitle=f'{sub} · Defensive Line Height', save_path=sp('31_LineHeight'))
        _safe(plot_team_metrics, events, shots, home_id, away_id, id_to_name, match_title,
              subtitle=f'{sub} · Team Metrics', save_path=sp('32_TeamMetrics'))
        _safe(plot_momentum, events, shots, home_id, away_id, id_to_name, match_title,
              subtitle=f'{sub} · Momentum', save_path=sp('33_Momentum'))
        _safe(plot_ppda, events, home_id, away_id, id_to_name, match_title,
              subtitle=f'{sub} · PPDA', save_path=sp('34_PPDA'))
        _safe(plot_xg_matrix, shots, home_id, away_id, id_to_name, match_title,
              subtitle=f'{sub} · xG Matrix', save_path=sp('35_xGMatrix'))
        _safe(plot_danger_zone_matrix, shots, home_id, away_id, id_to_name, match_title,
              subtitle=f'{sub} · Danger Zone Matrix', save_path=sp('36_DangerMatrix'))

        # ── Per-team event visuals ───────────────────────────────────────────
        for team_id, tname in id_to_name.items():
            print(f'  Generating {tname} visuals  [37–70]...')
            _safe(plot_pass_network_single, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Pass Network',
                  save_path=sp(f'37_PassNetwork_{tname}'), shirt_numbers=shirt_numbers)
            _safe(plot_pass_map, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Pass Map', save_path=sp(f'38_PassMap_{tname}'))
            _safe(plot_carry_map, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Carry Map', save_path=sp(f'39_CarryMap_{tname}'))
            _safe(plot_box_entry, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Box Entry', save_path=sp(f'40_BoxEntry_{tname}'))
            _safe(plot_half_spaces, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Half Spaces', save_path=sp(f'41_HalfSpaces_{tname}'))
            _safe(plot_zone14, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Zone 14', save_path=sp(f'42_Zone14_{tname}'))
            _safe(plot_corner_map, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Corner Map', save_path=sp(f'43_CornerMap_{tname}'))
            _safe(plot_duels_map, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Duels Map', save_path=sp(f'44_DuelsMap_{tname}'))
            _safe(plot_transition_triggers, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Transitions', save_path=sp(f'45_TransitionTriggers_{tname}'))
            _safe(plot_player_heatmap, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Player Heatmaps', save_path=sp(f'46_PlayerHeatmap_{tname}'))
            _safe(plot_box_defending, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Box Defending', save_path=sp(f'47_BoxDefending_{tname}'))
            _safe(plot_free_kick_map, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Free Kicks', save_path=sp(f'48_FreeKickMap_{tname}'))
            _safe(plot_defensive_actions, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Defensive Actions', save_path=sp(f'49_DefensiveActions_{tname}'))
            _safe(plot_progressive_passes, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Progressive Passes', save_path=sp(f'50_ProgressivePasses_{tname}'))
            _safe(plot_touches_in_box, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Touches in Box', save_path=sp(f'51_TouchesInBox_{tname}'))
            _safe(plot_press_map, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Press Map', save_path=sp(f'52_PressMap_{tname}'))
            _safe(plot_shape_comparison, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Shape', save_path=sp(f'53_ShapeComparison_{tname}'))
            _safe(plot_cross_map, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Crosses', save_path=sp(f'54_CrossMap_{tname}'))
            _safe(plot_key_pass_map, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Key Passes', save_path=sp(f'55_KeyPassMap_{tname}'))
            _safe(plot_long_ball_map, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Long Balls', save_path=sp(f'56_LongBallMap_{tname}'))
            _safe(plot_attacking_entries, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Attacking Entries', save_path=sp(f'57_AttackingEntries_{tname}'))
            _safe(plot_width_of_play, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Width of Play', save_path=sp(f'58_WidthOfPlay_{tname}'))
            _safe(plot_switch_map, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Switches', save_path=sp(f'59_SwitchMap_{tname}'))
            _safe(plot_gpa, events, shots, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — GPA', save_path=sp(f'60_GPA_{tname}'))
            _safe(plot_xg_buildup_player, events, shots, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — xG Buildup', save_path=sp(f'61_xGBuildup_{tname}'))
            _safe(plot_attacking_participation, events, shots, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Attacking Participation', save_path=sp(f'62_AttackingParticipation_{tname}'))
            _safe(plot_pass_completion_matrix, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Pass Completion Matrix', save_path=sp(f'63_PassMatrix_{tname}'))
            _safe(plot_touch_matrix, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Touch Density', save_path=sp(f'64_TouchMatrix_{tname}'))
            _safe(plot_ball_loss_map, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Ball Loss', save_path=sp(f'65_BallLoss_{tname}'))
            _safe(plot_xg_conceded_map, shots, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — xG Conceded', save_path=sp(f'66_xGConceded_{tname}'))
            _safe(plot_recovery_matrix, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Recovery Zones', save_path=sp(f'67_RecoveryMatrix_{tname}'))
            _safe(plot_pass_direction_matrix, events, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Forward Pass Ratio', save_path=sp(f'68_PassDirection_{tname}'))
            _safe(plot_progressive_passing_table, events, shots, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Progressive Passing Table', save_path=sp(f'69_ProgPassTable_{tname}'))
            _safe(plot_player_spotlight, events, shots, team_id, id_to_name, match_title,
                  subtitle=f'{sub} · {tname} — Player Spotlight', save_path=sp(f'70_PlayerSpotlight_{tname}'))

    if out_dir:
        game_dir = os.path.join(out_dir, fname)
        os.makedirs(game_dir, exist_ok=True)
        shots.to_csv(os.path.join(game_dir, 'shots.csv'), index=False)
        if len(events) > 0:
            events.to_csv(os.path.join(game_dir, 'events.csv'), index=False)

    total = 21 + 4 + 2*2 + (10 + 34*2 if len(events) > 0 else 0)
    print(f'  ✓ Done — {fname}  ({total} visuals)')
    return shots


# ══════════════════════════════════════════════════════════════════════════════
# ADVANCED ADD-ON VISUALS
# ══════════════════════════════════════════════════════════════════════════════

# ============================================================================
# ADVANCED ADD-ON VISUALS  (Opta format, report palette)
# Adds/improves: average positions, pass clusters, xT heatmap, GK distribution,
# possession chains, final-third entries, pressure regains, progressive lanes,
# territory control, and player influence.
# ============================================================================

import os, re, json, warnings
from collections import Counter
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from mplsoccer import Pitch

# Standard xT grid (12 columns x 8 rows, left to right attacking direction).
_XT_GRID = np.array([
    [0.0064,0.0078,0.0084,0.0096,0.0113,0.0141,0.0169,0.0212,0.0276,0.0349,0.0379,0.0212],
    [0.0075,0.0088,0.0099,0.0117,0.0143,0.0176,0.0212,0.0265,0.0349,0.0440,0.0604,0.1081],
    [0.0089,0.0100,0.0114,0.0135,0.0169,0.0212,0.0265,0.0349,0.0440,0.0604,0.1081,0.2575],
    [0.0094,0.0106,0.0122,0.0148,0.0187,0.0240,0.0308,0.0407,0.0560,0.0812,0.1479,0.2575],
    [0.0094,0.0106,0.0122,0.0148,0.0187,0.0240,0.0308,0.0407,0.0560,0.0812,0.1479,0.2575],
    [0.0089,0.0100,0.0114,0.0135,0.0169,0.0212,0.0265,0.0349,0.0440,0.0604,0.1081,0.2575],
    [0.0075,0.0088,0.0099,0.0117,0.0143,0.0176,0.0212,0.0265,0.0349,0.0440,0.0604,0.1081],
    [0.0064,0.0078,0.0084,0.0096,0.0113,0.0141,0.0169,0.0212,0.0276,0.0349,0.0379,0.0212],
])

PANEL_BG = '#161b22'
TEXT_MUTED = '#7a9ab0'
TEXT_SOFT = '#9ab0c0'
BAD_COL = '#627488'


def _xt_at(x, y):
    if pd.isna(x) or pd.isna(y):
        return np.nan
    col = min(11, max(0, int(float(x) / 100 * 12)))
    row = min(7, max(0, int(float(y) / 100 * 8)))
    return float(_XT_GRID[row, col])


def _team_color(team_id, id_to_name):
    return HOME_COL if team_id == list(id_to_name.keys())[0] else AWAY_COL


def _team_name(team_id, id_to_name):
    return id_to_name.get(team_id, str(team_id))


def _slug(value):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value)).strip('_') or 'team'


def _style_panel(ax):
    ax.set_facecolor(PANEL_BG)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor(PITCH_LINE)
        spine.set_linewidth(0.9)


def _pitch_axes(width=16, height=9, panel=True):
    fig = plt.figure(figsize=(width, height), facecolor=BG)
    if panel:
        ax = fig.add_axes([0.04, 0.10, 0.76, 0.72])
        side = fig.add_axes([0.83, 0.12, 0.14, 0.68])
        _style_panel(side)
    else:
        ax = fig.add_axes([0.05, 0.11, 0.90, 0.72])
        side = None
    pitch = Pitch(pitch_type='opta', pitch_color=BG, line_color=PITCH_LINE,
                  linewidth=0.9, goal_type='box', goal_alpha=0.5)
    pitch.draw(ax=ax)
    return fig, ax, side


def _metric_panel(ax, rows, title='KEY NUMBERS'):
    _style_panel(ax)
    ax.text(0.50, 0.96, title, color=TEXT_MUTED, fontsize=8.5,
            fontweight='bold', ha='center', va='top', transform=ax.transAxes)
    n = max(len(rows), 1)
    ys = np.linspace(0.80, 0.16, n)
    for y, (value, label, color) in zip(ys, rows):
        ax.text(0.50, y, str(value), color=color, fontsize=20,
                fontweight='bold', ha='center', va='center', transform=ax.transAxes)
        ax.text(0.50, y - 0.085, label, color=TEXT_MUTED, fontsize=7.5,
                fontweight='bold', ha='center', va='center', transform=ax.transAxes)


def _save_visual(fig, match_title, subtitle, save_path):
    _title_footer(fig, match_title, subtitle)
    _save_or_show(fig, save_path)


def _ordered_events(events):
    t = events.copy()
    if 'event_id' in t.columns:
        return t.sort_values('event_id')
    if 'abs_min' in t.columns:
        return t.sort_values(['abs_min', 'period_id'])
    return t.reset_index().sort_values('index')


def _movement_events(events, team_id, successful_only=True):
    t = events[(events['contestant_id'] == team_id) &
               (events['type_id'].isin({TID_PASS, TID_TAKEON}))]
    t = t.dropna(subset=['x', 'y', 'end_x', 'end_y']).copy()
    if successful_only:
        t = t[t['outcome'] == 1]
    if len(t):
        t['xt_start'] = t.apply(lambda r: _xt_at(r['x'], r['y']), axis=1)
        t['xt_end'] = t.apply(lambda r: _xt_at(r['end_x'], r['end_y']), axis=1)
        t['xt_gained'] = (t['xt_end'] - t['xt_start']).fillna(0)
        t['progress'] = t['end_x'] - t['x']
        t['distance'] = np.sqrt((t['end_x'] - t['x'])**2 + (t['end_y'] - t['y'])**2)
    return t


def _draw_arrows(ax, rows, color, alpha=0.70, lw=1.2, max_rows=None):
    if max_rows is not None and len(rows) > max_rows:
        rows = rows.head(max_rows)
    for _, row in rows.iterrows():
        ax.annotate('', xy=(row['end_x'], row['end_y']), xytext=(row['x'], row['y']),
                    arrowprops=dict(arrowstyle='->', color=color, lw=lw,
                                    alpha=alpha, mutation_scale=12))


def load_match_events(path):
    """Compatibility loader for the add-on runner: returns events plus match metadata."""
    events = load_opta_events(path)
    with open(path, 'r', encoding='utf-8-sig') as f:
        raw = json.load(f)
    base = Path(path).stem
    try:
        date_str, match_str = base.split('_', 1)
        home_name, away_name = match_str.split(' - ', 1)
    except ValueError:
        date_str, home_name, away_name = '', 'Home', 'Away'

    score = raw.get('matchDetails', {}).get('scores', {}).get('ft', {})
    competition = (raw.get('matchInfo', {}).get('competition', {}).get('name', '')
                   or raw.get('matchDetails', {}).get('competitionName', '')
                   or raw.get('matchDetails', {}).get('competition', {}).get('name', ''))

    ids = [str(x) for x in events['contestant_id'].dropna().unique().tolist() if str(x)]
    home_id = ids[0] if ids else 'home'
    match_events = raw.get('liveData', {}).get('event', raw.get('event', []))
    try:
        h_goals = int(score.get('home'))
        goal_counts = Counter(str(e.get('contestantId', '')) for e in match_events
                              if str(e.get('typeId')) == '16')
        candidates = [cid for cid in ids if goal_counts.get(cid, 0) == h_goals]
        if len(candidates) == 1:
            home_id = candidates[0]
        else:
            for e in match_events:
                if int(e.get('periodId', 0) or 0) == 16 and str(e.get('contestantId', '')) in ids:
                    home_id = str(e.get('contestantId', ''))
                    break
    except Exception:
        pass
    away_id = next((cid for cid in ids if cid != home_id), home_id)
    id_to_name = {home_id: home_name, away_id: away_name}
    meta = {
        'home_id': home_id, 'away_id': away_id, 'home_name': home_name,
        'away_name': away_name, 'id_to_name': id_to_name, 'score': score,
        'competition': competition, 'match_date': date_str,
    }
    return events, meta


# 1. Average Positions / Formation
# ----------------------------------------------------------------------------
def plot_avg_positions(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    touch_types = {TID_PASS, TID_TAKEON, TID_TOUCH, TID_TACKLE, TID_INTERC, TID_CLEAR, TID_RECOVERY}
    t = events[(events['contestant_id'] == team_id) & events['type_id'].isin(touch_types)].dropna(subset=['x', 'y'])
    if len(t) == 0:
        return

    grp = (t.groupby('player_name')
             .agg(avg_x=('x', 'mean'), avg_y=('y', 'mean'), touches=('x', 'count'))
             .reset_index())
    grp = grp[grp['touches'] >= 3].sort_values('avg_x')
    if len(grp) == 0:
        return

    moves = _movement_events(events, team_id, successful_only=True)
    links = (moves.groupby(['player_name'])
                  .agg(pass_count=('x', 'count'), avg_start_x=('x', 'mean'), avg_start_y=('y', 'mean'),
                       avg_end_x=('end_x', 'mean'), avg_end_y=('end_y', 'mean'))
                  .reset_index()) if len(moves) else pd.DataFrame()

    col = _team_color(team_id, id_to_name)
    name = _team_name(team_id, id_to_name)
    fig, ax, side = _pitch_axes()

    max_t = max(float(grp['touches'].max()), 1)
    for _, row in grp.iterrows():
        size = 220 + 760 * row['touches'] / max_t
        ax.scatter(row['avg_x'], row['avg_y'], s=size, color=col, edgecolors='white',
                   linewidth=1.2, zorder=5, alpha=0.88)
        label = str(row['player_name']).split()[-1][:10]
        ax.text(row['avg_x'], row['avg_y'], label, color='white', fontsize=6.6,
                fontweight='bold', ha='center', va='center', zorder=6)

    if len(links):
        top_links = links.nlargest(7, 'pass_count')
        for _, row in top_links.iterrows():
            ax.plot([row['avg_start_x'], row['avg_end_x']], [row['avg_start_y'], row['avg_end_y']],
                    color='white', alpha=0.17, lw=1.2, zorder=2)

    _metric_panel(side, [
        (len(grp), 'PLAYERS', col),
        (int(grp['touches'].sum()), 'TOUCHES', C_BLUE),
        (f"{grp['avg_x'].mean():.0f}", 'AVG X', C_YELLOW),
    ], title='FORMATION')
    _save_visual(fig, match_title, f'{subtitle} · {name} - Average Positions', save_path)


# 2. Pass Clusters
# ----------------------------------------------------------------------------
def plot_pass_clusters(events, team_id, id_to_name, match_title, subtitle='', n_clusters=6, save_path=None):
    t = _movement_events(events, team_id, successful_only=True)
    t = t[t['type_id'] == TID_PASS]
    if len(t) < 15:
        return

    try:
        from sklearn.cluster import KMeans
        k = min(n_clusters, max(2, len(t) // 25))
        km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(t[['x', 'y', 'end_x', 'end_y']].values)
        t = t.copy()
        t['cluster'] = km.labels_
    except Exception:
        k = min(n_clusters, 4)
        t = t.copy()
        t['cluster'] = pd.cut(t['x'], bins=k, labels=False, include_lowest=True)

    name = _team_name(team_id, id_to_name)
    cluster_cols = [HOME_COL, AWAY_COL, C_GREEN, C_YELLOW, C_PURPLE, '#f0f6fc']
    fig, ax, side = _pitch_axes()

    rows = []
    for idx, grp in t.groupby('cluster'):
        idx = int(idx)
        color = cluster_cols[idx % len(cluster_cols)]
        sample = grp.sample(min(90, len(grp)), random_state=42) if len(grp) > 90 else grp
        _draw_arrows(ax, sample, color, alpha=0.13, lw=0.6)
        cx, cy, ex, ey = grp[['x', 'y', 'end_x', 'end_y']].mean()
        ax.annotate('', xy=(ex, ey), xytext=(cx, cy),
                    arrowprops=dict(arrowstyle='->', color=color, lw=3.4, mutation_scale=20, alpha=0.96))
        ax.scatter(cx, cy, s=160, color=color, edgecolors=BG, linewidth=1.3, zorder=7)
        ax.text(cx, cy, str(idx + 1), color=BG, fontsize=8, fontweight='bold', ha='center', va='center', zorder=8)
        rows.append((f"{len(grp)}", f'CLUSTER {idx + 1}', color))

    _metric_panel(side, rows[:6], title='PASS FAMILIES')
    _save_visual(fig, match_title, f'{subtitle} · {name} - Pass Clusters', save_path)


# 3. xT Heatmap
# ----------------------------------------------------------------------------
def plot_xt_heatmap(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    t = _movement_events(events, team_id, successful_only=True)
    t = t[t['xt_gained'] > 0]
    if len(t) == 0:
        return

    col = _team_color(team_id, id_to_name)
    name = _team_name(team_id, id_to_name)
    fig, ax, side = _pitch_axes()

    heat, xedges, yedges = np.histogram2d(t['end_x'], t['end_y'], bins=[12, 8], range=[[0, 100], [0, 100]], weights=t['xt_gained'])
    cmap = LinearSegmentedColormap.from_list('xt_addon', [BG, '#1b3a4f', col, C_YELLOW])
    ax.imshow(heat.T, extent=[0, 100, 0, 100], origin='lower', cmap=cmap, alpha=0.78, aspect='auto', zorder=1)

    top = t.nlargest(18, 'xt_gained')
    max_xt = max(float(top['xt_gained'].max()), 0.001)
    for _, row in top.iterrows():
        ax.annotate('', xy=(row['end_x'], row['end_y']), xytext=(row['x'], row['y']),
                    arrowprops=dict(arrowstyle='->', color='white', lw=1.0 + 3.0 * row['xt_gained'] / max_xt,
                                    alpha=0.80, mutation_scale=14))
    ax.scatter(top['end_x'], top['end_y'], s=28, color=C_YELLOW, edgecolors=BG, linewidth=0.5, zorder=4)

    _metric_panel(side, [
        (f"{t['xt_gained'].sum():.3f}", 'TOTAL xT', col),
        (f"{t['xt_gained'].mean():.4f}", 'PER ACTION', C_YELLOW),
        (len(t), 'POSITIVE ACTS', C_BLUE),
    ], title='THREAT')
    _save_visual(fig, match_title, f'{subtitle} · {name} - xT Heatmap', save_path)


# 4. Goalkeeper / Deep Distribution
# ----------------------------------------------------------------------------
def plot_gk_distribution(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    t = events[(events['contestant_id'] == team_id) & events['type_id'].isin({TID_PASS, TID_CLEAR})]
    t = t.dropna(subset=['x', 'y', 'end_x', 'end_y']).copy()
    gk = t[t['x'] < 25]
    if len(gk) == 0:
        return

    col = _team_color(team_id, id_to_name)
    name = _team_name(team_id, id_to_name)
    complete = gk[gk['outcome'] == 1]
    incomplete = gk[gk['outcome'] != 1]
    fig, ax, side = _pitch_axes()

    _draw_arrows(ax, incomplete, BAD_COL, alpha=0.34, lw=0.8)
    _draw_arrows(ax, complete, col, alpha=0.72, lw=1.2)
    ax.scatter(gk['end_x'], gk['end_y'], s=26, color='white', alpha=0.28, zorder=4)
    ax.axvline(50, color=PITCH_LINE, lw=1, ls='--', alpha=0.65)

    dists = np.sqrt((gk['end_x'] - gk['x'])**2 + (gk['end_y'] - gk['y'])**2)
    long_pct = float((dists >= 45).mean() * 100) if len(dists) else 0
    pct = len(complete) / len(gk) * 100 if len(gk) else 0
    _metric_panel(side, [
        (len(gk), 'DEEP DIST.', col),
        (f'{pct:.0f}%', 'COMPLETE', C_GREEN),
        (f'{long_pct:.0f}%', '45M+ LONG', C_YELLOW),
    ], title='BUILD START')
    _save_visual(fig, match_title, f'{subtitle} · {name} - GK Distribution', save_path)


# 5. Possession Chain Lengths
# ----------------------------------------------------------------------------
def plot_possession_duration(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    t = _ordered_events(events[(events['contestant_id'] == team_id) & (events['type_id'] == TID_PASS)])
    if len(t) == 0:
        return

    chains, cur = [], 0
    for _, row in t.iterrows():
        cur += 1
        if row['outcome'] != 1:
            chains.append(cur)
            cur = 0
    if cur:
        chains.append(cur)
    chains = np.array(chains, dtype=float)
    if len(chains) == 0:
        return

    col = _team_color(team_id, id_to_name)
    name = _team_name(team_id, id_to_name)
    bins = [1, 2, 3, 4, 5, 6, 11, max(31, int(chains.max()) + 1)]
    labels = ['1', '2', '3', '4', '5', '6-10', '11+']
    counts = np.array([((chains >= lo) & (chains < hi)).sum() for lo, hi in zip(bins[:-1], bins[1:])], dtype=float)
    pcts = counts / max(len(chains), 1) * 100

    fig, (ax, side) = plt.subplots(1, 2, figsize=(16, 8), facecolor=BG, gridspec_kw={'width_ratios': [2.2, 0.85]})
    fig.subplots_adjust(left=0.07, right=0.96, bottom=0.15, top=0.82, wspace=0.08)
    ax.set_facecolor(BG)
    colors = [BAD_COL if i < 2 else '#3f5a70' if i < 5 else col for i in range(len(labels))]
    bars = ax.bar(labels, pcts, width=0.72, color=colors, edgecolor=BG, linewidth=0.6, zorder=3)
    for bar, p in zip(bars, pcts):
        if p > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, p + 0.6, f'{p:.0f}%', color='white',
                    fontsize=9, fontweight='bold', ha='center')
    ax.set_xlabel('Passes in chain', color=TEXT_MUTED, fontsize=10)
    ax.set_ylabel('% of chains', color=TEXT_MUTED, fontsize=10)
    ax.tick_params(colors=TEXT_MUTED)
    ax.grid(axis='y', color=PITCH_LINE, alpha=0.35, zorder=0)
    for spine in ax.spines.values():
        spine.set_edgecolor(PITCH_LINE)
    ax.set_ylim(0, max(pcts) * 1.22 + 2)

    _metric_panel(side, [
        (f'{chains.mean():.1f}', 'AVG LENGTH', C_YELLOW),
        (f'{np.median(chains):.0f}', 'MEDIAN', C_BLUE),
        (f'{(chains >= 6).mean() * 100:.0f}%', '6+ PASSES', col),
    ], title='CONTROL')
    _save_visual(fig, match_title, f'{subtitle} · {name} - Possession Chain Lengths', save_path)


# 6. Final Third Entries
# ----------------------------------------------------------------------------
def plot_final_third_entries(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    t = _movement_events(events, team_id, successful_only=True)
    entries = t[(t['x'] < 66.7) & (t['end_x'] >= 66.7)]
    if len(entries) == 0:
        return

    col = _team_color(team_id, id_to_name)
    name = _team_name(team_id, id_to_name)
    fig, ax, side = _pitch_axes()
    ax.axvspan(66.7, 100, color=col, alpha=0.08, zorder=0)
    ax.axvline(66.7, color=col, lw=1.3, ls='--', alpha=0.85)
    top = entries.sort_values('xt_gained', ascending=False)
    _draw_arrows(ax, top, col, alpha=0.54, lw=1.25, max_rows=80)
    ax.scatter(entries['end_x'], entries['end_y'], s=30 + 180 * entries['xt_gained'].clip(lower=0) / max(entries['xt_gained'].max(), 0.001),
               color=C_YELLOW, edgecolors=BG, linewidth=0.5, alpha=0.82, zorder=4)

    central = entries[(entries['end_y'].between(33.3, 66.7))]
    _metric_panel(side, [
        (len(entries), 'ENTRIES', col),
        (f'{len(central) / len(entries) * 100:.0f}%', 'CENTRAL', C_YELLOW),
        (f"{entries['xt_gained'].clip(lower=0).sum():.3f}", 'ENTRY xT', C_BLUE),
    ], title='FINAL THIRD')
    _save_visual(fig, match_title, f'{subtitle} · {name} - Final Third Entries', save_path)


# 7. Pressure Regains
# ----------------------------------------------------------------------------
def plot_pressure_regains(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    t = events[(events['contestant_id'] == team_id) & events['type_id'].isin({TID_TACKLE, TID_INTERC, TID_RECOVERY})]
    t = t.dropna(subset=['x', 'y']).copy()
    t = t[t['x'] >= 40]
    if len(t) == 0:
        return

    col = _team_color(team_id, id_to_name)
    name = _team_name(team_id, id_to_name)
    fig, ax, side = _pitch_axes()
    ax.axvspan(40, 100, color=col, alpha=0.06, zorder=0)
    sizes = np.where(t['x'] >= 66.7, 105, 58)
    ax.scatter(t['x'], t['y'], s=sizes, color=col, edgecolors='white', linewidth=0.55, alpha=0.78, zorder=5)

    high = t[t['x'] >= 66.7]
    left = ((t['y'] < 33.3).sum(), 'LEFT')
    middle = (t['y'].between(33.3, 66.7).sum(), 'CENTRE')
    right = ((t['y'] > 66.7).sum(), 'RIGHT')
    best_zone = max([left, middle, right], key=lambda x: x[0])[1]
    _metric_panel(side, [
        (len(t), 'REGAINS 40+', col),
        (len(high), 'HIGH REGAINS', C_YELLOW),
        (best_zone, 'HOT LANE', C_BLUE),
    ], title='PRESSURE')
    _save_visual(fig, match_title, f'{subtitle} · {name} - Pressure Regains', save_path)


# 8. Progressive Lane Map
# ----------------------------------------------------------------------------
def plot_progressive_lane_map(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    t = _movement_events(events, team_id, successful_only=False)
    prog = t[(t['progress'] >= 10) & (t['distance'] >= 12)]
    if len(prog) == 0:
        return

    col = _team_color(team_id, id_to_name)
    name = _team_name(team_id, id_to_name)
    fig, ax, side = _pitch_axes()
    lane_edges = [0, 20, 40, 60, 80, 100]
    lane_labels = ['L wide', 'L half', 'Centre', 'R half', 'R wide']
    lane_counts = []
    for lo, hi, label in zip(lane_edges[:-1], lane_edges[1:], lane_labels):
        ax.axhspan(lo, hi, color='white', alpha=0.025 if label != 'Centre' else 0.045, zorder=0)
        lane_counts.append((int(prog['y'].between(lo, hi, inclusive='left').sum()), label))

    complete = prog[prog['outcome'] == 1].sort_values('xt_gained', ascending=False)
    incomplete = prog[prog['outcome'] != 1]
    _draw_arrows(ax, incomplete, BAD_COL, alpha=0.30, lw=0.75, max_rows=90)
    _draw_arrows(ax, complete, col, alpha=0.62, lw=1.15, max_rows=110)
    best_lane = max(lane_counts, key=lambda x: x[0])
    comp_pct = len(complete) / len(prog) * 100
    _metric_panel(side, [
        (len(prog), 'PROG. ACTIONS', col),
        (f'{comp_pct:.0f}%', 'SUCCESS', C_GREEN),
        (best_lane[1], 'MAIN LANE', C_YELLOW),
    ], title='PROGRESSION')
    _save_visual(fig, match_title, f'{subtitle} · {name} - Progressive Lane Map', save_path)


# 9. Touch Territory Control
# ----------------------------------------------------------------------------
def plot_touch_territory(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    touch_types = {TID_PASS, TID_TAKEON, TID_TOUCH, TID_RECOVERY}
    t = events[(events['contestant_id'] == team_id) & events['type_id'].isin(touch_types)].dropna(subset=['x', 'y'])
    if len(t) == 0:
        return

    col = _team_color(team_id, id_to_name)
    name = _team_name(team_id, id_to_name)
    fig, ax, side = _pitch_axes()
    heat, xedges, yedges = np.histogram2d(t['x'], t['y'], bins=[6, 5], range=[[0, 100], [0, 100]])
    cmap = LinearSegmentedColormap.from_list('territory', [BG, '#26384a', col])
    ax.imshow(heat.T, extent=[0, 100, 0, 100], origin='lower', cmap=cmap, alpha=0.82, aspect='auto', zorder=1)
    for x in xedges[1:-1]:
        ax.axvline(x, color=BG, lw=1.0, alpha=0.6, zorder=2)
    for y in yedges[1:-1]:
        ax.axhline(y, color=BG, lw=1.0, alpha=0.6, zorder=2)
    for ix in range(6):
        for iy in range(5):
            val = int(heat[ix, iy])
            if val > 0:
                ax.text((xedges[ix] + xedges[ix + 1]) / 2, (yedges[iy] + yedges[iy + 1]) / 2,
                        str(val), color='white', fontsize=8, fontweight='bold', ha='center', va='center', zorder=4)

    final_pct = (t['x'] >= 66.7).mean() * 100
    halfspace_pct = (t['y'].between(20, 40) | t['y'].between(60, 80)).mean() * 100
    _metric_panel(side, [
        (len(t), 'TOUCHES', col),
        (f'{final_pct:.0f}%', 'FINAL 3RD', C_YELLOW),
        (f'{halfspace_pct:.0f}%', 'HALFSPACE', C_BLUE),
    ], title='TERRITORY')
    _save_visual(fig, match_title, f'{subtitle} · {name} - Touch Territory', save_path)


# 10. Player Influence
# ----------------------------------------------------------------------------
def plot_player_influence(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    touch_types = {TID_PASS, TID_TAKEON, TID_TOUCH, TID_RECOVERY, TID_TACKLE, TID_INTERC}
    touches = events[(events['contestant_id'] == team_id) & events['type_id'].isin(touch_types)].copy()
    if len(touches) == 0:
        return

    moves = _movement_events(events, team_id, successful_only=True)
    touch_tbl = touches.groupby('player_name').agg(touches=('x', 'count')).reset_index()
    if len(moves):
        mv_tbl = (moves.groupby('player_name')
                       .agg(xt=('xt_gained', lambda s: s.clip(lower=0).sum()),
                            progressive=('progress', lambda s: int((s >= 10).sum())))
                       .reset_index())
    else:
        mv_tbl = pd.DataFrame(columns=['player_name', 'xt', 'progressive'])
    def_tbl = (events[(events['contestant_id'] == team_id) & events['type_id'].isin({TID_TACKLE, TID_INTERC, TID_RECOVERY})]
               .groupby('player_name').size().rename('def_actions').reset_index())
    tbl = touch_tbl.merge(mv_tbl, on='player_name', how='left').merge(def_tbl, on='player_name', how='left').fillna(0)
    tbl['score'] = tbl['touches'] * 0.25 + tbl['progressive'] * 2.5 + tbl['xt'] * 180 + tbl['def_actions'] * 1.2
    tbl = tbl.nlargest(10, 'score').sort_values('score')
    if len(tbl) == 0:
        return

    col = _team_color(team_id, id_to_name)
    name = _team_name(team_id, id_to_name)
    fig, ax = plt.subplots(figsize=(14, 8), facecolor=BG)
    fig.subplots_adjust(left=0.24, right=0.96, bottom=0.13, top=0.82)
    ax.set_facecolor(BG)
    bars = ax.barh(tbl['player_name'].astype(str).str.split().str[-1], tbl['score'], color=col, alpha=0.86, height=0.64)
    max_score = max(float(tbl['score'].max()), 1)
    for bar, (_, row) in zip(bars, tbl.iterrows()):
        ax.text(bar.get_width() + max_score * 0.015, bar.get_y() + bar.get_height() / 2,
                f"{int(row['touches'])} tch | {int(row['progressive'])} prog | {row['xt']:.2f} xT",
                color=TEXT_SOFT, fontsize=8.5, va='center')
    ax.set_xlabel('Influence index', color=TEXT_MUTED, fontsize=10)
    ax.tick_params(colors=TEXT_MUTED)
    ax.grid(axis='x', color=PITCH_LINE, alpha=0.35, zorder=0)
    for spine in ax.spines.values():
        spine.set_edgecolor(PITCH_LINE)
    ax.set_xlim(0, max_score * 1.30)
    _save_visual(fig, match_title, f'{subtitle} · {name} - Player Influence', save_path)



# 11. Team Template Radar (Cannon idea, report styling)
# ----------------------------------------------------------------------------
def _team_match_metrics(events, team_id):
    t_all = events[events['contestant_id'] == team_id].copy()
    moves = _movement_events(events, team_id, successful_only=True)
    passes = events[(events['contestant_id'] == team_id) & (events['type_id'] == TID_PASS)].copy()
    touch_types = {TID_PASS, TID_TAKEON, TID_TOUCH, TID_RECOVERY}
    touches = events[(events['contestant_id'] == team_id) & events['type_id'].isin(touch_types)].dropna(subset=['x', 'y'])
    defensive = events[(events['contestant_id'] == team_id) & events['type_id'].isin({TID_TACKLE, TID_INTERC, TID_RECOVERY})]
    regains_high = defensive.dropna(subset=['x'])
    regains_high = regains_high[regains_high['x'] >= 66.7]

    ordered_passes = _ordered_events(passes) if len(passes) else passes
    chains, cur = [], 0
    for _, row in ordered_passes.iterrows():
        cur += 1
        if row['outcome'] != 1:
            chains.append(cur)
            cur = 0
    if cur:
        chains.append(cur)

    entries = moves[(moves['x'] < 66.7) & (moves['end_x'] >= 66.7)] if len(moves) else moves
    progressive = moves[(moves['progress'] >= 10) & (moves['distance'] >= 12)] if len(moves) else moves
    deep = moves[moves['end_x'] >= 83] if len(moves) else moves
    completed_passes = passes[passes['outcome'] == 1]

    return {
        'xT': float(moves['xt_gained'].clip(lower=0).sum()) if len(moves) else 0.0,
        'Final 3rd entries': float(len(entries)),
        'Deep completions': float(len(deep)),
        'Progressive actions': float(len(progressive)),
        'Pass completion %': float(len(completed_passes) / len(passes) * 100) if len(passes) else 0.0,
        'Avg chain': float(np.mean(chains)) if chains else 0.0,
        'Final 3rd touches': float((touches['x'] >= 66.7).sum()) if len(touches) else 0.0,
        'High regains': float(len(regains_high)),
        'Def actions': float(len(defensive)),
    }


def _score_metric(value, target, invert=False):
    if target <= 0:
        return 0.0
    score = 100 * float(value) / float(target)
    if invert:
        score = 100 - score
    return float(np.clip(score, 0, 100))


def _save_dark_report(fig, match_title, subtitle, save_path):
    _title_footer(fig, match_title, subtitle)
    _save_or_show(fig, save_path)


def plot_cannon_team_template(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Radial team template using Cannon's idea but the notebook's dark report style."""
    metrics = _team_match_metrics(events, team_id)
    name = _team_name(team_id, id_to_name)
    team_col = _team_color(team_id, id_to_name)
    targets = {
        'xT': 1.20, 'Final 3rd entries': 38, 'Deep completions': 24,
        'Progressive actions': 58, 'Pass completion %': 84, 'Avg chain': 4.8,
        'Final 3rd touches': 90, 'High regains': 12, 'Def actions': 72,
    }
    order = [
        ('xT', C_ORANGE), ('Final 3rd entries', C_ORANGE), ('Deep completions', C_ORANGE),
        ('Progressive actions', C_BLUE), ('Pass completion %', C_BLUE), ('Avg chain', C_BLUE),
        ('Final 3rd touches', C_PURPLE), ('High regains', C_GREEN), ('Def actions', C_GREEN),
    ]
    labels = [m for m, _ in order]
    scores = np.array([_score_metric(metrics[m], targets[m]) for m, _ in order], dtype=float)
    colors = [c for _, c in order]

    fig = plt.figure(figsize=(16, 9), facecolor=BG)
    ax = fig.add_axes([0.06, 0.11, 0.62, 0.72], projection='polar', facecolor=BG)
    side = fig.add_axes([0.73, 0.13, 0.23, 0.68])
    _style_panel(side)

    theta = np.linspace(0, 2 * np.pi, len(labels), endpoint=False)
    width = 2 * np.pi / len(labels) * 0.82
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_ylim(0, 100)
    ax.set_yticks([20, 40, 60, 80, 100])
    ax.set_yticklabels(['20', '40', '60', '80', '100'], fontsize=7, color=TEXT_MUTED)
    ax.set_xticks(theta)
    ax.set_xticklabels([''] * len(labels))
    ax.grid(color=PITCH_LINE, lw=0.8, alpha=0.85)
    ax.spines['polar'].set_color(PITCH_LINE)
    ax.bar(theta, scores, width=width, bottom=0, color=colors, alpha=0.82, edgecolor=BG, linewidth=1.4)
    ax.plot(np.r_[theta, theta[0]], np.r_[scores, scores[0]], color='white', lw=1.5, alpha=0.85)
    ax.scatter(theta, scores, s=32, color='white', edgecolors=team_col, linewidth=1.1, zorder=5)

    for ang, label, score in zip(theta, labels, scores):
        ha = 'left' if np.cos(ang) >= 0 else 'right'
        ax.text(ang, 108, label, fontsize=7.8, color='#d7e3ee', ha=ha, va='center')
        ax.text(ang, max(score - 8, 10), f'{score:.0f}', fontsize=7, color='white', ha='center', va='center', fontweight='bold')

    side.text(0.5, 0.96, 'TEMPLATE OUTPUT', color=TEXT_MUTED, fontsize=8.5,
              fontweight='bold', ha='center', va='top', transform=side.transAxes)
    y = 0.86
    for metric, color in order:
        raw = metrics[metric]
        raw_txt = f'{raw:.2f}' if metric in {'xT', 'Avg chain'} else f'{raw:.0f}'
        side.add_patch(mpatches.Rectangle((0.05, y - 0.018), 0.035, 0.030, transform=side.transAxes,
                                          color=color, alpha=0.95))
        side.text(0.12, y, metric, fontsize=7.4, color=TEXT_SOFT, va='center', transform=side.transAxes)
        side.text(0.93, y, raw_txt, fontsize=8.3, color='white', va='center', ha='right',
                  fontweight='bold', transform=side.transAxes)
        y -= 0.077
    side.text(0.5, 0.045, 'Capped to match benchmarks; further out is stronger.', fontsize=7,
              color=TEXT_MUTED, ha='center', va='bottom', transform=side.transAxes)
    _save_dark_report(fig, match_title, f'{subtitle} · {name} - Team Template Radar', save_path)


# 12. KPI Target Board (Cannon idea, report styling)
# ----------------------------------------------------------------------------
def plot_cannon_kpi_target_board(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Target-vs-actual KPI board in the existing dark report language."""
    metrics = _team_match_metrics(events, team_id)
    name = _team_name(team_id, id_to_name)
    team_col = _team_color(team_id, id_to_name)
    rows = [
        ('xT', metrics['xT'], 1.20, C_ORANGE),
        ('Final 3rd entries', metrics['Final 3rd entries'], 38, C_ORANGE),
        ('Deep completions', metrics['Deep completions'], 24, C_ORANGE),
        ('Progressive actions', metrics['Progressive actions'], 58, C_BLUE),
        ('Pass completion %', metrics['Pass completion %'], 84, C_BLUE),
        ('Avg chain', metrics['Avg chain'], 4.8, C_BLUE),
        ('High regains', metrics['High regains'], 12, C_GREEN),
        ('Def actions', metrics['Def actions'], 72, C_GREEN),
    ]

    fig, ax = plt.subplots(figsize=(16, 9), facecolor=BG)
    fig.subplots_adjust(left=0.27, right=0.88, top=0.82, bottom=0.14)
    ax.set_facecolor(BG)
    y_pos = np.arange(len(rows))[::-1]
    ax.set_xlim(0, 130)
    ax.set_ylim(-0.7, len(rows) - 0.3)
    ax.axvline(100, color=C_YELLOW, lw=1.8, alpha=0.95)
    ax.text(100, len(rows) - 0.04, 'TARGET', color=C_YELLOW, fontsize=8, ha='center', va='bottom', fontweight='bold')

    for y, (label, value, target, color) in zip(y_pos, rows):
        pct = _score_metric(value, target)
        ax.barh(y, 130, color=PANEL_BG, height=0.56, edgecolor=PITCH_LINE, linewidth=0.7, zorder=1)
        ax.barh(y, min(pct, 130), color=color if pct >= 90 else BAD_COL, height=0.56, alpha=0.88, zorder=3)
        raw_txt = f'{value:.2f}' if label in {'xT', 'Avg chain'} else f'{value:.0f}'
        target_txt = f'{target:.2f}' if label in {'xT', 'Avg chain'} else f'{target:.0f}'
        status = 'EXCEEDING' if pct >= 105 else 'ON TRACK' if pct >= 90 else 'BELOW'
        ax.text(-4, y, label, ha='right', va='center', fontsize=9, color='#d7e3ee', fontweight='bold')
        ax.text(132, y, f'{raw_txt} / {target_txt}', ha='left', va='center', fontsize=8.5, color='white', fontweight='bold')
        ax.text(132, y - 0.22, status, ha='left', va='center', fontsize=7.2,
                color=color if pct >= 90 else TEXT_MUTED, fontweight='bold')

    ax.set_yticks([])
    ax.set_xticks([0, 50, 100, 130])
    ax.set_xticklabels(['0', '50', '100', '130'], color=TEXT_MUTED, fontsize=8)
    ax.grid(axis='x', color=PITCH_LINE, lw=0.8, alpha=0.55)
    for spine in ax.spines.values():
        spine.set_edgecolor(PITCH_LINE)
    fig.text(0.07, 0.785, name, color=team_col, fontsize=16, fontweight='bold', ha='left')
    fig.text(0.07, 0.748, 'KPI target board', color=TEXT_MUTED, fontsize=9, ha='left', fontweight='bold')
    _save_dark_report(fig, match_title, f'{subtitle} · {name} - KPI Target Board', save_path)


# 13. Player Slices (Cannon idea, report styling)
# ----------------------------------------------------------------------------
def plot_cannon_player_slices(events, team_id, id_to_name, match_title, subtitle='', save_path=None):
    """Player percentile slices restyled to match the dark event report."""
    name = _team_name(team_id, id_to_name)
    team_col = _team_color(team_id, id_to_name)
    touch_types = {TID_PASS, TID_TAKEON, TID_TOUCH, TID_RECOVERY, TID_TACKLE, TID_INTERC}
    touches = events[(events['contestant_id'] == team_id) & events['type_id'].isin(touch_types)].copy()
    if len(touches) == 0:
        return
    moves = _movement_events(events, team_id, successful_only=True)
    player_rows = touches.groupby('player_name').agg(touches=('x', 'count')).reset_index()
    if len(moves):
        move_rows = (moves.groupby('player_name')
                          .agg(xt=('xt_gained', lambda s: s.clip(lower=0).sum()),
                               prog=('progress', lambda s: int((s >= 10).sum())),
                               final_entries=('end_x', lambda s: int((s >= 66.7).sum())))
                          .reset_index())
    else:
        move_rows = pd.DataFrame(columns=['player_name', 'xt', 'prog', 'final_entries'])
    def_rows = (events[(events['contestant_id'] == team_id) & events['type_id'].isin({TID_TACKLE, TID_INTERC, TID_RECOVERY})]
                .groupby('player_name').size().rename('def_actions').reset_index())
    tbl = player_rows.merge(move_rows, on='player_name', how='left').merge(def_rows, on='player_name', how='left').fillna(0)
    tbl = tbl[tbl['touches'] >= 3].copy()
    if len(tbl) == 0:
        return
    tbl['attack_score'] = tbl['xt'].rank(pct=True) * 60 + tbl['final_entries'].rank(pct=True) * 40
    tbl['pos_score'] = tbl['touches'].rank(pct=True) * 45 + tbl['prog'].rank(pct=True) * 55
    tbl['def_score'] = tbl['def_actions'].rank(pct=True) * 100
    tbl['overall'] = tbl[['attack_score', 'pos_score', 'def_score']].mean(axis=1)
    tbl = tbl.nlargest(8, 'overall').sort_values('overall', ascending=False)

    fig = plt.figure(figsize=(16, 9), facecolor=BG)
    ax = fig.add_axes([0.24, 0.14, 0.58, 0.67])
    ax.set_facecolor(BG)
    cats = [('attack_score', 'ATT', C_ORANGE), ('pos_score', 'POS', C_BLUE), ('def_score', 'DEF', C_GREEN)]
    row_gap = 1.0
    bar_h = 0.18
    yticks, ylabels = [], []
    for i, (_, row) in enumerate(tbl.iterrows()):
        base_y = (len(tbl) - 1 - i) * row_gap
        yticks.append(base_y)
        ylabels.append(str(row['player_name']).split()[-1])
        for j, (col_name, label, color) in enumerate(cats):
            y = base_y + (1 - j) * bar_h * 1.25
            ax.barh(y, 100, color=PANEL_BG, height=bar_h, edgecolor=PITCH_LINE, linewidth=0.4)
            ax.barh(y, row[col_name], color=color, height=bar_h, edgecolor='none', alpha=0.92)
            ax.text(102, y, f'{row[col_name]:.0f}', va='center', ha='left', fontsize=8, color='white', fontweight='bold')
        ax.text(-5, base_y - 0.36, f"{int(row['touches'])} touches | {int(row['prog'])} prog | {row['xt']:.2f} xT | {int(row['def_actions'])} def",
                va='center', ha='right', fontsize=7.5, color=TEXT_MUTED)

    ax.set_xlim(0, 112)
    ax.set_ylim(-0.8, len(tbl) * row_gap - 0.05)
    ax.set_yticks(yticks)
    ax.set_yticklabels(ylabels, fontsize=10, color='#d7e3ee', fontweight='bold')
    ax.set_xticks([0, 25, 50, 75, 100])
    ax.set_xticklabels(['0', '25', '50', '75', '100'], fontsize=8, color=TEXT_MUTED)
    ax.grid(axis='x', color=PITCH_LINE, lw=0.8, alpha=0.55)
    for spine in ax.spines.values():
        spine.set_edgecolor(PITCH_LINE)

    leg = fig.add_axes([0.84, 0.52, 0.12, 0.22])
    _style_panel(leg)
    leg.text(0.5, 0.88, 'SLICES', color=TEXT_MUTED, fontsize=8.5, fontweight='bold', ha='center', transform=leg.transAxes)
    for k, (_, label, color) in enumerate(cats):
        y = 0.64 - k * 0.22
        leg.add_patch(mpatches.Rectangle((0.16, y - 0.04), 0.10, 0.08, transform=leg.transAxes, color=color))
        leg.text(0.34, y, label, color='white', fontsize=8.5, va='center', transform=leg.transAxes)
    fig.text(0.07, 0.785, name, color=team_col, fontsize=16, fontweight='bold', ha='left')
    fig.text(0.07, 0.748, 'Player percentile slices', color=TEXT_MUTED, fontsize=9, ha='left', fontweight='bold')
    _save_dark_report(fig, match_title, f'{subtitle} · {name} - Player Slices', save_path)


# Convenience runner
# ----------------------------------------------------------------------------
def generate_pdf_visuals(events, home_id, away_id, id_to_name, match_title, subtitle='', out_dir=None):
    """Generate thirteen advanced add-on visuals for both teams (26 total)."""
    if out_dir is not None:
        Path(out_dir).mkdir(parents=True, exist_ok=True)

    visuals = [
        ('avg_positions', plot_avg_positions),
        ('pass_clusters', plot_pass_clusters),
        ('xt_heatmap', plot_xt_heatmap),
        ('gk_distribution', plot_gk_distribution),
        ('possession_chains', plot_possession_duration),
        ('final_third_entries', plot_final_third_entries),
        ('pressure_regains', plot_pressure_regains),
        ('progressive_lanes', plot_progressive_lane_map),
        ('touch_territory', plot_touch_territory),
        ('player_influence', plot_player_influence),
        ('team_template_radar', plot_cannon_team_template),
        ('kpi_target_board', plot_cannon_kpi_target_board),
        ('player_slices', plot_cannon_player_slices),
    ]

    def _path(team, tag, n):
        if out_dir is None:
            return None
        tname = _slug(id_to_name.get(team, str(team)))
        return str(Path(out_dir) / '08_AdvancedAddOns' / f'{n:02d}_{tname}_{tag}.png')

    for n_off, tid in enumerate([home_id, away_id]):
        base = 71 + n_off * len(visuals)
        for idx, (tag, fn) in enumerate(visuals):
            try:
                fn(events, tid, id_to_name, match_title, subtitle, save_path=_path(tid, tag, base + idx))
            except Exception as exc:
                import traceback
                print(f'    warning: {fn.__name__} failed for {id_to_name.get(tid, tid)}: {exc}')
                traceback.print_exc()
                plt.close('all')


    # ── xCrossAcc visuals (both teams in one call) ────────────────────────────
    def _xpath(tag, n):
        if out_dir is None:
            return None
        return str(Path(out_dir) / '09_CrossModel' / f'{n:02d}_{tag}.png')

    for n_c, (tag, fn) in enumerate([
        ('xcross_map',       plot_xcross_map),
        ('xcross_zones',     plot_xcross_zone),
        ('xcross_player_oe', plot_xcross_player_oe),
    ]):
        try:
            fn(events, home_id, away_id, id_to_name, match_title,
               subtitle=subtitle, save_path=_xpath(tag, 97 + n_c))
        except Exception as exc:
            import traceback
            print(f'    warning: {fn.__name__} failed: {exc}')
            traceback.print_exc()
            plt.close('all')
    print('Advanced add-on visuals complete.')


# ══════════════════════════════════════════════════════════════════════════════
# ONE-CELL REPORT RUNNER: STANDARD + ADVANCED VISUALS
# ══════════════════════════════════════════════════════════════════════════════

import glob, os
from pathlib import Path

# ── Configure these paths ─────────────────────────────────────────────────────
MATCH_GLOB = "/Users/marclamberts/Event data/WC 2026/NEW/*.json"
OUT_DIR    = "/Users/marclamberts/Event data/WC 2026/reports"

# ── Run the full report + advanced add-on visuals ───────────────────────────
for path in sorted(glob.glob(MATCH_GLOB)):
    fname = Path(path).name
    match_out = str(Path(OUT_DIR) / Path(path).stem)
    print(f"\n{'─'*60}")
    print(f"Processing: {fname}")

    try:
        # 1. Standard 70+ visuals
        generate_game_report(path, out_dir=OUT_DIR)

        # 2. Advanced add-on visuals (13 per team, including Cannon-style charts)
        events, meta = load_match_events(path)
        home_id    = meta['home_id']
        away_id    = meta['away_id']
        id_to_name = meta['id_to_name']
        hn = meta.get('home_name', home_id)
        an = meta.get('away_name', away_id)
        sc = meta.get('score', {})
        title    = f"{hn} {sc.get('home','')}\u2013{sc.get('away','')} {an}"
        subtitle = meta.get('competition', '') + '  ·  ' + meta.get('match_date', '')

        generate_pdf_visuals(
            events, home_id, away_id, id_to_name,
            title, subtitle=subtitle,
            out_dir=match_out,
        )

        # 3. Brand-style add-ons + curated PDF report
        try:
            from match_report_addons import generate_addon_reports
        except ImportError:
            _pip('reportlab', 'pillow')
            from match_report_addons import generate_addon_reports

        try:
            addon_result = generate_addon_reports(match_out, include_contact_sheet=True)
            print(f"  ✓ PDF report: {addon_result['report_pdf']}")
        except Exception as addon_exc:
            import traceback
            print(f"  ⚠ Brand-style PDF add-ons failed: {addon_exc}")
            traceback.print_exc()

        print(f"  ✓ {fname}")
    except Exception as exc:
        import traceback
        print(f"  ✗ {fname}: {exc}")
        traceback.print_exc()
